In [1]:
# Cell 1: Clean setup and canonical Corsi/HAR parameter-sweep panel
#
# Purpose:
#   Start a clean parameter-sweep workbook using only:
#
#       har_total_simple as the forecast denominator
#
#   This cell:
#       1. Finds the latest Corsi/HAR OOS signal panel.
#       2. Finds the latest naked ATM put outcome file.
#       3. Joins outcomes onto date x tenor rows.
#       4. Adds RSI14 and RV21D from the production feature panel if needed.
#       5. Creates clean, renamed columns for the new sweep.
#       6. Saves the clean panel for this workbook.
#
# Important:
#   No parameters are tested in this cell.
#   No filters are removed.
#   No signal decision is made here.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Project paths
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "forecast_model_corsi_v1"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"

PRODUCTION_FEATURE_PANEL_PATH = PROJECT_ROOT / "data" / "processed" / "production_feature_panel_v0_1.parquet"

PARAM_SWEEP_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

print("Corsi parameter sweep clean setup")
print(f"Project root:              {PROJECT_ROOT}")
print(f"Corsi processed dir:        {CORSI_PROCESSED_DIR}")
print(f"Corsi audit dir:            {CORSI_AUDIT_DIR}")
print(f"Sweep processed dir:        {PARAM_SWEEP_PROCESSED_DIR}")
print(f"Sweep audit dir:            {PARAM_SWEEP_AUDIT_DIR}")
print(f"Run timestamp:              {RUN_TIMESTAMP}")


# -----------------------------
# 2. Locate latest Corsi OOS signal panel
# -----------------------------

signal_panel_candidates = sorted(
    CORSI_PROCESSED_DIR.glob("corsi_cell12_oos_forecast_vrp_signal_panel_*.parquet")
)

if not signal_panel_candidates:
    raise FileNotFoundError(
        f"No Corsi OOS signal panel found in {CORSI_PROCESSED_DIR}"
    )

SIGNAL_PANEL_PATH = signal_panel_candidates[-1]

print()
print("Selected Corsi signal panel:")
print(SIGNAL_PANEL_PATH)

signal_raw = pd.read_parquet(SIGNAL_PANEL_PATH).copy()

signal_raw.columns = [str(c).strip() for c in signal_raw.columns]

print()
print("Raw Corsi signal panel summary:")
display(pd.DataFrame([{
    "rows": len(signal_raw),
    "columns": len(signal_raw.columns),
    "first_date": pd.to_datetime(signal_raw["date"]).min() if "date" in signal_raw.columns else pd.NaT,
    "last_date": pd.to_datetime(signal_raw["date"]).max() if "date" in signal_raw.columns else pd.NaT,
    "unique_dates": signal_raw["date"].nunique() if "date" in signal_raw.columns else np.nan,
    "unique_tenors": signal_raw["tenor"].nunique() if "tenor" in signal_raw.columns else np.nan,
}]))


# -----------------------------
# 3. Locate latest outcome file
# -----------------------------

outcome_discovery_candidates = sorted(
    CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
)

if not outcome_discovery_candidates:
    raise FileNotFoundError(
        f"No Cell 11 outcome discovery file found in {CORSI_AUDIT_DIR}"
    )

OUTCOME_DISCOVERY_PATH = outcome_discovery_candidates[-1]

print()
print("Selected outcome discovery file:")
print(OUTCOME_DISCOVERY_PATH)

outcome_discovery = pd.read_csv(OUTCOME_DISCOVERY_PATH)
outcome_discovery.columns = [str(c).strip() for c in outcome_discovery.columns]

display(outcome_discovery)

# The discovery file schema may use either:
#   old names: chosen_file / outcome_col / outcome_kind
#   new names: path / chosen_outcome_col / chosen_outcome_kind
#
# Handle both explicitly.

file_col = "chosen_file" if "chosen_file" in outcome_discovery.columns else "path"
chosen_col_col = "outcome_col" if "outcome_col" in outcome_discovery.columns else "chosen_outcome_col"
chosen_kind_col = "outcome_kind" if "outcome_kind" in outcome_discovery.columns else "chosen_outcome_kind"

required_discovery_cols = [
    file_col,
    "date_col",
    "trade_date_col",
    "tenor_col",
    chosen_col_col,
    chosen_kind_col,
]

missing_discovery_cols = [
    c for c in required_discovery_cols
    if c not in outcome_discovery.columns
]

if missing_discovery_cols:
    raise ValueError(f"Outcome discovery file missing required columns: {missing_discovery_cols}")

preferred = outcome_discovery.copy()

# Keep only eligible rows if the column exists.
if "eligible_for_backtest" in preferred.columns:
    preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

if preferred.empty:
    raise ValueError("No eligible outcome files found in outcome discovery file.")

preferred["_preferred_outcome_col"] = preferred[chosen_col_col].eq("normalized_pnl_dollars")
preferred["_preferred_outcome_kind"] = preferred[chosen_kind_col].eq("pnl")

if "score" in preferred.columns:
    preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
else:
    preferred["_score"] = 0.0

preferred = preferred.sort_values(
    ["_preferred_outcome_col", "_preferred_outcome_kind", "_score"],
    ascending=[False, False, False],
)

chosen_outcome_row = preferred.iloc[0].to_dict()

OUTCOME_PATH = Path(chosen_outcome_row[file_col])
OUTCOME_DATE_COL = chosen_outcome_row.get("date_col", "trade_date")
OUTCOME_TRADE_DATE_COL = chosen_outcome_row.get("trade_date_col", "trade_date")
OUTCOME_TENOR_COL = chosen_outcome_row.get("tenor_col", "tenor")
OUTCOME_VALUE_COL = chosen_outcome_row.get(chosen_col_col, "normalized_pnl_dollars")

print()
print("Chosen outcome file:")
print(OUTCOME_PATH)
print(f"date_col:        {OUTCOME_DATE_COL}")
print(f"trade_date_col:  {OUTCOME_TRADE_DATE_COL}")
print(f"tenor_col:       {OUTCOME_TENOR_COL}")
print(f"outcome_col:     {OUTCOME_VALUE_COL}")

outcome_raw = (
    pd.read_parquet(OUTCOME_PATH)
    if OUTCOME_PATH.suffix.lower() == ".parquet"
    else pd.read_csv(OUTCOME_PATH)
)

outcome_raw.columns = [str(c).strip() for c in outcome_raw.columns]

required_outcome_cols = [OUTCOME_TRADE_DATE_COL, OUTCOME_TENOR_COL, OUTCOME_VALUE_COL]
missing_outcome_cols = [c for c in required_outcome_cols if c not in outcome_raw.columns]

if missing_outcome_cols:
    raise ValueError(f"Outcome file missing required columns: {missing_outcome_cols}")

outcome = outcome_raw[
    [OUTCOME_TRADE_DATE_COL, OUTCOME_TENOR_COL, OUTCOME_VALUE_COL]
].copy()

outcome = outcome.rename(columns={
    OUTCOME_TRADE_DATE_COL: "trade_date",
    OUTCOME_TENOR_COL: "tenor",
    OUTCOME_VALUE_COL: "outcome_value",
})

outcome["trade_date"] = pd.to_numeric(outcome["trade_date"], errors="coerce").astype("Int64")
outcome["tenor"] = pd.to_numeric(outcome["tenor"], errors="coerce").astype("Int64")
outcome["outcome_value"] = pd.to_numeric(outcome["outcome_value"], errors="coerce")

outcome = outcome.dropna(subset=["trade_date", "tenor"])
outcome["trade_date"] = outcome["trade_date"].astype(int)
outcome["tenor"] = outcome["tenor"].astype(int)

duplicate_outcome_rows = outcome.duplicated(["trade_date", "tenor"]).sum()

if duplicate_outcome_rows:
    raise ValueError(f"Outcome file has duplicate trade_date x tenor rows: {duplicate_outcome_rows}")

print()
print("Outcome table summary:")
display(pd.DataFrame([{
    "rows": len(outcome),
    "first_trade_date": outcome["trade_date"].min(),
    "last_trade_date": outcome["trade_date"].max(),
    "unique_trade_dates": outcome["trade_date"].nunique(),
    "unique_tenors": outcome["tenor"].nunique(),
    "duplicate_trade_date_tenor_rows": duplicate_outcome_rows,
    "outcome_non_null_rows": int(outcome["outcome_value"].notna().sum()),
}]))


# -----------------------------
# 4. Normalize signal panel and standardize har_total_simple columns
# -----------------------------

signal = signal_raw.copy()
signal.columns = [str(c).strip() for c in signal.columns]

print()
print("Corsi signal panel columns related to model / forecast / VRP / score:")
diagnostic_cols = [
    c for c in signal.columns
    if any(x in c.lower() for x in ["model", "forecast", "vrp", "score", "z_3m", "z_1y", "har"])
]
display(pd.DataFrame({"column": diagnostic_cols}))

base_required_cols = [
    "date",
    "trade_date",
    "tenor",
    "implied_variance",
    "vix_style_vol",
]

missing_base_cols = [c for c in base_required_cols if c not in signal.columns]

if missing_base_cols:
    raise ValueError(f"Corsi signal panel missing base required columns: {missing_base_cols}")

# Normalize date / tenor keys first.
signal["date"] = pd.to_datetime(signal["date"])
signal["trade_date"] = pd.to_numeric(signal["trade_date"], errors="coerce").astype("Int64")
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype("Int64")

signal = signal.dropna(subset=["trade_date", "tenor"]).copy()
signal["trade_date"] = signal["trade_date"].astype(int)
signal["tenor"] = signal["tenor"].astype(int)

valid_tenors = [9, 12, 15, 18, 21, 24, 27, 30, 33]
signal = signal.loc[signal["tenor"].isin(valid_tenors)].copy()

# Tenor bucket.
# Do this with explicit loc assignment instead of np.select
# to avoid string / NaN dtype promotion issues in newer numpy.
signal["tenor_bucket"] = pd.NA
signal.loc[signal["tenor"].isin([9, 12, 15]), "tenor_bucket"] = "front"
signal.loc[signal["tenor"].isin([18, 21, 24]), "tenor_bucket"] = "middle"
signal.loc[signal["tenor"].isin([27, 30, 33]), "tenor_bucket"] = "back"

signal = signal.replace([np.inf, -np.inf], np.nan)


# -----------------------------
# 4A. Confirm this is a wide panel, not a long model panel
# -----------------------------

model_col_candidates = [
    "model_label",
    "forecast_model",
    "model_name",
    "candidate_model",
]

model_col = next((c for c in model_col_candidates if c in signal.columns), None)

if model_col is not None:
    print()
    print(f"Detected model column: {model_col}")
    model_values = sorted(signal[model_col].dropna().astype(str).unique())
    display(pd.DataFrame({model_col: model_values}))

    model_lower = signal[model_col].astype(str).str.lower()

    har_total_mask = (
        model_lower.eq("har_total")
        | model_lower.eq("har_total_simple")
        | model_lower.str.contains("har_total_simple", na=False)
        | (
            model_lower.str.contains("har_total", na=False)
            & ~model_lower.str.contains("implied", na=False)
        )
    )

    if har_total_mask.any():
        before_rows = len(signal)
        signal = signal.loc[har_total_mask].copy()
        after_rows = len(signal)

        print()
        print(f"Filtered to har_total / har_total_simple rows: {after_rows:,} of {before_rows:,}")
    else:
        raise ValueError(
            f"Found model column '{model_col}', but no har_total / har_total_simple rows were detected."
        )
else:
    print()
    print("No model-label column detected. Treating panel as wide model panel.")


# -----------------------------
# 4B. Standardize har_total_simple forecast / VRP / z-score columns
# -----------------------------

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


col_map = {
    "forecast_variance_har_total": first_existing_col(signal, [
        "forecast_variance_har_total_oos",
        "forecast_variance_har_total",
        "har_total_forecast_variance",
    ]),
    "forecast_vol_har_total": first_existing_col(signal, [
        "forecast_vol_har_total_oos",
        "forecast_vol_har_total",
        "har_total_forecast_vol",
    ]),
    "vrp_log_har_total": first_existing_col(signal, [
        "vrp_log_har_total_oos",
        "vrp_log_har_total",
        "score_har_total_raw_vrp_log",
        "har_total_vrp_log",
    ]),
    "vrp_z_3m_har_total": first_existing_col(signal, [
        "vrp_z_3m_har_total_oos_window",
        "vrp_z_3m_har_total",
        "score_har_total_z_3m",
        "har_total_vrp_z_3m",
    ]),
    "vrp_z_1y_har_total": first_existing_col(signal, [
        "vrp_z_1y_har_total_oos_window",
        "vrp_z_1y_har_total",
        "score_har_total_z_1y",
        "har_total_vrp_z_1y",
    ]),
}

print()
print("har_total_simple column mapping:")
display(pd.DataFrame([
    {"standard_column": k, "source_column": v}
    for k, v in col_map.items()
]))

missing_mapped_cols = [k for k, v in col_map.items() if v is None]

if missing_mapped_cols:
    raise ValueError(
        "Could not map required har_total_simple columns. "
        f"Missing standard columns: {missing_mapped_cols}. "
        "The diagnostic column table above shows what exists in the file."
    )

# Create the standard column names expected by the rest of Cell 1.
for standard_col, source_col in col_map.items():
    signal[standard_col] = pd.to_numeric(signal[source_col], errors="coerce")

# If forecast vol is missing, recompute from variance.
if signal["forecast_vol_har_total"].isna().all():
    signal["forecast_vol_har_total"] = np.sqrt(signal["forecast_variance_har_total"]) * 100.0

# Hard check: the denominator must be positive.
bad_forecast_rows = signal["forecast_variance_har_total"].le(0).sum()

if bad_forecast_rows:
    raise ValueError(f"Non-positive har_total forecast variance rows detected: {bad_forecast_rows}")

# Hard check: one row per trade_date x tenor.
duplicate_signal_rows = signal.duplicated(["trade_date", "tenor"]).sum()

if duplicate_signal_rows:
    dup_sample = (
        signal.loc[signal.duplicated(["trade_date", "tenor"], keep=False)]
        .sort_values(["trade_date", "tenor"])
        .head(20)
    )
    display(dup_sample)
    raise ValueError(f"Signal panel has duplicate trade_date x tenor rows after har_total standardization: {duplicate_signal_rows}")

print()
print("Standardized har_total_simple signal panel summary:")
display(pd.DataFrame([{
    "rows": len(signal),
    "first_date": signal["date"].min(),
    "last_date": signal["date"].max(),
    "unique_dates": signal["date"].nunique(),
    "unique_tenors": signal["tenor"].nunique(),
    "forecast_variance_non_null": int(signal["forecast_variance_har_total"].notna().sum()),
    "forecast_vol_non_null": int(signal["forecast_vol_har_total"].notna().sum()),
    "vrp_log_non_null": int(signal["vrp_log_har_total"].notna().sum()),
    "z_3m_non_null": int(signal["vrp_z_3m_har_total"].notna().sum()),
    "z_1y_non_null": int(signal["vrp_z_1y_har_total"].notna().sum()),
}]))

# -----------------------------
# 5. Add RSI14 / RV21D from production feature panel if needed
# -----------------------------

needs_prod_join = ("rsi14" not in signal.columns) or ("rv21d" not in signal.columns)

if needs_prod_join:
    if not PRODUCTION_FEATURE_PANEL_PATH.exists():
        raise FileNotFoundError(
            f"Need rsi14/rv21d but production feature panel does not exist: {PRODUCTION_FEATURE_PANEL_PATH}"
        )

    print()
    print("Adding RSI14 / RV21D from production feature panel:")
    print(PRODUCTION_FEATURE_PANEL_PATH)

    prod = pd.read_parquet(PRODUCTION_FEATURE_PANEL_PATH).copy()
    prod.columns = [str(c).strip() for c in prod.columns]

    prod["trade_date"] = pd.to_numeric(prod["trade_date"], errors="coerce").astype("Int64")
    prod["tenor"] = pd.to_numeric(prod["tenor"], errors="coerce").astype("Int64")

    prod = prod.dropna(subset=["trade_date", "tenor"])
    prod["trade_date"] = prod["trade_date"].astype(int)
    prod["tenor"] = prod["tenor"].astype(int)

    prod_cols = [
        "trade_date",
        "tenor",
        "spx_close",
        "spx_log_return",
        "rv21d",
        "rsi14",
    ]

    prod_cols = [c for c in prod_cols if c in prod.columns]

    prod_state = prod[prod_cols].drop_duplicates(["trade_date", "tenor"]).copy()

    # Drop any overlapping state columns first so the join is clean.
    overlap_cols = [c for c in prod_state.columns if c in signal.columns and c not in ["trade_date", "tenor"]]
    if overlap_cols:
        signal = signal.drop(columns=overlap_cols)

    signal = signal.merge(
        prod_state,
        on=["trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )

else:
    print()
    print("RSI14 and RV21D already present in Corsi signal panel.")


# -----------------------------
# 6. Join outcomes and filter to available outcome range
# -----------------------------

outcome_min_trade_date = int(outcome["trade_date"].min())
outcome_max_trade_date = int(outcome["trade_date"].max())

signal_before_outcome_filter_rows = len(signal)

signal = signal.loc[
    signal["trade_date"].between(outcome_min_trade_date, outcome_max_trade_date)
].copy()

signal_after_outcome_filter_rows = len(signal)

panel = signal.merge(
    outcome,
    on=["trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

panel["has_outcome"] = panel["outcome_value"].notna()


# -----------------------------
# 7. Build clean canonical columns
# -----------------------------

clean_panel = pd.DataFrame({
    "date": panel["date"],
    "trade_date": panel["trade_date"],
    "tenor": panel["tenor"],
    "tenor_bucket": panel["tenor_bucket"],

    "implied_variance": pd.to_numeric(panel["implied_variance"], errors="coerce"),
    "vix_style_vol": pd.to_numeric(panel["vix_style_vol"], errors="coerce"),

    "forecast_model": "har_total_simple",
    "forecast_variance": pd.to_numeric(panel["forecast_variance_har_total"], errors="coerce"),
    "forecast_vol": pd.to_numeric(panel["forecast_vol_har_total"], errors="coerce"),

    "vrp_log": pd.to_numeric(panel["vrp_log_har_total"], errors="coerce"),
    "vrp_z_3m": pd.to_numeric(panel["vrp_z_3m_har_total"], errors="coerce"),
    "vrp_z_1y": pd.to_numeric(panel["vrp_z_1y_har_total"], errors="coerce"),

    "rv21d": pd.to_numeric(panel["rv21d"], errors="coerce"),
    "rsi14": pd.to_numeric(panel["rsi14"], errors="coerce"),

    "outcome_value": pd.to_numeric(panel["outcome_value"], errors="coerce"),
    "has_outcome": panel["has_outcome"].astype(bool),
})

# Optional state columns if available.
for optional_col in ["spx_close", "spx_log_return"]:
    if optional_col in panel.columns:
        clean_panel[optional_col] = pd.to_numeric(panel[optional_col], errors="coerce")

clean_panel = clean_panel.replace([np.inf, -np.inf], np.nan)

clean_panel["is_parameter_sweep_eligible"] = (
    clean_panel["tenor"].isin(valid_tenors)
    & clean_panel["tenor_bucket"].notna()
    & clean_panel["implied_variance"].gt(0)
    & clean_panel["forecast_variance"].gt(0)
    & clean_panel["vrp_log"].notna()
    & clean_panel["vrp_z_3m"].notna()
    & clean_panel["vrp_z_1y"].notna()
    & clean_panel["rv21d"].notna()
    & clean_panel["rsi14"].notna()
    & clean_panel["has_outcome"]
)

duplicate_clean_rows = clean_panel.duplicated(["trade_date", "tenor"]).sum()

if duplicate_clean_rows:
    raise ValueError(f"Clean panel has duplicate trade_date x tenor rows: {duplicate_clean_rows}")


# -----------------------------
# 8. Validation summaries
# -----------------------------

summary = pd.DataFrame([{
    "raw_signal_rows": len(signal_raw),
    "signal_rows_before_outcome_filter": signal_before_outcome_filter_rows,
    "signal_rows_after_outcome_filter": signal_after_outcome_filter_rows,
    "clean_panel_rows": len(clean_panel),
    "first_date": clean_panel["date"].min(),
    "last_date": clean_panel["date"].max(),
    "unique_dates": clean_panel["date"].nunique(),
    "unique_tenors": clean_panel["tenor"].nunique(),
    "outcome_rows": int(clean_panel["has_outcome"].sum()),
    "outcome_coverage_pct": float(clean_panel["has_outcome"].mean()),
    "sweep_eligible_rows": int(clean_panel["is_parameter_sweep_eligible"].sum()),
    "sweep_eligible_pct": float(clean_panel["is_parameter_sweep_eligible"].mean()),
    "duplicate_trade_date_tenor_rows": int(duplicate_clean_rows),
}])

availability = []

for col in [
    "implied_variance",
    "vix_style_vol",
    "forecast_variance",
    "forecast_vol",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "outcome_value",
]:
    availability.append({
        "column": col,
        "non_null_rows": int(clean_panel[col].notna().sum()),
        "non_null_pct": float(clean_panel[col].notna().mean()),
        "min": clean_panel[col].min(skipna=True),
        "median": clean_panel[col].median(skipna=True),
        "max": clean_panel[col].max(skipna=True),
    })

availability = pd.DataFrame(availability)

bucket_summary = (
    clean_panel
    .groupby("tenor_bucket")
    .agg(
        rows=("date", "size"),
        unique_dates=("date", "nunique"),
        unique_tenors=("tenor", "nunique"),
        outcome_rows=("has_outcome", "sum"),
        eligible_rows=("is_parameter_sweep_eligible", "sum"),
        mean_vrp_log=("vrp_log", "mean"),
        median_vrp_log=("vrp_log", "median"),
        mean_z_3m=("vrp_z_3m", "mean"),
        mean_z_1y=("vrp_z_1y", "mean"),
        mean_rv21d=("rv21d", "mean"),
        mean_rsi14=("rsi14", "mean"),
    )
    .reset_index()
)

print()
print("Clean panel summary:")
display(summary)

print()
print("Column availability / distribution:")
display(availability)

print()
print("Bucket summary:")
display(bucket_summary)

print()
print("Sample clean panel:")
display(clean_panel.head(20))


# -----------------------------
# 9. Save clean panel and audit files
# -----------------------------

safe_start = clean_panel["date"].min().strftime("%Y%m%d")
safe_end = clean_panel["date"].max().strftime("%Y%m%d")

clean_panel_path = (
    PARAM_SWEEP_PROCESSED_DIR
    / f"02_corsi_parameter_sweep_clean_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell1_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

availability_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell1_availability_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

bucket_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell1_bucket_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

clean_panel.to_parquet(clean_panel_path, index=False)
summary.to_csv(summary_path, index=False)
availability.to_csv(availability_path, index=False)
bucket_summary.to_csv(bucket_summary_path, index=False)

print()
print("Cell 1 outputs saved:")
print(f"Clean panel:          {clean_panel_path}")
print(f"Summary:              {summary_path}")
print(f"Availability:         {availability_path}")
print(f"Bucket summary:       {bucket_summary_path}")

print()
print("Cell 1 complete.")

Corsi parameter sweep clean setup
Project root:              C:\Users\patri\vrp_project
Corsi processed dir:        C:\Users\patri\vrp_project\data\processed\forecast_model_corsi_v1
Corsi audit dir:            C:\Users\patri\vrp_project\data\audit\forecast_model_corsi_v1
Sweep processed dir:        C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean
Sweep audit dir:            C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean
Run timestamp:              20260704_085138

Selected Corsi signal panel:
C:\Users\patri\vrp_project\data\processed\forecast_model_corsi_v1\corsi_cell12_oos_forecast_vrp_signal_panel_20200102_20260623_utp_cta_20260703_211350.parquet

Raw Corsi signal panel summary:


,rows,columns,first_date,last_date,unique_dates,unique_tenors
0,14565,65,2020-01-02,2026-06-23,1626,9



Selected outcome discovery file:
C:\Users\patri\vrp_project\data\audit\forecast_model_corsi_v1\corsi_cell11_trade_outcome_file_discovery_20180724_20260610_utp_cta_20260703_211350.csv


,path,rows_est,columns,date_col,trade_date_col,tenor_col,pnl_cols,return_cols,chosen_outcome_col,chosen_outcome_kind,has_outcome_col,coverage_pct_sample,score,eligible_for_backtest,read_error,parse_error
0,C:\Users\patri\vrp_project\data\processed\put_...,18099,55,trade_date,trade_date,tenor,"short_option_pnl_points, normalized_pnl_dollar...",NaN,normalized_pnl_dollars,pnl,True,1.0,122.0,True,NaN,NaN
1,C:\Users\patri\vrp_project\data\processed\put_...,18099,79,trade_date,trade_date,tenor,"short_option_pnl_points, normalized_pnl_dollar...",NaN,normalized_pnl_dollars,pnl,True,1.0,116.0,True,NaN,NaN
2,C:\Users\patri\vrp_project\data\processed\stag...,18144,48,date,trade_date,target_days,NaN,"trailing_num_returns, forward_num_returns, spx...",spx_log_return,return,True,1.0,104.0,True,NaN,NaN
3,C:\Users\patri\vrp_project\data\processed\stag...,18144,48,date,trade_date,target_days,NaN,"trailing_num_returns, forward_num_returns, spx...",spx_log_return,return,True,1.0,104.0,True,NaN,NaN
4,C:\Users\patri\vrp_project\data\processed\stag...,18144,57,date,trade_date,tenor,NaN,"spx_log_return, trailing_num_returns, forward_...",spx_log_return,return,True,1.0,104.0,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,C:\Users\patri\vrp_project\data\audit\locked_2...,682,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,-100.0,False,NaN,NaN
275,C:\Users\patri\vrp_project\data\audit\locked_2...,16,11,NaN,NaN,avg_selected_tenor,"avg_pnl_per_day, total_pnl, worst_trade_pnl, w...",NaN,total_pnl,pnl,True,NaN,-100.0,False,NaN,NaN
276,C:\Users\patri\vrp_project\data\audit\locked_2...,6,10,overlap_trade_dates,NaN,same_selected_tenor_dates,NaN,NaN,NaN,NaN,False,0.0,-100.0,False,NaN,NaN
277,C:\Users\patri\vrp_project\data\audit\locked_2...,2500,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,-100.0,False,NaN,NaN



Chosen outcome file:
C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet
date_col:        trade_date
trade_date_col:  trade_date
tenor_col:       tenor
outcome_col:     normalized_pnl_dollars

Outcome table summary:


...


Bucket summary:


,tenor_bucket,rows,unique_dates,unique_tenors,outcome_rows,eligible_rows,mean_vrp_log,median_vrp_log,mean_z_3m,mean_z_1y,mean_rv21d,mean_rsi14
0,back,4837,1615,3,4815,4440,0.600850,0.549459,0.017983,-0.125573,17.068347,55.683280
1,front,4873,1626,3,4852,4477,0.594017,0.556620,-0.007766,-0.134758,17.046819,55.673881
2,middle,4855,1620,3,4832,4457,0.592551,0.543285,0.011142,-0.129286,17.051588,55.683432



Sample clean panel:


,date,trade_date,tenor,tenor_bucket,implied_variance,vix_style_vol,forecast_model,forecast_variance,forecast_vol,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,outcome_value,has_outcome,spx_close,spx_log_return,is_parameter_sweep_eligible
0,2020-01-02,20200102,9,front,0.011050,10.511905,har_total_simple,0.010737,10.361778,0.028769,NaN,NaN,7.084495,75.483183,4266.617555,True,3257.85,0.008344,False
1,2020-01-02,20200102,12,front,0.012187,11.039263,har_total_simple,0.011029,10.501803,0.099823,NaN,NaN,7.084495,75.483183,6445.968967,True,3257.85,0.008344,False
2,2020-01-02,20200102,15,front,0.012860,11.340214,har_total_simple,0.011279,10.620104,0.131213,NaN,NaN,7.084495,75.483183,6445.968967,True,3257.85,0.008344,False
3,2020-01-02,20200102,18,middle,0.013239,11.505885,har_total_simple,0.011507,10.727108,0.140169,NaN,NaN,7.084495,75.483183,6445.968967,True,3257.85,0.008344,False
4,2020-01-02,20200102,21,middle,0.013509,11.622776,har_total_simple,0.011712,10.822217,0.142731,NaN,NaN,7.084495,75.483183,8256.979296,True,3257.85,0.008344,False
5,2020-01-02,20200102,24,middle,0.014163,11.900700,har_total_simple,0.011904,10.910424,0.173757,NaN,NaN,7.084495,75.483183,8256.979296,True,3257.85,0.008344,False
6,2020-01-02,20200102,27,back,0.014872,12.194919,har_total_simple,0.011945,10.929427,0.219121,NaN,NaN,7.084495,75.483183,1418.113173,True,3257.85,0.008344,False
7,2020-01-02,20200102,30,back,0.015482,12.442864,har_total_simple,0.012117,11.007830,0.245081,NaN,NaN,7.084495,75.483183,1418.113173,True,3257.85,0.008344,False
8,2020-01-02,20200102,33,back,0.016062,12.673517,har_total_simple,0.012283,11.082960,0.268211,NaN,NaN,7.084495,75.483183,12308.731218,True,3257.85,0.008344,False
9,2020-01-03,20200103,9,front,0.016851,12.981207,har_total_simple,0.018259,13.512570,-0.080235,NaN,NaN,7.152270,65.738991,4853.393511,True,3234.85,-0.007085,False



Cell 1 outputs saved:
Clean panel:          C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
Summary:              C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell1_summary_20200102_20260623_20260704_085138.csv
Availability:         C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell1_availability_20200102_20260623_20260704_085138.csv
Bucket summary:       C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell1_bucket_summary_20200102_20260623_20260704_085138.csv

Cell 1 complete.


In [2]:
# Cell 2: Define Phase 1 starting parameter grid only
#
# Approved scope:
#   - Define the Phase 1 Core and Secondary parameter grids.
#   - Enforce bucket-shape constraints.
#   - Save the grid/spec files for later testing.
#
# Not included in this cell:
#   - No outcome testing.
#   - No candidate-pool audit.
#   - No Core/Secondary combination.
#   - No tenor selection.
#   - No sizing.
#   - No final ranking.
#
# Denominator:
#   har_total_simple
#
# Parameters included:
#   VRP log
#   3m z-score
#   1y z-score
#   RSI cap
#   RV21D floor
#
# Phase 1 rule:
#   3m z-score threshold = 1y z-score threshold within each bucket.

from pathlib import Path
import itertools
import pandas as pd


# -----------------------------
# 1. Paths
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

print("Cell 2: Define Phase 1 starting parameter grid only")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Audit dir:         {PARAM_SWEEP_AUDIT_DIR}")
print(f"Run timestamp:     {RUN_TIMESTAMP}")


# -----------------------------
# 2. Approved Phase 1 grid
# -----------------------------
#
# Format:
#   (Front, Middle, Back)
#
# Rules:
#   VRP log:    Front <= Middle <= Back
#   z-score:    Front <= Middle <= Back
#   RV21D:      Front <= Middle <= Back
#   RSI cap:    Front >= Middle >= Back
#
# Phase 1:
#   z_3m threshold = z_1y threshold within each bucket


# ===== Core grid =====

CORE_VRP_TRIPLES = [
    (0.70, 0.80, 0.90),
    (0.80, 0.90, 1.00),
    (0.80, 1.00, 1.10),
    (0.90, 1.00, 1.20),
]

CORE_Z_TRIPLES = [
    (0.50, 0.75, 0.75),
    (0.50, 0.75, 1.00),
    (0.75, 1.00, 1.25),
    (1.00, 1.00, 1.50),
]

CORE_RSI_TRIPLES = [
    (72, 70, 68),
    (70, 68, 66),
    (68, 66, 64),
]

CORE_RV21D_TRIPLES = [
    (6.5, 6.5, 6.5),
    (7.5, 7.5, 7.5),
    (8.5, 8.5, 8.5),
    (8.5, 9.0, 10.0),
]


# ===== Secondary grid =====

SECONDARY_VRP_TRIPLES = [
    (0.50, 0.60, 0.70),
    (0.60, 0.70, 0.80),
    (0.60, 0.80, 0.90),
    (0.70, 0.80, 1.00),
]

SECONDARY_Z_TRIPLES = [
    (0.25, 0.50, 0.50),
    (0.25, 0.50, 0.75),
    (0.50, 0.75, 1.00),
    (0.75, 1.00, 1.25),
]

SECONDARY_RSI_TRIPLES = [
    (76, 74, 72),
    (74, 72, 70),
    (72, 70, 68),
]

SECONDARY_RV21D_TRIPLES = [
    (5.5, 5.5, 5.5),
    (6.5, 6.5, 6.5),
    (7.5, 7.5, 7.5),
    (6.5, 7.5, 8.5),
]


# -----------------------------
# 3. Validation helpers
# -----------------------------

def validate_bucket_shape(vrp_triple, z_triple, rsi_triple, rv21d_triple):
    """
    Enforce approved bucket-shape rules.

    Richness / strictness:
        Front <= Middle <= Back

    RSI cap:
        Front >= Middle >= Back
    """
    checks = {
        "vrp_front_le_middle_le_back": vrp_triple[0] <= vrp_triple[1] <= vrp_triple[2],
        "z_front_le_middle_le_back": z_triple[0] <= z_triple[1] <= z_triple[2],
        "rv21d_front_le_middle_le_back": rv21d_triple[0] <= rv21d_triple[1] <= rv21d_triple[2],
        "rsi_front_ge_middle_ge_back": rsi_triple[0] >= rsi_triple[1] >= rsi_triple[2],
    }

    return all(checks.values()), checks


def build_layer_specs(
    role,
    vrp_triples,
    z_triples,
    rsi_triples,
    rv21d_triples,
):
    rows = []

    for spec_num, (vrp, z, rsi, rv21d) in enumerate(
        itertools.product(vrp_triples, z_triples, rsi_triples, rv21d_triples),
        start=1,
    ):
        is_valid, checks = validate_bucket_shape(vrp, z, rsi, rv21d)

        if not is_valid:
            raise ValueError(
                f"Invalid {role} spec #{spec_num}: "
                f"vrp={vrp}, z={z}, rsi={rsi}, rv21d={rv21d}, checks={checks}"
            )

        spec_id = (
            f"{role.lower()}_phase1_"
            f"vrp_{vrp[0]:.2f}_{vrp[1]:.2f}_{vrp[2]:.2f}_"
            f"z_{z[0]:.2f}_{z[1]:.2f}_{z[2]:.2f}_"
            f"rsi_{int(rsi[0])}_{int(rsi[1])}_{int(rsi[2])}_"
            f"rv21d_{rv21d[0]:.1f}_{rv21d[1]:.1f}_{rv21d[2]:.1f}"
        )

        rows.append({
            "spec_id": spec_id,
            "role": role,
            "phase": "phase1_starting_grid",
            "forecast_denominator": "har_total_simple",

            "front_vrp_log_min": vrp[0],
            "middle_vrp_log_min": vrp[1],
            "back_vrp_log_min": vrp[2],

            "front_z_3m_min": z[0],
            "middle_z_3m_min": z[1],
            "back_z_3m_min": z[2],

            "front_z_1y_min": z[0],
            "middle_z_1y_min": z[1],
            "back_z_1y_min": z[2],

            "front_rsi14_max": rsi[0],
            "middle_rsi14_max": rsi[1],
            "back_rsi14_max": rsi[2],

            "front_rv21d_min": rv21d[0],
            "middle_rv21d_min": rv21d[1],
            "back_rv21d_min": rv21d[2],

            "phase1_z_3m_equals_z_1y": True,
        })

    return pd.DataFrame(rows)


# -----------------------------
# 4. Build approved specs
# -----------------------------

core_specs = build_layer_specs(
    role="Core",
    vrp_triples=CORE_VRP_TRIPLES,
    z_triples=CORE_Z_TRIPLES,
    rsi_triples=CORE_RSI_TRIPLES,
    rv21d_triples=CORE_RV21D_TRIPLES,
)

secondary_specs = build_layer_specs(
    role="Secondary",
    vrp_triples=SECONDARY_VRP_TRIPLES,
    z_triples=SECONDARY_Z_TRIPLES,
    rsi_triples=SECONDARY_RSI_TRIPLES,
    rv21d_triples=SECONDARY_RV21D_TRIPLES,
)

all_specs = pd.concat([core_specs, secondary_specs], ignore_index=True)


# -----------------------------
# 5. Summary
# -----------------------------

grid_summary = pd.DataFrame([
    {
        "role": "Core",
        "vrp_triples": len(CORE_VRP_TRIPLES),
        "z_triples": len(CORE_Z_TRIPLES),
        "rsi_triples": len(CORE_RSI_TRIPLES),
        "rv21d_triples": len(CORE_RV21D_TRIPLES),
        "total_specs": len(core_specs),
    },
    {
        "role": "Secondary",
        "vrp_triples": len(SECONDARY_VRP_TRIPLES),
        "z_triples": len(SECONDARY_Z_TRIPLES),
        "rsi_triples": len(SECONDARY_RSI_TRIPLES),
        "rv21d_triples": len(SECONDARY_RV21D_TRIPLES),
        "total_specs": len(secondary_specs),
    },
    {
        "role": "Total",
        "vrp_triples": None,
        "z_triples": None,
        "rsi_triples": None,
        "rv21d_triples": None,
        "total_specs": len(all_specs),
    },
])

print()
print("Phase 1 grid summary:")
display(grid_summary)

print()
print("Core specs sample:")
display(core_specs.head(10))

print()
print("Secondary specs sample:")
display(secondary_specs.head(10))


# -----------------------------
# 6. Save outputs
# -----------------------------

core_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell2_core_phase1_specs_{RUN_TIMESTAMP}.csv"
)

secondary_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell2_secondary_phase1_specs_{RUN_TIMESTAMP}.csv"
)

all_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_{RUN_TIMESTAMP}.csv"
)

grid_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell2_phase1_grid_summary_{RUN_TIMESTAMP}.csv"
)

core_specs.to_csv(core_specs_path, index=False)
secondary_specs.to_csv(secondary_specs_path, index=False)
all_specs.to_csv(all_specs_path, index=False)
grid_summary.to_csv(grid_summary_path, index=False)

print()
print("Cell 2 outputs saved:")
print(f"Core specs:          {core_specs_path}")
print(f"Secondary specs:     {secondary_specs_path}")
print(f"All specs:           {all_specs_path}")
print(f"Grid summary:        {grid_summary_path}")

print()
print("Cell 2 complete.")

Cell 2: Define Phase 1 starting parameter grid only
Project root:      C:\Users\patri\vrp_project
Audit dir:         C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean
Run timestamp:     20260704_085147

Phase 1 grid summary:


,role,vrp_triples,z_triples,rsi_triples,rv21d_triples,total_specs
0,Core,4.0,4.0,3.0,4.0,192
1,Secondary,4.0,4.0,3.0,4.0,192
2,Total,NaN,NaN,NaN,NaN,384



Core specs sample:


,spec_id,role,phase,forecast_denominator,front_vrp_log_min,middle_vrp_log_min,back_vrp_log_min,front_z_3m_min,middle_z_3m_min,back_z_3m_min,front_z_1y_min,middle_z_1y_min,back_z_1y_min,front_rsi14_max,middle_rsi14_max,back_rsi14_max,front_rv21d_min,middle_rv21d_min,back_rv21d_min,phase1_z_3m_equals_z_1y
0,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,72,70,68,6.5,6.5,6.5,True
1,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,72,70,68,7.5,7.5,7.5,True
2,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,72,70,68,8.5,8.5,8.5,True
3,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,72,70,68,8.5,9.0,10.0,True
4,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,70,68,66,6.5,6.5,6.5,True
5,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,70,68,66,7.5,7.5,7.5,True
6,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,70,68,66,8.5,8.5,8.5,True
7,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,70,68,66,8.5,9.0,10.0,True
8,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,68,66,64,6.5,6.5,6.5,True
9,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,Core,phase1_starting_grid,har_total_simple,0.7,0.8,0.9,0.5,0.75,0.75,0.5,0.75,0.75,68,66,64,7.5,7.5,7.5,True



Secondary specs sample:


,spec_id,role,phase,forecast_denominator,front_vrp_log_min,middle_vrp_log_min,back_vrp_log_min,front_z_3m_min,middle_z_3m_min,back_z_3m_min,front_z_1y_min,middle_z_1y_min,back_z_1y_min,front_rsi14_max,middle_rsi14_max,back_rsi14_max,front_rv21d_min,middle_rv21d_min,back_rv21d_min,phase1_z_3m_equals_z_1y
0,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,76,74,72,5.5,5.5,5.5,True
1,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,76,74,72,6.5,6.5,6.5,True
2,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,76,74,72,7.5,7.5,7.5,True
3,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,76,74,72,6.5,7.5,8.5,True
4,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,74,72,70,5.5,5.5,5.5,True
5,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,74,72,70,6.5,6.5,6.5,True
6,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,74,72,70,7.5,7.5,7.5,True
7,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,74,72,70,6.5,7.5,8.5,True
8,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,72,70,68,5.5,5.5,5.5,True
9,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...,Secondary,phase1_starting_grid,har_total_simple,0.5,0.6,0.7,0.25,0.5,0.5,0.25,0.5,0.5,72,70,68,6.5,6.5,6.5,True



Cell 2 outputs saved:
Core specs:          C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_core_phase1_specs_20260704_085147.csv
Secondary specs:     C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_secondary_phase1_specs_20260704_085147.csv
All specs:           C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_20260704_085147.csv
Grid summary:        C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_phase1_grid_summary_20260704_085147.csv

Cell 2 complete.


In [5]:
# Cell 3: Phase 1 full Core x Secondary sweep using old locked-process selection logic
#
# Approved scope:
#   - Test every Core spec x Secondary spec combination.
#   - Use har_total_simple denominator.
#   - Use all parameters:
#       VRP log
#       3m z-score
#       1y z-score
#       RSI cap
#       RV21D floor
#   - Phase 1 keeps 3m z threshold = 1y z threshold within each bucket.
#   - Core first, Secondary second.
#   - Select at most one tenor per date.
#
# Old-model-aligned tenor selection:
#   1. Compute conditional tenor stats from the relevant candidate universe.
#   2. Find highest conditional win probability among qualifying tenors.
#   3. Keep tenors within 25 bps of best conditional win probability.
#   4. Choose highest conditional average P&L/day.
#   5. Tie-break by conditional aggregate P&L/day.
#   6. Tie-break by conditional win probability.
#   7. Tie-break by longer tenor.
#
# Secondary conditional stats:
#   For each Core x Secondary combination, Secondary stats are calculated only on:
#       dates where that Core spec has no qualifying Core tenor
#       AND that Secondary spec qualifies.
#
# P&L/day:
#   normalized_pnl_dollars / actual_dte if actual_dte is available.
#   Falls back to normalized_pnl_dollars / tenor if actual_dte is unavailable.
#
# Minimum conditional observation rule:
#   Tenors with >= 20 conditional observations are preferred.
#   Low-sample tenors are only considered if all qualifying tenors are low-sample.
#
# Important:
#   This is a full-sample descriptive sweep for parameter discovery / plateau finding.
#   It is not final walk-forward validation.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest files
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

spec_candidates = sorted(
    PARAM_SWEEP_AUDIT_DIR.glob("02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_*.csv")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No clean panel found in {PARAM_SWEEP_PROCESSED_DIR}")

if not spec_candidates:
    raise FileNotFoundError(f"No Cell 2 all-specs file found in {PARAM_SWEEP_AUDIT_DIR}")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]
ALL_SPECS_PATH = spec_candidates[-1]

print("Cell 3: Phase 1 full Core x Secondary sweep")
print(f"Clean panel path: {CLEAN_PANEL_PATH}")
print(f"All specs path:   {ALL_SPECS_PATH}")
print(f"Run timestamp:    {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()
all_specs = pd.read_csv(ALL_SPECS_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_panel_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "has_outcome",
    "is_parameter_sweep_eligible",
]

missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]

if missing_panel_cols:
    raise ValueError(f"Clean panel missing required columns: {missing_panel_cols}")

required_spec_cols = [
    "spec_id",
    "role",
    "forecast_denominator",

    "front_vrp_log_min",
    "middle_vrp_log_min",
    "back_vrp_log_min",

    "front_z_3m_min",
    "middle_z_3m_min",
    "back_z_3m_min",

    "front_z_1y_min",
    "middle_z_1y_min",
    "back_z_1y_min",

    "front_rsi14_max",
    "middle_rsi14_max",
    "back_rsi14_max",

    "front_rv21d_min",
    "middle_rv21d_min",
    "back_rv21d_min",
]

missing_spec_cols = [c for c in required_spec_cols if c not in all_specs.columns]

if missing_spec_cols:
    raise ValueError(f"Specs file missing required columns: {missing_spec_cols}")


# -----------------------------
# 2. Add actual_dte if available
# -----------------------------

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_actual_dte_lookup():
    """
    Locate the same outcome universe used by Cell 1 and pull actual_dte if available.
    """
    outcome_discovery_candidates = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not outcome_discovery_candidates:
        return None, "no_outcome_discovery_file"

    discovery_path = outcome_discovery_candidates[-1]
    discovery = pd.read_csv(discovery_path)
    discovery.columns = [str(c).strip() for c in discovery.columns]

    file_col = "chosen_file" if "chosen_file" in discovery.columns else "path"

    if file_col not in discovery.columns:
        return None, "discovery_missing_path_column"

    preferred = discovery.copy()

    if "eligible_for_backtest" in preferred.columns:
        preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

    if preferred.empty:
        return None, "no_eligible_outcome_files"

    if "score" in preferred.columns:
        preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
    else:
        preferred["_score"] = 0.0

    if "chosen_outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["chosen_outcome_col"].eq("normalized_pnl_dollars")
    elif "outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["outcome_col"].eq("normalized_pnl_dollars")
    else:
        preferred["_preferred_pnl"] = False

    preferred = preferred.sort_values(
        ["_preferred_pnl", "_score"],
        ascending=[False, False],
    )

    for _, row in preferred.iterrows():
        path = Path(row[file_col])

        if not path.exists():
            continue

        try:
            raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
        except Exception:
            continue

        raw.columns = [str(c).strip() for c in raw.columns]

        trade_date_col = first_existing_col(raw, ["trade_date", "trade_dt", "date"])
        tenor_col = first_existing_col(raw, ["tenor", "entry_tenor", "target_tenor"])
        actual_dte_col = first_existing_col(raw, ["actual_dte"])

        if trade_date_col is None or tenor_col is None or actual_dte_col is None:
            continue

        lookup = raw[[trade_date_col, tenor_col, actual_dte_col]].copy()
        lookup = lookup.rename(columns={
            trade_date_col: "trade_date",
            tenor_col: "tenor",
            actual_dte_col: "actual_dte_from_outcomes",
        })

        lookup["trade_date"] = pd.to_numeric(lookup["trade_date"], errors="coerce").astype("Int64")
        lookup["tenor"] = pd.to_numeric(lookup["tenor"], errors="coerce").astype("Int64")
        lookup["actual_dte_from_outcomes"] = pd.to_numeric(
            lookup["actual_dte_from_outcomes"],
            errors="coerce",
        )

        lookup = lookup.dropna(subset=["trade_date", "tenor"])
        lookup["trade_date"] = lookup["trade_date"].astype(int)
        lookup["tenor"] = lookup["tenor"].astype(int)

        lookup = lookup.drop_duplicates(["trade_date", "tenor"])

        return lookup, str(path)

    return None, "actual_dte_not_found_in_outcome_files"


if "actual_dte" not in panel.columns:
    actual_dte_lookup, actual_dte_source = load_actual_dte_lookup()

    if actual_dte_lookup is not None:
        panel = panel.merge(
            actual_dte_lookup,
            on=["trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )
        panel["actual_dte"] = panel["actual_dte_from_outcomes"]
        panel = panel.drop(columns=["actual_dte_from_outcomes"])
    else:
        panel["actual_dte"] = np.nan
else:
    actual_dte_source = "clean_panel_existing_actual_dte"

panel["actual_dte"] = pd.to_numeric(panel["actual_dte"], errors="coerce")

panel["actual_dte_for_pnl_day"] = panel["actual_dte"]

fallback_mask = panel["actual_dte_for_pnl_day"].isna() | panel["actual_dte_for_pnl_day"].le(0)

panel.loc[fallback_mask, "actual_dte_for_pnl_day"] = panel.loc[fallback_mask, "tenor"]

panel["pnl_per_day"] = panel["outcome_value"] / panel["actual_dte_for_pnl_day"]

print()
print("Actual DTE source:")
print(actual_dte_source)

print()
print("Actual DTE availability:")
display(pd.DataFrame([{
    "rows": len(panel),
    "actual_dte_non_null": int(panel["actual_dte"].notna().sum()),
    "actual_dte_non_null_pct": float(panel["actual_dte"].notna().mean()),
    "fallback_to_tenor_rows": int(fallback_mask.sum()),
    "fallback_to_tenor_pct": float(fallback_mask.mean()),
}]))


# -----------------------------
# 3. Build complete eligible matrix panel
# -----------------------------

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "front",
    12: "front",
    15: "front",
    18: "middle",
    21: "middle",
    24: "middle",
    27: "back",
    30: "back",
    33: "back",
}

BUCKETS = ["front", "middle", "back"]
WIN_BAND = 0.0025
MIN_CONDITIONAL_OBS = 20

eligible = panel.loc[panel["is_parameter_sweep_eligible"].astype(bool)].copy()
eligible = eligible.replace([np.inf, -np.inf], np.nan)

needed_for_matrix = [
    "trade_date",
    "date",
    "tenor",
    "tenor_bucket",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "actual_dte_for_pnl_day",
    "pnl_per_day",
]

eligible = eligible.dropna(subset=needed_for_matrix).copy()

eligible = eligible.loc[
    eligible["tenor"].isin(TENORS)
    & eligible["actual_dte_for_pnl_day"].gt(0)
].copy()

duplicate_rows = eligible.duplicated(["trade_date", "tenor"]).sum()

if duplicate_rows:
    raise ValueError(f"Eligible panel has duplicate trade_date x tenor rows: {duplicate_rows}")

date_counts = eligible.groupby("trade_date")["tenor"].nunique()

complete_dates = date_counts.loc[date_counts.eq(len(TENORS))].index.tolist()

eligible = eligible.loc[eligible["trade_date"].isin(complete_dates)].copy()

eligible_dates = sorted(eligible["trade_date"].unique())
num_dates = len(eligible_dates)
num_tenors = len(TENORS)

date_lookup = (
    eligible[["trade_date", "date"]]
    .drop_duplicates("trade_date")
    .set_index("trade_date")
    .loc[eligible_dates]
)

date_array = date_lookup["date"].to_numpy()

def pivot_matrix(col, dtype=float):
    mat = (
        eligible
        .pivot(index="trade_date", columns="tenor", values=col)
        .reindex(index=eligible_dates, columns=TENORS)
    )

    if mat.isna().any().any():
        missing = int(mat.isna().sum().sum())
        raise ValueError(f"Matrix for {col} has missing cells after complete-date filter: {missing}")

    return mat.to_numpy(dtype=dtype)

vrp_mat = pivot_matrix("vrp_log")
z3m_mat = pivot_matrix("vrp_z_3m")
z1y_mat = pivot_matrix("vrp_z_1y")
rsi_mat = pivot_matrix("rsi14")
rv21d_mat = pivot_matrix("rv21d")
pnl_mat = pivot_matrix("outcome_value")
actual_dte_mat = pivot_matrix("actual_dte_for_pnl_day")
pnl_per_day_mat = pivot_matrix("pnl_per_day")

win_mat = (pnl_mat > 0).astype(float)
valid_outcome_mat = np.isfinite(pnl_mat) & np.isfinite(pnl_per_day_mat) & np.isfinite(actual_dte_mat)

tenor_array = np.array(TENORS, dtype=int)
tenor_bucket_array = np.array([TENOR_BUCKET_MAP[int(t)] for t in tenor_array], dtype=object)

core_specs = all_specs.loc[all_specs["role"].eq("Core")].copy().reset_index(drop=True)
secondary_specs = all_specs.loc[all_specs["role"].eq("Secondary")].copy().reset_index(drop=True)

print()
print("Cell 3 input summary:")
display(pd.DataFrame([{
    "eligible_rows_after_complete_date_filter": len(eligible),
    "eligible_dates": num_dates,
    "first_date": pd.to_datetime(date_array.min()),
    "last_date": pd.to_datetime(date_array.max()),
    "unique_tenors": num_tenors,
    "core_specs": len(core_specs),
    "secondary_specs": len(secondary_specs),
    "full_model_combinations": len(core_specs) * len(secondary_specs),
    "win_band_bps": WIN_BAND * 10000,
    "min_conditional_obs": MIN_CONDITIONAL_OBS,
    "forecast_model": ", ".join(sorted(eligible["forecast_model"].astype(str).unique())),
}]))


# -----------------------------
# 4. Matrix helper functions
# -----------------------------

def threshold_array_for_spec(spec_row, field_suffix):
    """
    Example field_suffix:
        vrp_log_min
        z_3m_min
        z_1y_min
        rsi14_max
        rv21d_min
    """
    vals = []

    for tenor in TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]
        vals.append(float(spec_row[f"{bucket}_{field_suffix}"]))

    return np.array(vals, dtype=float)


def build_pass_matrix(spec_row):
    vrp_threshold = threshold_array_for_spec(spec_row, "vrp_log_min")
    z3m_threshold = threshold_array_for_spec(spec_row, "z_3m_min")
    z1y_threshold = threshold_array_for_spec(spec_row, "z_1y_min")
    rsi_cap = threshold_array_for_spec(spec_row, "rsi14_max")
    rv21d_floor = threshold_array_for_spec(spec_row, "rv21d_min")

    return (
        (vrp_mat >= vrp_threshold[None, :])
        & (z3m_mat >= z3m_threshold[None, :])
        & (z1y_mat >= z1y_threshold[None, :])
        & (rsi_mat <= rsi_cap[None, :])
        & (rv21d_mat >= rv21d_floor[None, :])
        & valid_outcome_mat
    )


def compute_conditional_stats(pass_matrix, date_mask, layer, spec_id, core_spec_id=None, secondary_spec_id=None):
    rows = []

    date_mask = np.asarray(date_mask, dtype=bool)

    for col_idx, tenor in enumerate(TENORS):
        candidate_mask = pass_matrix[:, col_idx] & date_mask & valid_outcome_mat[:, col_idx]
        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            row = {
                "layer": layer,
                "spec_id": spec_id,
                "core_spec_id": core_spec_id,
                "secondary_spec_id": secondary_spec_id,
                "tenor": int(tenor),
                "tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
                "conditional_obs": 0,
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_pnl_per_trade": np.nan,
                "conditional_total_pnl": 0.0,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_sample_ok": False,
            }
            rows.append(row)
            continue

        pnl = pnl_mat[candidate_mask, col_idx]
        pnl_day = pnl_per_day_mat[candidate_mask, col_idx]
        dte = actual_dte_mat[candidate_mask, col_idx]

        total_dte = np.nansum(dte)

        if total_dte > 0:
            aggregate_pnl_per_day = np.nansum(pnl) / total_dte
        else:
            aggregate_pnl_per_day = np.nan

        row = {
            "layer": layer,
            "spec_id": spec_id,
            "core_spec_id": core_spec_id,
            "secondary_spec_id": secondary_spec_id,
            "tenor": int(tenor),
            "tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "conditional_obs": candidate_count,
            "conditional_win_probability": float(np.nanmean(pnl > 0)),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_day)),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_day)),
            "conditional_aggregate_pnl_per_day": float(aggregate_pnl_per_day),
            "conditional_avg_pnl_per_trade": float(np.nanmean(pnl)),
            "conditional_total_pnl": float(np.nansum(pnl)),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl)),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_day)),
            "conditional_avg_actual_dte": float(np.nanmean(dte)),
            "conditional_sample_ok": candidate_count >= MIN_CONDITIONAL_OBS,
        }

        rows.append(row)

    return pd.DataFrame(rows)


def select_cols_from_pass_and_stats(pass_matrix, stats_df, date_mask):
    """
    Vectorized old-model-aligned selector.

    Ranking:
      1. Prefer tenors with conditional_obs >= MIN_CONDITIONAL_OBS.
      2. Find best conditional win probability among qualifying tenors.
      3. Keep tenors within WIN_BAND of that best win probability.
      4. Select highest conditional average P&L/day.
      5. Tie-break by conditional aggregate P&L/day.
      6. Tie-break by conditional win probability.
      7. Tie-break by longer tenor.
    """
    date_mask = np.asarray(date_mask, dtype=bool)

    active_pass = pass_matrix & date_mask[:, None]

    stats_df = stats_df.sort_values("tenor").reset_index(drop=True)

    conditional_obs = stats_df["conditional_obs"].to_numpy(dtype=float)
    conditional_win = stats_df["conditional_win_probability"].to_numpy(dtype=float)
    conditional_avg_pnl_day = stats_df["conditional_avg_pnl_per_day"].to_numpy(dtype=float)
    conditional_agg_pnl_day = stats_df["conditional_aggregate_pnl_per_day"].to_numpy(dtype=float)

    has_stats = (
        (conditional_obs > 0)
        & np.isfinite(conditional_win)
        & np.isfinite(conditional_avg_pnl_day)
        & np.isfinite(conditional_agg_pnl_day)
    )

    sample_ok = conditional_obs >= MIN_CONDITIONAL_OBS

    active = active_pass & has_stats[None, :]

    active_sample_ok = active & sample_ok[None, :]
    row_has_sample_ok = active_sample_ok.any(axis=1)

    ranking_universe = np.where(
        row_has_sample_ok[:, None],
        active_sample_ok,
        active,
    )

    win_masked = np.where(
        ranking_universe,
        conditional_win[None, :],
        -np.inf,
    )

    best_win = win_masked.max(axis=1)
    row_has_any = np.isfinite(best_win)

    inside_win_band = (
        ranking_universe
        & row_has_any[:, None]
        & (conditional_win[None, :] >= (best_win[:, None] - WIN_BAND))
    )

    selected_col = np.full(num_dates, -1, dtype=int)

    best_avg = np.full(num_dates, -np.inf)
    best_agg = np.full(num_dates, -np.inf)
    best_win_for_tie = np.full(num_dates, -np.inf)
    best_tenor = np.full(num_dates, -np.inf)

    for col_idx, tenor in enumerate(TENORS):
        cand = inside_win_band[:, col_idx]

        avg_val = conditional_avg_pnl_day[col_idx]
        agg_val = conditional_agg_pnl_day[col_idx]
        win_val = conditional_win[col_idx]
        tenor_val = float(tenor)

        better = (
            cand
            & (
                (avg_val > best_avg)
                | (
                    (avg_val == best_avg)
                    & (agg_val > best_agg)
                )
                | (
                    (avg_val == best_avg)
                    & (agg_val == best_agg)
                    & (win_val > best_win_for_tie)
                )
                | (
                    (avg_val == best_avg)
                    & (agg_val == best_agg)
                    & (win_val == best_win_for_tie)
                    & (tenor_val > best_tenor)
                )
            )
        )

        selected_col[better] = col_idx
        best_avg[better] = avg_val
        best_agg[better] = agg_val
        best_win_for_tie[better] = win_val
        best_tenor[better] = tenor_val

    return selected_col


def profit_factor(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    gains = values[values > 0].sum()
    losses = values[values < 0].sum()

    if losses < 0:
        return gains / abs(losses)

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def summarize_selected_cols(selected_cols, prefix):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    if date_idx.size == 0:
        return {
            f"{prefix}_trades": 0,
            f"{prefix}_frequency": 0.0,
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_day_weighted_pnl_per_day": np.nan,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade_pnl": np.nan,
            f"{prefix}_worst_pnl_per_day": np.nan,
            f"{prefix}_best_trade_pnl": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_avg_selected_tenor": np.nan,
        }

    cols = selected_cols[date_idx]

    pnl = pnl_mat[date_idx, cols]
    pnl_day = pnl_per_day_mat[date_idx, cols]
    dte = actual_dte_mat[date_idx, cols]
    selected_tenors = tenor_array[cols]

    total_dte = np.nansum(dte)

    if total_dte > 0:
        exposure_day_weighted_pnl_per_day = np.nansum(pnl) / total_dte
    else:
        exposure_day_weighted_pnl_per_day = np.nan

    return {
        f"{prefix}_trades": int(date_idx.size),
        f"{prefix}_frequency": float(date_idx.size / num_dates) if num_dates else np.nan,
        f"{prefix}_win_rate": float(np.nanmean(pnl > 0)),
        f"{prefix}_total_pnl": float(np.nansum(pnl)),
        f"{prefix}_avg_pnl_per_trade": float(np.nanmean(pnl)),
        f"{prefix}_avg_pnl_per_day": float(np.nanmean(pnl_day)),
        f"{prefix}_exposure_day_weighted_pnl_per_day": float(exposure_day_weighted_pnl_per_day),
        f"{prefix}_profit_factor": float(profit_factor(pnl)),
        f"{prefix}_worst_trade_pnl": float(np.nanmin(pnl)),
        f"{prefix}_worst_pnl_per_day": float(np.nanmin(pnl_day)),
        f"{prefix}_best_trade_pnl": float(np.nanmax(pnl)),
        f"{prefix}_avg_actual_dte": float(np.nanmean(dte)),
        f"{prefix}_avg_selected_tenor": float(np.nanmean(selected_tenors)),
    }


def bucket_metrics(selected_cols):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    out = {}

    for bucket in BUCKETS:
        out[f"{bucket}_selected_trades"] = 0
        out[f"{bucket}_selected_win_rate"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_total_pnl"] = 0.0

    if date_idx.size == 0:
        return out

    cols = selected_cols[date_idx]
    selected_buckets = tenor_bucket_array[cols]

    for bucket in BUCKETS:
        bucket_mask = selected_buckets == bucket

        if not bucket_mask.any():
            continue

        bucket_date_idx = date_idx[bucket_mask]
        bucket_cols = cols[bucket_mask]

        pnl = pnl_mat[bucket_date_idx, bucket_cols]
        pnl_day = pnl_per_day_mat[bucket_date_idx, bucket_cols]

        out[f"{bucket}_selected_trades"] = int(bucket_mask.sum())
        out[f"{bucket}_selected_win_rate"] = float(np.nanmean(pnl > 0))
        out[f"{bucket}_selected_avg_pnl_per_day"] = float(np.nanmean(pnl_day))
        out[f"{bucket}_selected_total_pnl"] = float(np.nansum(pnl))

    return out


# -----------------------------
# 5. Precompute Core selections and Secondary pass matrices
# -----------------------------

core_objects = {}
core_layer_summary_rows = []
core_conditional_stats_rows = []

print()
print("Building Core pass matrices, conditional stats, and selections...")

all_dates_mask = np.ones(num_dates, dtype=bool)

for i, (_, spec_row) in enumerate(core_specs.iterrows(), start=1):
    spec_id = spec_row["spec_id"]

    pass_matrix = build_pass_matrix(spec_row)
    core_any_date = pass_matrix.any(axis=1)

    stats = compute_conditional_stats(
        pass_matrix=pass_matrix,
        date_mask=all_dates_mask,
        layer="Core",
        spec_id=spec_id,
        core_spec_id=spec_id,
        secondary_spec_id=None,
    )

    selected_cols = select_cols_from_pass_and_stats(
        pass_matrix=pass_matrix,
        stats_df=stats,
        date_mask=all_dates_mask,
    )

    core_objects[spec_id] = {
        "pass_matrix": pass_matrix,
        "core_any_date": core_any_date,
        "selected_cols": selected_cols,
        "stats": stats,
    }

    core_summary = {
        "spec_id": spec_id,
        "role": "Core",
        "candidate_dates": int(core_any_date.sum()),
        "candidate_frequency": float(core_any_date.sum() / num_dates) if num_dates else np.nan,
        **summarize_selected_cols(selected_cols, "core_layer"),
    }

    core_layer_summary_rows.append(core_summary)
    core_conditional_stats_rows.append(stats)

    if i % 25 == 0 or i == len(core_specs):
        print(f"Core specs processed: {i:,} / {len(core_specs):,}")

core_layer_summary = pd.DataFrame(core_layer_summary_rows)
core_conditional_stats = pd.concat(core_conditional_stats_rows, ignore_index=True)

print()
print("Building Secondary pass matrices...")

secondary_pass_matrices = {}

for i, (_, spec_row) in enumerate(secondary_specs.iterrows(), start=1):
    spec_id = spec_row["spec_id"]
    secondary_pass_matrices[spec_id] = build_pass_matrix(spec_row)

    if i % 25 == 0 or i == len(secondary_specs):
        print(f"Secondary specs processed: {i:,} / {len(secondary_specs):,}")

print()
print("Core layer summary:")

core_layer_summary_display = pd.DataFrame([{
    "specs": core_layer_summary["spec_id"].nunique(),

    "min_candidate_frequency": core_layer_summary["candidate_frequency"].min(),
    "median_candidate_frequency": core_layer_summary["candidate_frequency"].median(),
    "max_candidate_frequency": core_layer_summary["candidate_frequency"].max(),

    "min_core_layer_win_rate": core_layer_summary["core_layer_win_rate"].min(),
    "median_core_layer_win_rate": core_layer_summary["core_layer_win_rate"].median(),
    "max_core_layer_win_rate": core_layer_summary["core_layer_win_rate"].max(),

    "min_core_layer_avg_pnl_per_day": core_layer_summary["core_layer_avg_pnl_per_day"].min(),
    "median_core_layer_avg_pnl_per_day": core_layer_summary["core_layer_avg_pnl_per_day"].median(),
    "max_core_layer_avg_pnl_per_day": core_layer_summary["core_layer_avg_pnl_per_day"].max(),
}])

display(core_layer_summary_display)


# -----------------------------
# 6. Full Core x Secondary model sweep
# -----------------------------

model_summary_rows = []
secondary_conditional_stats_rows = []

core_ids = core_specs["spec_id"].tolist()
secondary_ids = secondary_specs["spec_id"].tolist()

total_combinations = len(core_ids) * len(secondary_ids)
combo_counter = 0

print()
print(f"Running full model combinations: {total_combinations:,}")

for core_id in core_ids:
    core_obj = core_objects[core_id]

    core_selected_cols = core_obj["selected_cols"]
    core_any_date = core_obj["core_any_date"]

    no_core_date_mask = ~core_any_date

    for secondary_id in secondary_ids:
        combo_counter += 1

        secondary_pass_matrix = secondary_pass_matrices[secondary_id]

        secondary_stats = compute_conditional_stats(
            pass_matrix=secondary_pass_matrix,
            date_mask=no_core_date_mask,
            layer="Secondary",
            spec_id=secondary_id,
            core_spec_id=core_id,
            secondary_spec_id=secondary_id,
        )

        secondary_selected_cols = select_cols_from_pass_and_stats(
            pass_matrix=secondary_pass_matrix,
            stats_df=secondary_stats,
            date_mask=no_core_date_mask,
        )

        combined_selected_cols = np.where(
            core_selected_cols >= 0,
            core_selected_cols,
            secondary_selected_cols,
        )

        secondary_candidate_date_count = int((secondary_pass_matrix & no_core_date_mask[:, None]).any(axis=1).sum())

        row = {
            "core_spec_id": core_id,
            "secondary_spec_id": secondary_id,

            "secondary_candidate_dates_after_core_filter": secondary_candidate_date_count,
            "secondary_candidate_frequency_after_core_filter": (
                float(secondary_candidate_date_count / num_dates) if num_dates else np.nan
            ),

            **summarize_selected_cols(combined_selected_cols, "overall"),
            **summarize_selected_cols(core_selected_cols, "core"),
            **summarize_selected_cols(secondary_selected_cols, "secondary"),
            **bucket_metrics(combined_selected_cols),
        }

        model_summary_rows.append(row)

        # Save secondary conditional stats for audit.
        # This is combo-dependent because Secondary can only operate where that Core spec has no pass.
        secondary_conditional_stats_rows.append(secondary_stats)

    if combo_counter % (25 * len(secondary_ids)) == 0 or combo_counter == total_combinations:
        print(f"Full models processed: {combo_counter:,} / {total_combinations:,}")

model_summary = pd.DataFrame(model_summary_rows)

secondary_conditional_stats = pd.concat(
    secondary_conditional_stats_rows,
    ignore_index=True,
)

conditional_stats = pd.concat(
    [core_conditional_stats, secondary_conditional_stats],
    ignore_index=True,
)


# -----------------------------
# 7. Add parameter columns and target flags
# -----------------------------

core_param_cols = [
    c for c in core_specs.columns
    if c not in ["role", "phase", "forecast_denominator"]
]

secondary_param_cols = [
    c for c in secondary_specs.columns
    if c not in ["role", "phase", "forecast_denominator"]
]

core_params_for_merge = core_specs[core_param_cols].copy()
secondary_params_for_merge = secondary_specs[secondary_param_cols].copy()

core_params_for_merge = core_params_for_merge.add_prefix("core_param_")
secondary_params_for_merge = secondary_params_for_merge.add_prefix("secondary_param_")

model_summary = model_summary.merge(
    core_params_for_merge,
    left_on="core_spec_id",
    right_on="core_param_spec_id",
    how="left",
    validate="many_to_one",
)

model_summary = model_summary.merge(
    secondary_params_for_merge,
    left_on="secondary_spec_id",
    right_on="secondary_param_spec_id",
    how="left",
    validate="many_to_one",
)

model_summary["meets_core_win_85"] = model_summary["core_win_rate"].ge(0.85)
model_summary["meets_overall_win_80"] = model_summary["overall_win_rate"].ge(0.80)
model_summary["meets_frequency_33"] = model_summary["overall_frequency"].gt(0.33)

model_summary["meets_all_targets"] = (
    model_summary["meets_core_win_85"]
    & model_summary["meets_overall_win_80"]
    & model_summary["meets_frequency_33"]
)

# Review rank only. This is not a final model decision.
model_summary = model_summary.sort_values(
    [
        "meets_all_targets",
        "overall_win_rate",
        "core_win_rate",
        "overall_avg_pnl_per_day",
        "overall_frequency",
        "overall_total_pnl",
        "overall_worst_trade_pnl",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        False,
        False,
        False,
    ],
).reset_index(drop=True)

model_summary["review_rank"] = np.arange(1, len(model_summary) + 1)


# -----------------------------
# 8. Display review output
# -----------------------------

print()
print("Full model summary:")
display(pd.DataFrame([{
    "full_models": len(model_summary),
    "models_meeting_all_targets": int(model_summary["meets_all_targets"].sum()),
    "models_core_win_85": int(model_summary["meets_core_win_85"].sum()),
    "models_overall_win_80": int(model_summary["meets_overall_win_80"].sum()),
    "models_frequency_33": int(model_summary["meets_frequency_33"].sum()),

    "max_overall_frequency": model_summary["overall_frequency"].max(),
    "median_overall_frequency": model_summary["overall_frequency"].median(),

    "max_overall_win_rate": model_summary["overall_win_rate"].max(),
    "median_overall_win_rate": model_summary["overall_win_rate"].median(),

    "max_core_win_rate": model_summary["core_win_rate"].max(),
    "median_core_win_rate": model_summary["core_win_rate"].median(),

    "max_overall_avg_pnl_per_day": model_summary["overall_avg_pnl_per_day"].max(),
    "median_overall_avg_pnl_per_day": model_summary["overall_avg_pnl_per_day"].median(),
}]))

review_cols = [
    "review_rank",
    "meets_all_targets",

    "overall_trades",
    "overall_frequency",
    "overall_win_rate",
    "overall_avg_pnl_per_day",
    "overall_exposure_day_weighted_pnl_per_day",
    "overall_total_pnl",
    "overall_profit_factor",
    "overall_worst_trade_pnl",
    "overall_worst_pnl_per_day",
    "overall_avg_actual_dte",
    "overall_avg_selected_tenor",

    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_avg_pnl_per_day",
    "core_exposure_day_weighted_pnl_per_day",
    "core_total_pnl",
    "core_worst_trade_pnl",
    "core_avg_actual_dte",
    "core_avg_selected_tenor",

    "secondary_trades",
    "secondary_frequency",
    "secondary_win_rate",
    "secondary_avg_pnl_per_day",
    "secondary_exposure_day_weighted_pnl_per_day",
    "secondary_total_pnl",
    "secondary_worst_trade_pnl",
    "secondary_avg_actual_dte",
    "secondary_avg_selected_tenor",

    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_day",

    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_day",

    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_day",

    "core_spec_id",
    "secondary_spec_id",
]

print()
print("Top full models by review rank:")
display(model_summary[review_cols].head(50))

print()
print("Top models meeting all targets:")
display(
    model_summary
    .loc[model_summary["meets_all_targets"], review_cols]
    .head(50)
)

print()
print("Highest-frequency models with overall win rate >= 80% and core win rate >= 85%:")
display(
    model_summary
    .loc[
        model_summary["overall_win_rate"].ge(0.80)
        & model_summary["core_win_rate"].ge(0.85),
        review_cols
    ]
    .sort_values(
        [
            "overall_frequency",
            "overall_win_rate",
            "overall_avg_pnl_per_day",
            "overall_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(50)
)

print()
print("Highest average P&L/day models meeting all targets:")
display(
    model_summary
    .loc[model_summary["meets_all_targets"], review_cols]
    .sort_values(
        [
            "overall_avg_pnl_per_day",
            "overall_win_rate",
            "overall_frequency",
            "overall_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(50)
)


# -----------------------------
# 9. Save outputs
# -----------------------------

safe_start = pd.to_datetime(date_array.min()).strftime("%Y%m%d")
safe_end = pd.to_datetime(date_array.max()).strftime("%Y%m%d")

core_layer_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_core_layer_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

conditional_stats_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_conditional_stats_old_process_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

model_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_full_model_summary_old_process_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

top_models_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_top_models_old_process_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_models_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_models_meeting_targets_old_process_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

high_frequency_win_models_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell3_high_frequency_win_models_old_process_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_layer_summary.to_csv(core_layer_summary_path, index=False)
conditional_stats.to_csv(conditional_stats_path, index=False)
model_summary.to_csv(model_summary_path, index=False)
model_summary.head(500).to_csv(top_models_path, index=False)

model_summary.loc[
    model_summary["meets_all_targets"]
].to_csv(target_models_path, index=False)

model_summary.loc[
    model_summary["overall_win_rate"].ge(0.80)
    & model_summary["core_win_rate"].ge(0.85)
].sort_values(
    [
        "overall_frequency",
        "overall_win_rate",
        "overall_avg_pnl_per_day",
        "overall_total_pnl",
    ],
    ascending=[False, False, False, False],
).to_csv(high_frequency_win_models_path, index=False)

print()
print("Cell 3 outputs saved:")
print(f"Core layer summary:        {core_layer_summary_path}")
print(f"Conditional stats:         {conditional_stats_path}")
print(f"Full model summary:        {model_summary_path}")
print(f"Top models:                {top_models_path}")
print(f"Target models:             {target_models_path}")
print(f"High frequency win models: {high_frequency_win_models_path}")

print()
print("Cell 3 complete.")

Cell 3: Phase 1 full Core x Secondary sweep
Clean panel path: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
All specs path:   C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_20260704_085147.csv
Run timestamp:    20260704_090941

Actual DTE source:
C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet

Actual DTE availability:


,rows,actual_dte_non_null,actual_dte_non_null_pct,fallback_to_tenor_rows,fallback_to_tenor_pct
0,14565,14565,1.0,0,0.0



Cell 3 input summary:


,eligible_rows_after_complete_date_filter,eligible_dates,first_date,last_date,unique_tenors,core_specs,secondary_specs,full_model_combinations,win_band_bps,min_conditional_obs,forecast_model
0,13302,1478,2020-07-01,2026-05-19,9,192,192,36864,25.0,20,har_total_simple



Building Core pass matrices, conditional stats, and selections...
Core specs processed: 25 / 192
Core specs processed: 50 / 192
Core specs processed: 75 / 192
Core specs processed: 100 / 192
Core specs processed: 125 / 192
Core specs processed: 150 / 192
Core specs processed: 175 / 192
Core specs processed: 192 / 192

Building Secondary pass matrices...
Secondary specs processed: 25 / 192
Secondary specs processed: 50 / 192
Secondary specs processed: 75 / 192
Secondary specs processed: 100 / 192
Secondary specs processed: 125 / 192
Secondary specs processed: 150 / 192
Secondary specs processed: 175 / 192
Secondary specs processed: 192 / 192

Core layer summary:


...


Highest-frequency models with overall win rate >= 80% and core win rate >= 85%:


,review_rank,meets_all_targets,overall_trades,overall_frequency,overall_win_rate,overall_avg_pnl_per_day,overall_exposure_day_weighted_pnl_per_day,overall_total_pnl,overall_profit_factor,overall_worst_trade_pnl,...,front_selected_win_rate,front_selected_avg_pnl_per_day,middle_selected_trades,middle_selected_win_rate,middle_selected_avg_pnl_per_day,back_selected_trades,back_selected_win_rate,back_selected_avg_pnl_per_day,core_spec_id,secondary_spec_id



Highest average P&L/day models meeting all targets:


,review_rank,meets_all_targets,overall_trades,overall_frequency,overall_win_rate,overall_avg_pnl_per_day,overall_exposure_day_weighted_pnl_per_day,overall_total_pnl,overall_profit_factor,overall_worst_trade_pnl,...,front_selected_win_rate,front_selected_avg_pnl_per_day,middle_selected_trades,middle_selected_win_rate,middle_selected_avg_pnl_per_day,back_selected_trades,back_selected_win_rate,back_selected_avg_pnl_per_day,core_spec_id,secondary_spec_id



Cell 3 outputs saved:
Core layer summary:        C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_core_layer_summary_20200701_20260519_20260704_090941.csv
Conditional stats:         C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_conditional_stats_old_process_20200701_20260519_20260704_090941.csv
Full model summary:        C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_full_model_summary_old_process_20200701_20260519_20260704_090941.csv
Top models:                C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_top_models_old_process_20200701_20260519_20260704_090941.csv
Target models:             C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_models_meeting_targets_old_process_20200701_20260519_20260704_090941.csv

In [6]:
# Cell 4: Phase 1 diagnostic breakdown
#
# Approved scope:
#   Diagnose Cell 3 results only.
#
# This cell answers:
#   1. Which Core specs came closest to 85% win rate?
#   2. Which parameter triples helped or hurt Core win rate?
#   3. Is the Core problem front / middle / back?
#   4. Which full models nearly met targets?
#   5. How did top models perform by year?
#   6. Where are losses concentrated?
#
# This cell does NOT:
#   - Create new parameter grids.
#   - Change thresholds.
#   - Lock a model.
#   - Run sizing.
#   - Run walk-forward validation.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest files
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

spec_candidates = sorted(
    PARAM_SWEEP_AUDIT_DIR.glob("02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_*.csv")
)

model_summary_candidates = sorted(
    PARAM_SWEEP_AUDIT_DIR.glob("02_corsi_parameter_sweep_clean_cell3_full_model_summary_old_process_*.csv")
)

core_layer_summary_candidates = sorted(
    PARAM_SWEEP_AUDIT_DIR.glob("02_corsi_parameter_sweep_clean_cell3_core_layer_summary_*.csv")
)

conditional_stats_candidates = sorted(
    PARAM_SWEEP_AUDIT_DIR.glob("02_corsi_parameter_sweep_clean_cell3_conditional_stats_old_process_*.csv")
)

if not clean_panel_candidates:
    raise FileNotFoundError("No clean panel found.")

if not spec_candidates:
    raise FileNotFoundError("No Cell 2 specs file found.")

if not model_summary_candidates:
    raise FileNotFoundError("No Cell 3 full model summary file found.")

if not core_layer_summary_candidates:
    raise FileNotFoundError("No Cell 3 core layer summary file found.")

if not conditional_stats_candidates:
    raise FileNotFoundError("No Cell 3 conditional stats file found.")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]
ALL_SPECS_PATH = spec_candidates[-1]
MODEL_SUMMARY_PATH = model_summary_candidates[-1]
CORE_LAYER_SUMMARY_PATH = core_layer_summary_candidates[-1]
CONDITIONAL_STATS_PATH = conditional_stats_candidates[-1]

print("Cell 4: Phase 1 diagnostic breakdown")
print(f"Clean panel:             {CLEAN_PANEL_PATH}")
print(f"All specs:               {ALL_SPECS_PATH}")
print(f"Cell 3 model summary:    {MODEL_SUMMARY_PATH}")
print(f"Cell 3 core summary:     {CORE_LAYER_SUMMARY_PATH}")
print(f"Cell 3 conditional stats:{CONDITIONAL_STATS_PATH}")
print(f"Run timestamp:           {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()
all_specs = pd.read_csv(ALL_SPECS_PATH).copy()
model_summary = pd.read_csv(MODEL_SUMMARY_PATH).copy()
core_layer_summary = pd.read_csv(CORE_LAYER_SUMMARY_PATH).copy()
conditional_stats = pd.read_csv(CONDITIONAL_STATS_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

core_specs = all_specs.loc[all_specs["role"].eq("Core")].copy()
secondary_specs = all_specs.loc[all_specs["role"].eq("Secondary")].copy()

print()
print("Loaded input summary:")
display(pd.DataFrame([{
    "clean_panel_rows": len(panel),
    "all_specs": len(all_specs),
    "core_specs": len(core_specs),
    "secondary_specs": len(secondary_specs),
    "full_models": len(model_summary),
    "core_layer_summary_rows": len(core_layer_summary),
    "conditional_stats_rows": len(conditional_stats),
}]))


# -----------------------------
# 2. Merge Core params onto Core layer summary
# -----------------------------

core_diag = core_layer_summary.merge(
    core_specs,
    on="spec_id",
    how="left",
    validate="one_to_one",
)

# Useful composite labels for grouping.
core_diag["vrp_triple"] = (
    core_diag["front_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_diag["middle_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_diag["back_vrp_log_min"].map("{:.2f}".format)
)

core_diag["z_triple"] = (
    core_diag["front_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_diag["middle_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_diag["back_z_3m_min"].map("{:.2f}".format)
)

core_diag["rsi_triple"] = (
    core_diag["front_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_diag["middle_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_diag["back_rsi14_max"].astype(int).astype(str)
)

core_diag["rv21d_triple"] = (
    core_diag["front_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_diag["middle_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_diag["back_rv21d_min"].map("{:.1f}".format)
)

core_diag["core_win_gap_to_85"] = 0.85 - core_diag["core_layer_win_rate"]
core_diag["core_win_80_plus"] = core_diag["core_layer_win_rate"].ge(0.80)
core_diag["core_win_82_plus"] = core_diag["core_layer_win_rate"].ge(0.82)
core_diag["core_win_85_plus"] = core_diag["core_layer_win_rate"].ge(0.85)


# -----------------------------
# 3. Core specs closest to 85%
# -----------------------------

core_review_cols = [
    "spec_id",
    "candidate_frequency",
    "core_layer_trades",
    "core_layer_frequency",
    "core_layer_win_rate",
    "core_win_gap_to_85",
    "core_layer_avg_pnl_per_day",
    "core_layer_exposure_day_weighted_pnl_per_day",
    "core_layer_total_pnl",
    "core_layer_profit_factor",
    "core_layer_worst_trade_pnl",
    "core_layer_worst_pnl_per_day",
    "core_layer_avg_actual_dte",
    "core_layer_avg_selected_tenor",
    "vrp_triple",
    "z_triple",
    "rsi_triple",
    "rv21d_triple",
]

print()
print("Core specs closest to 85% win rate:")
display(
    core_diag[core_review_cols]
    .sort_values(
        [
            "core_layer_win_rate",
            "core_layer_avg_pnl_per_day",
            "core_layer_frequency",
            "core_layer_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(50)
)


# -----------------------------
# 4. Core parameter-group diagnostics
# -----------------------------

def parameter_group_summary(df, group_col):
    out = (
        df
        .groupby(group_col)
        .agg(
            specs=("spec_id", "nunique"),
            max_core_win_rate=("core_layer_win_rate", "max"),
            median_core_win_rate=("core_layer_win_rate", "median"),
            min_core_win_gap_to_85=("core_win_gap_to_85", "min"),
            max_core_frequency=("core_layer_frequency", "max"),
            median_core_frequency=("core_layer_frequency", "median"),
            max_core_avg_pnl_per_day=("core_layer_avg_pnl_per_day", "max"),
            median_core_avg_pnl_per_day=("core_layer_avg_pnl_per_day", "median"),
            max_core_total_pnl=("core_layer_total_pnl", "max"),
            specs_core_win_80_plus=("core_win_80_plus", "sum"),
            specs_core_win_82_plus=("core_win_82_plus", "sum"),
            specs_core_win_85_plus=("core_win_85_plus", "sum"),
        )
        .reset_index()
        .sort_values(
            [
                "max_core_win_rate",
                "median_core_win_rate",
                "max_core_avg_pnl_per_day",
            ],
            ascending=[False, False, False],
        )
    )
    return out


core_by_vrp = parameter_group_summary(core_diag, "vrp_triple")
core_by_z = parameter_group_summary(core_diag, "z_triple")
core_by_rsi = parameter_group_summary(core_diag, "rsi_triple")
core_by_rv21d = parameter_group_summary(core_diag, "rv21d_triple")

print()
print("Core win rate by VRP triple:")
display(core_by_vrp)

print()
print("Core win rate by z-score triple:")
display(core_by_z)

print()
print("Core win rate by RSI cap triple:")
display(core_by_rsi)

print()
print("Core win rate by RV21D floor triple:")
display(core_by_rv21d)


# -----------------------------
# 5. Core conditional stats by tenor / bucket
# -----------------------------

core_conditional = conditional_stats.loc[
    conditional_stats["layer"].eq("Core")
].copy()

# Keep only top Core specs by Core win rate for focused diagnostics.
top_core_ids = (
    core_diag
    .sort_values(
        [
            "core_layer_win_rate",
            "core_layer_avg_pnl_per_day",
            "core_layer_frequency",
        ],
        ascending=[False, False, False],
    )
    .head(25)["spec_id"]
    .tolist()
)

top_core_conditional = core_conditional.loc[
    core_conditional["spec_id"].isin(top_core_ids)
].copy()

print()
print("Top Core specs: conditional tenor stats:")
display(
    top_core_conditional[
        [
            "spec_id",
            "tenor",
            "tenor_bucket",
            "conditional_obs",
            "conditional_sample_ok",
            "conditional_win_probability",
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_total_pnl",
            "conditional_worst_trade_pnl",
            "conditional_worst_pnl_per_day",
            "conditional_avg_actual_dte",
        ]
    ]
    .sort_values(
        [
            "spec_id",
            "conditional_win_probability",
            "conditional_avg_pnl_per_day",
            "tenor",
        ],
        ascending=[True, False, False, False],
    )
)

core_conditional_by_bucket = (
    top_core_conditional
    .groupby("tenor_bucket")
    .agg(
        rows=("spec_id", "size"),
        median_conditional_obs=("conditional_obs", "median"),
        max_conditional_obs=("conditional_obs", "max"),
        median_win_probability=("conditional_win_probability", "median"),
        max_win_probability=("conditional_win_probability", "max"),
        median_avg_pnl_per_day=("conditional_avg_pnl_per_day", "median"),
        max_avg_pnl_per_day=("conditional_avg_pnl_per_day", "max"),
        worst_trade_pnl=("conditional_worst_trade_pnl", "min"),
    )
    .reset_index()
    .sort_values("median_win_probability", ascending=False)
)

print()
print("Top Core specs: conditional stats by bucket:")
display(core_conditional_by_bucket)


# -----------------------------
# 6. Top full models / near-target models
# -----------------------------

model_summary["overall_win_gap_to_80"] = 0.80 - model_summary["overall_win_rate"]
model_summary["core_win_gap_to_85"] = 0.85 - model_summary["core_win_rate"]
model_summary["frequency_gap_to_33"] = 0.33 - model_summary["overall_frequency"]

model_summary["near_target_score"] = (
    model_summary["overall_win_gap_to_80"].clip(lower=0)
    + model_summary["core_win_gap_to_85"].clip(lower=0)
    + model_summary["frequency_gap_to_33"].clip(lower=0)
)

near_target_cols = [
    "review_rank",
    "near_target_score",
    "meets_all_targets",
    "overall_trades",
    "overall_frequency",
    "frequency_gap_to_33",
    "overall_win_rate",
    "overall_win_gap_to_80",
    "core_win_rate",
    "core_win_gap_to_85",
    "overall_avg_pnl_per_day",
    "overall_exposure_day_weighted_pnl_per_day",
    "overall_total_pnl",
    "overall_profit_factor",
    "overall_worst_trade_pnl",
    "core_trades",
    "core_frequency",
    "core_avg_pnl_per_day",
    "core_total_pnl",
    "secondary_trades",
    "secondary_frequency",
    "secondary_win_rate",
    "secondary_avg_pnl_per_day",
    "secondary_total_pnl",
    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_day",
    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_day",
    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_day",
    "core_spec_id",
    "secondary_spec_id",
]

print()
print("Top models closest to all targets:")
display(
    model_summary[near_target_cols]
    .sort_values(
        [
            "near_target_score",
            "overall_win_rate",
            "core_win_rate",
            "overall_frequency",
            "overall_avg_pnl_per_day",
        ],
        ascending=[True, False, False, False, False],
    )
    .head(50)
)

print()
print("Highest-frequency models with overall win rate >= 79%:")
display(
    model_summary
    .loc[model_summary["overall_win_rate"].ge(0.79), near_target_cols]
    .sort_values(
        [
            "overall_frequency",
            "overall_win_rate",
            "overall_avg_pnl_per_day",
        ],
        ascending=[False, False, False],
    )
    .head(50)
)

print()
print("Highest overall win-rate models:")
display(
    model_summary[near_target_cols]
    .sort_values(
        [
            "overall_win_rate",
            "overall_avg_pnl_per_day",
            "overall_frequency",
        ],
        ascending=[False, False, False],
    )
    .head(50)
)


# -----------------------------
# 7. Reconstruct selected trades for top models for year / loss diagnostics
# -----------------------------

# This section reconstructs selected rows for a small set of top models only.
# It repeats the Cell 3 old-process logic, but only for diagnostics.

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_actual_dte_lookup():
    outcome_discovery_candidates = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not outcome_discovery_candidates:
        return None

    discovery_path = outcome_discovery_candidates[-1]
    discovery = pd.read_csv(discovery_path)
    discovery.columns = [str(c).strip() for c in discovery.columns]

    file_col = "chosen_file" if "chosen_file" in discovery.columns else "path"

    if file_col not in discovery.columns:
        return None

    preferred = discovery.copy()

    if "eligible_for_backtest" in preferred.columns:
        preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

    if preferred.empty:
        return None

    if "score" in preferred.columns:
        preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
    else:
        preferred["_score"] = 0.0

    if "chosen_outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["chosen_outcome_col"].eq("normalized_pnl_dollars")
    elif "outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["outcome_col"].eq("normalized_pnl_dollars")
    else:
        preferred["_preferred_pnl"] = False

    preferred = preferred.sort_values(
        ["_preferred_pnl", "_score"],
        ascending=[False, False],
    )

    for _, row in preferred.iterrows():
        path = Path(row[file_col])

        if not path.exists():
            continue

        try:
            raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
        except Exception:
            continue

        raw.columns = [str(c).strip() for c in raw.columns]

        trade_date_col = first_existing_col(raw, ["trade_date", "trade_dt", "date"])
        tenor_col = first_existing_col(raw, ["tenor", "entry_tenor", "target_tenor"])
        actual_dte_col = first_existing_col(raw, ["actual_dte"])

        if trade_date_col is None or tenor_col is None or actual_dte_col is None:
            continue

        lookup = raw[[trade_date_col, tenor_col, actual_dte_col]].copy()
        lookup = lookup.rename(columns={
            trade_date_col: "trade_date",
            tenor_col: "tenor",
            actual_dte_col: "actual_dte",
        })

        lookup["trade_date"] = pd.to_numeric(lookup["trade_date"], errors="coerce").astype("Int64")
        lookup["tenor"] = pd.to_numeric(lookup["tenor"], errors="coerce").astype("Int64")
        lookup["actual_dte"] = pd.to_numeric(lookup["actual_dte"], errors="coerce")

        lookup = lookup.dropna(subset=["trade_date", "tenor"])
        lookup["trade_date"] = lookup["trade_date"].astype(int)
        lookup["tenor"] = lookup["tenor"].astype(int)

        lookup = lookup.drop_duplicates(["trade_date", "tenor"])

        return lookup

    return None


diagnostic_panel = panel.copy()

if "actual_dte" not in diagnostic_panel.columns:
    actual_dte_lookup = load_actual_dte_lookup()

    if actual_dte_lookup is not None:
        diagnostic_panel = diagnostic_panel.merge(
            actual_dte_lookup,
            on=["trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )
    else:
        diagnostic_panel["actual_dte"] = np.nan

diagnostic_panel["actual_dte"] = pd.to_numeric(diagnostic_panel["actual_dte"], errors="coerce")
diagnostic_panel["actual_dte_for_pnl_day"] = diagnostic_panel["actual_dte"]

fallback_mask = (
    diagnostic_panel["actual_dte_for_pnl_day"].isna()
    | diagnostic_panel["actual_dte_for_pnl_day"].le(0)
)

diagnostic_panel.loc[fallback_mask, "actual_dte_for_pnl_day"] = diagnostic_panel.loc[fallback_mask, "tenor"]
diagnostic_panel["pnl_per_day"] = diagnostic_panel["outcome_value"] / diagnostic_panel["actual_dte_for_pnl_day"]
diagnostic_panel["year"] = diagnostic_panel["date"].dt.year

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
TENOR_BUCKET_MAP = {
    9: "front",
    12: "front",
    15: "front",
    18: "middle",
    21: "middle",
    24: "middle",
    27: "back",
    30: "back",
    33: "back",
}
BUCKETS = ["front", "middle", "back"]
WIN_BAND = 0.0025
MIN_CONDITIONAL_OBS = 20

eligible = diagnostic_panel.loc[
    diagnostic_panel["is_parameter_sweep_eligible"].astype(bool)
].copy()

eligible = eligible.replace([np.inf, -np.inf], np.nan)

needed_for_matrix = [
    "trade_date",
    "date",
    "year",
    "tenor",
    "tenor_bucket",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "actual_dte_for_pnl_day",
    "pnl_per_day",
]

eligible = eligible.dropna(subset=needed_for_matrix).copy()
eligible = eligible.loc[eligible["tenor"].isin(TENORS)].copy()

date_counts = eligible.groupby("trade_date")["tenor"].nunique()
complete_dates = date_counts.loc[date_counts.eq(len(TENORS))].index.tolist()
eligible = eligible.loc[eligible["trade_date"].isin(complete_dates)].copy()

eligible_dates = sorted(eligible["trade_date"].unique())
num_dates = len(eligible_dates)
num_tenors = len(TENORS)

date_meta = (
    eligible[["trade_date", "date", "year"]]
    .drop_duplicates("trade_date")
    .set_index("trade_date")
    .loc[eligible_dates]
)

date_array = date_meta["date"].to_numpy()
year_array = date_meta["year"].to_numpy()

def pivot_matrix(col, dtype=float):
    mat = (
        eligible
        .pivot(index="trade_date", columns="tenor", values=col)
        .reindex(index=eligible_dates, columns=TENORS)
    )

    if mat.isna().any().any():
        raise ValueError(f"Missing matrix cells for {col}")

    return mat.to_numpy(dtype=dtype)

vrp_mat = pivot_matrix("vrp_log")
z3m_mat = pivot_matrix("vrp_z_3m")
z1y_mat = pivot_matrix("vrp_z_1y")
rsi_mat = pivot_matrix("rsi14")
rv21d_mat = pivot_matrix("rv21d")
pnl_mat = pivot_matrix("outcome_value")
pnl_per_day_mat = pivot_matrix("pnl_per_day")
actual_dte_mat = pivot_matrix("actual_dte_for_pnl_day")

tenor_array = np.array(TENORS, dtype=int)
tenor_bucket_array = np.array([TENOR_BUCKET_MAP[int(t)] for t in tenor_array], dtype=object)
valid_outcome_mat = np.isfinite(pnl_mat) & np.isfinite(pnl_per_day_mat)

def threshold_array_for_spec(spec_row, field_suffix):
    vals = []
    for tenor in TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]
        vals.append(float(spec_row[f"{bucket}_{field_suffix}"]))
    return np.array(vals, dtype=float)

def build_pass_matrix(spec_row):
    vrp_threshold = threshold_array_for_spec(spec_row, "vrp_log_min")
    z3m_threshold = threshold_array_for_spec(spec_row, "z_3m_min")
    z1y_threshold = threshold_array_for_spec(spec_row, "z_1y_min")
    rsi_cap = threshold_array_for_spec(spec_row, "rsi14_max")
    rv21d_floor = threshold_array_for_spec(spec_row, "rv21d_min")

    return (
        (vrp_mat >= vrp_threshold[None, :])
        & (z3m_mat >= z3m_threshold[None, :])
        & (z1y_mat >= z1y_threshold[None, :])
        & (rsi_mat <= rsi_cap[None, :])
        & (rv21d_mat >= rv21d_floor[None, :])
        & valid_outcome_mat
    )

def compute_stats_for_selection(pass_matrix, date_mask):
    rows = []
    date_mask = np.asarray(date_mask, dtype=bool)

    for col_idx, tenor in enumerate(TENORS):
        candidate_mask = pass_matrix[:, col_idx] & date_mask
        count = int(candidate_mask.sum())

        if count == 0:
            rows.append({
                "tenor": int(tenor),
                "conditional_obs": 0,
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
            })
            continue

        pnl = pnl_mat[candidate_mask, col_idx]
        pnl_day = pnl_per_day_mat[candidate_mask, col_idx]
        dte = actual_dte_mat[candidate_mask, col_idx]

        rows.append({
            "tenor": int(tenor),
            "conditional_obs": count,
            "conditional_win_probability": float(np.nanmean(pnl > 0)),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_day)),
            "conditional_aggregate_pnl_per_day": float(np.nansum(pnl) / np.nansum(dte)),
        })

    return pd.DataFrame(rows)

def select_cols(pass_matrix, stats_df, date_mask):
    date_mask = np.asarray(date_mask, dtype=bool)
    active_pass = pass_matrix & date_mask[:, None]

    stats_df = stats_df.sort_values("tenor").reset_index(drop=True)

    obs = stats_df["conditional_obs"].to_numpy(dtype=float)
    win = stats_df["conditional_win_probability"].to_numpy(dtype=float)
    avg_day = stats_df["conditional_avg_pnl_per_day"].to_numpy(dtype=float)
    agg_day = stats_df["conditional_aggregate_pnl_per_day"].to_numpy(dtype=float)

    has_stats = (
        (obs > 0)
        & np.isfinite(win)
        & np.isfinite(avg_day)
        & np.isfinite(agg_day)
    )

    sample_ok = obs >= MIN_CONDITIONAL_OBS

    active = active_pass & has_stats[None, :]
    active_sample_ok = active & sample_ok[None, :]
    row_has_sample_ok = active_sample_ok.any(axis=1)

    ranking_universe = np.where(row_has_sample_ok[:, None], active_sample_ok, active)

    win_masked = np.where(ranking_universe, win[None, :], -np.inf)
    best_win = win_masked.max(axis=1)
    row_has_any = np.isfinite(best_win)

    inside_band = (
        ranking_universe
        & row_has_any[:, None]
        & (win[None, :] >= best_win[:, None] - WIN_BAND)
    )

    selected_col = np.full(num_dates, -1, dtype=int)

    best_avg = np.full(num_dates, -np.inf)
    best_agg = np.full(num_dates, -np.inf)
    best_win_for_tie = np.full(num_dates, -np.inf)
    best_tenor = np.full(num_dates, -np.inf)

    for col_idx, tenor in enumerate(TENORS):
        cand = inside_band[:, col_idx]

        avg_val = avg_day[col_idx]
        agg_val = agg_day[col_idx]
        win_val = win[col_idx]
        tenor_val = float(tenor)

        better = (
            cand
            & (
                (avg_val > best_avg)
                | ((avg_val == best_avg) & (agg_val > best_agg))
                | ((avg_val == best_avg) & (agg_val == best_agg) & (win_val > best_win_for_tie))
                | ((avg_val == best_avg) & (agg_val == best_agg) & (win_val == best_win_for_tie) & (tenor_val > best_tenor))
            )
        )

        selected_col[better] = col_idx
        best_avg[better] = avg_val
        best_agg[better] = agg_val
        best_win_for_tie[better] = win_val
        best_tenor[better] = tenor_val

    return selected_col

def selected_trade_frame(core_spec_id, secondary_spec_id):
    core_row = core_specs.loc[core_specs["spec_id"].eq(core_spec_id)].iloc[0]
    secondary_row = secondary_specs.loc[secondary_specs["spec_id"].eq(secondary_spec_id)].iloc[0]

    core_pass = build_pass_matrix(core_row)
    all_dates_mask = np.ones(num_dates, dtype=bool)
    core_stats = compute_stats_for_selection(core_pass, all_dates_mask)
    core_selected_cols = select_cols(core_pass, core_stats, all_dates_mask)

    core_any_date = core_pass.any(axis=1)
    no_core_date_mask = ~core_any_date

    secondary_pass = build_pass_matrix(secondary_row)
    secondary_stats = compute_stats_for_selection(secondary_pass, no_core_date_mask)
    secondary_selected_cols = select_cols(secondary_pass, secondary_stats, no_core_date_mask)

    combined_cols = np.where(core_selected_cols >= 0, core_selected_cols, secondary_selected_cols)

    rows = []

    for date_idx, col_idx in enumerate(combined_cols):
        if col_idx < 0:
            continue

        layer = "Core" if core_selected_cols[date_idx] >= 0 else "Secondary"
        tenor = int(tenor_array[col_idx])

        rows.append({
            "trade_date": int(eligible_dates[date_idx]),
            "date": pd.to_datetime(date_array[date_idx]),
            "year": int(year_array[date_idx]),
            "layer": layer,
            "tenor": tenor,
            "tenor_bucket": TENOR_BUCKET_MAP[tenor],
            "outcome_value": float(pnl_mat[date_idx, col_idx]),
            "pnl_per_day": float(pnl_per_day_mat[date_idx, col_idx]),
            "actual_dte": float(actual_dte_mat[date_idx, col_idx]),
            "vrp_log": float(vrp_mat[date_idx, col_idx]),
            "vrp_z_3m": float(z3m_mat[date_idx, col_idx]),
            "vrp_z_1y": float(z1y_mat[date_idx, col_idx]),
            "rsi14": float(rsi_mat[date_idx, col_idx]),
            "rv21d": float(rv21d_mat[date_idx, col_idx]),
            "core_spec_id": core_spec_id,
            "secondary_spec_id": secondary_spec_id,
        })

    return pd.DataFrame(rows)

# Top 20 by review rank plus top 20 closest-to-target.
top_model_pairs = (
    pd.concat([
        model_summary.sort_values("review_rank").head(20),
        model_summary.sort_values(
            [
                "near_target_score",
                "overall_win_rate",
                "core_win_rate",
                "overall_frequency",
                "overall_avg_pnl_per_day",
            ],
            ascending=[True, False, False, False, False],
        ).head(20),
    ])
    [["core_spec_id", "secondary_spec_id"]]
    .drop_duplicates()
    .head(30)
)

selected_trade_frames = []

for _, pair in top_model_pairs.iterrows():
    selected_trade_frames.append(
        selected_trade_frame(
            core_spec_id=pair["core_spec_id"],
            secondary_spec_id=pair["secondary_spec_id"],
        )
    )

selected_diagnostics = pd.concat(selected_trade_frames, ignore_index=True)

year_summary = (
    selected_diagnostics
    .groupby(["core_spec_id", "secondary_spec_id", "year"])
    .agg(
        trades=("trade_date", "count"),
        win_rate=("outcome_value", lambda x: float((x > 0).mean())),
        total_pnl=("outcome_value", "sum"),
        avg_pnl_per_day=("pnl_per_day", "mean"),
        worst_trade_pnl=("outcome_value", "min"),
        core_trades=("layer", lambda x: int((x == "Core").sum())),
        secondary_trades=("layer", lambda x: int((x == "Secondary").sum())),
        front_trades=("tenor_bucket", lambda x: int((x == "front").sum())),
        middle_trades=("tenor_bucket", lambda x: int((x == "middle").sum())),
        back_trades=("tenor_bucket", lambda x: int((x == "back").sum())),
    )
    .reset_index()
)

loss_diagnostics = selected_diagnostics.loc[
    selected_diagnostics["outcome_value"] < 0
].copy()

loss_by_bucket_layer = (
    loss_diagnostics
    .groupby(["layer", "tenor_bucket"])
    .agg(
        losses=("trade_date", "count"),
        avg_loss=("outcome_value", "mean"),
        worst_loss=("outcome_value", "min"),
        avg_loss_per_day=("pnl_per_day", "mean"),
        median_vrp_log=("vrp_log", "median"),
        median_z_3m=("vrp_z_3m", "median"),
        median_z_1y=("vrp_z_1y", "median"),
        median_rsi14=("rsi14", "median"),
        median_rv21d=("rv21d", "median"),
    )
    .reset_index()
    .sort_values(["layer", "losses"], ascending=[True, False])
)

loss_by_year = (
    loss_diagnostics
    .groupby(["year", "layer", "tenor_bucket"])
    .agg(
        losses=("trade_date", "count"),
        total_loss=("outcome_value", "sum"),
        worst_loss=("outcome_value", "min"),
        avg_loss_per_day=("pnl_per_day", "mean"),
    )
    .reset_index()
    .sort_values(["year", "total_loss"], ascending=[True, True])
)

worst_selected_trades = (
    selected_diagnostics
    .sort_values("outcome_value")
    .head(100)
)

print()
print("Year-by-year performance for top diagnostic models:")
display(year_summary.head(100))

print()
print("Loss diagnostics by layer / bucket for top diagnostic models:")
display(loss_by_bucket_layer)

print()
print("Loss diagnostics by year / layer / bucket for top diagnostic models:")
display(loss_by_year.head(100))

print()
print("Worst selected trades across top diagnostic models:")
display(worst_selected_trades.head(100))


# -----------------------------
# 8. Save outputs
# -----------------------------

safe_start = eligible["date"].min().strftime("%Y%m%d")
safe_end = eligible["date"].max().strftime("%Y%m%d")

core_diag_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_core_diag_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_vrp_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_core_by_vrp_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_z_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_core_by_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rsi_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_core_by_rsi_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rv21d_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_core_by_rv21d_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

near_target_models_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_near_target_models_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

year_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_year_summary_top_models_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

loss_by_bucket_layer_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_loss_by_bucket_layer_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

loss_by_year_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_loss_by_year_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

worst_trades_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell4_worst_selected_trades_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_diag.to_csv(core_diag_path, index=False)
core_by_vrp.to_csv(core_by_vrp_path, index=False)
core_by_z.to_csv(core_by_z_path, index=False)
core_by_rsi.to_csv(core_by_rsi_path, index=False)
core_by_rv21d.to_csv(core_by_rv21d_path, index=False)

model_summary[near_target_cols].sort_values(
    [
        "near_target_score",
        "overall_win_rate",
        "core_win_rate",
        "overall_frequency",
        "overall_avg_pnl_per_day",
    ],
    ascending=[True, False, False, False, False],
).to_csv(near_target_models_path, index=False)

year_summary.to_csv(year_summary_path, index=False)
loss_by_bucket_layer.to_csv(loss_by_bucket_layer_path, index=False)
loss_by_year.to_csv(loss_by_year_path, index=False)
worst_selected_trades.to_csv(worst_trades_path, index=False)

print()
print("Cell 4 outputs saved:")
print(f"Core diagnostic:          {core_diag_path}")
print(f"Core by VRP:              {core_by_vrp_path}")
print(f"Core by z-score:          {core_by_z_path}")
print(f"Core by RSI:              {core_by_rsi_path}")
print(f"Core by RV21D:            {core_by_rv21d_path}")
print(f"Near-target models:       {near_target_models_path}")
print(f"Year summary top models:  {year_summary_path}")
print(f"Loss by bucket/layer:     {loss_by_bucket_layer_path}")
print(f"Loss by year:             {loss_by_year_path}")
print(f"Worst selected trades:    {worst_trades_path}")

print()
print("Cell 4 complete.")

Cell 4: Phase 1 diagnostic breakdown
Clean panel:             C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
All specs:               C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell2_all_phase1_specs_20260704_085147.csv
Cell 3 model summary:    C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_full_model_summary_old_process_20200701_20260519_20260704_090941.csv
Cell 3 core summary:     C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_core_layer_summary_20200701_20260519_20260704_090941.csv
Cell 3 conditional stats:C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell3_conditional_stats_old_process_20200701_20260519_20260704_090941.csv
Run timestamp:           20260704_092202

Load

,clean_panel_rows,all_specs,core_specs,secondary_specs,full_models,core_layer_summary_rows,conditional_stats_rows
0,14565,384,192,192,36864,192,333504



Core specs closest to 85% win rate:


,spec_id,candidate_frequency,core_layer_trades,core_layer_frequency,core_layer_win_rate,core_win_gap_to_85,core_layer_avg_pnl_per_day,core_layer_exposure_day_weighted_pnl_per_day,core_layer_total_pnl,core_layer_profit_factor,core_layer_worst_trade_pnl,core_layer_worst_pnl_per_day,core_layer_avg_actual_dte,core_layer_avg_selected_tenor,vrp_triple,z_triple,rsi_triple,rv21d_triple
6,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,0.182679,270,0.182679,0.803704,0.046296,389.906415,429.522773,2.286350e+06,2.740716,-108475.796818,-6026.433157,19.714815,19.388889,0.70 / 0.80 / 0.90,0.50 / 0.75 / 0.75,70 / 68 / 66,8.5 / 8.5 / 8.5
54,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_0.7...,0.140731,208,0.140731,0.802885,0.047115,384.433523,458.121069,1.853100e+06,2.807476,-108475.796818,-11224.939578,19.447115,19.514423,0.80 / 0.90 / 1.00,0.50 / 0.75 / 0.75,70 / 68 / 66,8.5 / 8.5 / 8.5
66,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_1.0...,0.138701,205,0.138701,0.800000,0.050000,387.452421,457.677056,1.776245e+06,2.732514,-108475.796818,-11224.939578,18.931707,18.936585,0.80 / 0.90 / 1.00,0.50 / 0.75 / 1.00,70 / 68 / 66,8.5 / 8.5 / 8.5
2,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,0.185386,274,0.185386,0.795620,0.054380,380.769989,418.040672,2.284592e+06,2.719853,-108475.796818,-6026.433157,19.945255,19.653285,0.70 / 0.80 / 0.90,0.50 / 0.75 / 0.75,72 / 70 / 68,8.5 / 8.5 / 8.5
7,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,0.178620,264,0.178620,0.795455,0.054545,393.181871,430.875218,2.141881e+06,2.626253,-108475.796818,-6026.433157,18.829545,18.465909,0.70 / 0.80 / 0.90,0.50 / 0.75 / 0.75,70 / 68 / 66,8.5 / 9.0 / 10.0
55,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_0.7...,0.138024,204,0.138024,0.794118,0.055882,387.748904,458.460242,1.732521e+06,2.683935,-108475.796818,-11224.939578,18.524510,18.352941,0.80 / 0.90 / 1.00,0.50 / 0.75 / 0.75,70 / 68 / 66,8.5 / 9.0 / 10.0
18,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_1.0...,0.180650,267,0.180650,0.794007,0.055993,374.402595,406.281162,2.081785e+06,2.503133,-108475.796818,-6026.433157,19.191011,18.988764,0.70 / 0.80 / 0.90,0.50 / 0.75 / 1.00,70 / 68 / 66,8.5 / 8.5 / 8.5
67,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_1.0...,0.137348,203,0.137348,0.793103,0.056897,389.794760,456.773298,1.689148e+06,2.641778,-108475.796818,-11224.939578,18.216749,18.029557,0.80 / 0.90 / 1.00,0.50 / 0.75 / 1.00,70 / 68 / 66,8.5 / 9.0 / 10.0
42,core_phase1_vrp_0.70_0.80_0.90_z_1.00_1.00_1.5...,0.110961,164,0.110961,0.792683,0.057317,427.586115,445.944468,1.386441e+06,2.766968,-108475.796818,-6026.433157,18.957317,18.841463,0.70 / 0.80 / 0.90,1.00 / 1.00 / 1.50,70 / 68 / 66,8.5 / 8.5 / 8.5
62,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_1.0...,0.140054,207,0.140054,0.792271,0.057729,374.496844,444.544858,1.765732e+06,2.699869,-108475.796818,-11224.939578,19.188406,19.217391,0.80 / 0.90 / 1.00,0.50 / 0.75 / 1.00,72 / 70 / 68,8.5 / 8.5 / 8.5



Core win rate by VRP triple:


...


Loss diagnostics by year / layer / bucket for top diagnostic models:


,year,layer,tenor_bucket,losses,total_loss,worst_loss,avg_loss_per_day
1,2020,Core,front,30,-5.846208e+05,-19487.360567,-1299.157371
3,2020,Secondary,middle,7,-1.319861e+05,-18855.158235,-785.631593
2,2020,Core,middle,4,-4.038668e+03,-1009.666898,-48.079376
0,2020,Core,back,4,-3.898969e+02,-97.474236,-3.361181
7,2021,Secondary,middle,114,-1.111896e+06,-23729.930996,-425.951713
5,2021,Secondary,back,40,-5.177680e+05,-27243.282335,-440.049329
4,2021,Core,front,30,-4.322278e+05,-14407.593649,-960.506243
6,2021,Secondary,front,28,-3.496757e+05,-21369.054483,-979.716966
9,2022,Core,front,513,-1.497353e+07,-108475.796818,-1951.140314
10,2022,Core,middle,90,-4.353145e+06,-108475.796818,-2542.861450



Worst selected trades across top diagnostic models:


,trade_date,date,year,layer,tenor,tenor_bucket,outcome_value,pnl_per_day,actual_dte,vrp_log,vrp_z_3m,vrp_z_1y,rsi14,rv21d,core_spec_id,secondary_spec_id
10946,20250325,2025-03-25,2025,Secondary,12,front,-110915.496005,-11091.549601,10.0,0.515474,0.764735,0.413949,49.722178,20.886771,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
8036,20250325,2025-03-25,2025,Secondary,12,front,-110915.496005,-11091.549601,10.0,0.515474,0.764735,0.413949,49.722178,20.886771,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
6563,20250325,2025-03-25,2025,Secondary,12,front,-110915.496005,-11091.549601,10.0,0.515474,0.764735,0.413949,49.722178,20.886771,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
10466,20250325,2025-03-25,2025,Secondary,12,front,-110915.496005,-11091.549601,10.0,0.515474,0.764735,0.413949,49.722178,20.886771,core_phase1_vrp_0.80_0.90_1.00_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
7054,20250325,2025-03-25,2025,Secondary,12,front,-110915.496005,-11091.549601,10.0,0.515474,0.764735,0.413949,49.722178,20.886771,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2893,20220824,2022-08-24,2022,Core,30,back,-85194.299611,-2839.809987,30.0,0.922526,2.662252,1.319919,51.383575,18.281227,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.60_0.70_0.80_z_0.75_1.0...
3827,20220824,2022-08-24,2022,Core,30,back,-85194.299611,-2839.809987,30.0,0.922526,2.662252,1.319919,51.383575,18.281227,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.60_0.80_0.90_z_0.75_1.0...
8792,20220824,2022-08-24,2022,Core,30,back,-85194.299611,-2839.809987,30.0,0.922526,2.662252,1.319919,51.383575,18.281227,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.50_0.60_0.70_z_0.25_0.5...
5383,20220824,2022-08-24,2022,Core,30,back,-85194.299611,-2839.809987,30.0,0.922526,2.662252,1.319919,51.383575,18.281227,core_phase1_vrp_0.70_0.80_0.90_z_0.50_0.75_0.7...,secondary_phase1_vrp_0.60_0.70_0.80_z_0.75_1.0...



Cell 4 outputs saved:
Core diagnostic:          C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell4_core_diag_20200701_20260519_20260704_092202.csv
Core by VRP:              C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell4_core_by_vrp_20200701_20260519_20260704_092202.csv
Core by z-score:          C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell4_core_by_z_20200701_20260519_20260704_092202.csv
Core by RSI:              C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell4_core_by_rsi_20200701_20260519_20260704_092202.csv
Core by RV21D:            C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell4_core_by_rv21d_20200701_20260519_20260704_092202.csv
Near-target models:       C:\Users\patri\vrp_project\data\audit\corsi_parameter_swee

In [7]:
# Cell 5: Core-only revised Phase 1 sweep
#
# Approved scope:
#   - Core only.
#   - No Secondary.
#   - No sizing.
#   - No final lock.
#   - Use all Core parameters:
#       VRP log
#       3m z-score
#       1y z-score
#       RSI cap
#       RV21D floor
#
# User-approved grid:
#   Front 9D / 12D / 15D:
#       VRP:   [0.60, 0.70]
#       z:     [0.40, 0.50, 0.60]
#       RSI:   [68, 70, 72]
#       RV21D: [6.5, 7.0, 7.5, 8.5]
#
#   Middle 18D / 21D / 24D:
#       VRP:   [0.70, 0.80]
#       z:     [0.60, 0.70, 0.80]
#       RSI:   [66, 68, 70]
#       RV21D: [7.0, 7.5, 8.5]
#
#   Back 27D / 30D / 33D:
#       VRP:   [0.70, 0.80, 0.90]
#       z:     [0.60, 0.70, 0.80]
#       RSI:   [66, 68]
#       RV21D: [7.0, 7.5, 8.5]
#
# Constraints:
#   - 3m z-score threshold = 1y z-score threshold within each bucket.
#   - For VRP / z / RV21D:
#         Front <= Middle <= Back
#   - For RSI cap:
#         Front >= Middle >= Back
#
# Core selection logic:
#   - Apply thresholds across all 9 tenors.
#   - If no tenor passes on a date: no trade.
#   - If multiple tenors pass: select one tenor using old-process logic:
#       1. Prefer tenors with conditional sample size >= 20.
#       2. Highest conditional win probability.
#       3. Keep tenors within 25 bps of best win probability.
#       4. Highest conditional average P&L/day.
#       5. Highest conditional aggregate P&L/day.
#       6. Highest conditional win probability.
#       7. Longer tenor.
#
# P&L/day:
#   normalized_pnl_dollars / actual_dte if actual_dte is available.
#   Falls back to normalized_pnl_dollars / tenor if actual_dte is unavailable.
#
# Target:
#   Core win rate >= 85%, with highest frequency possible.

from pathlib import Path
import itertools
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest clean panel
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No clean panel found in {PARAM_SWEEP_PROCESSED_DIR}")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]

print("Cell 5: Core-only revised Phase 1 sweep")
print(f"Clean panel path: {CLEAN_PANEL_PATH}")
print(f"Run timestamp:    {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_panel_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "has_outcome",
    "is_parameter_sweep_eligible",
]

missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]

if missing_panel_cols:
    raise ValueError(f"Clean panel missing required columns: {missing_panel_cols}")


# -----------------------------
# 2. Actual DTE helper
# -----------------------------

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_actual_dte_lookup():
    """
    Locate the same outcome universe used by Cell 1 and pull actual_dte if available.
    """
    outcome_discovery_candidates = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not outcome_discovery_candidates:
        return None, "no_outcome_discovery_file"

    discovery_path = outcome_discovery_candidates[-1]
    discovery = pd.read_csv(discovery_path)
    discovery.columns = [str(c).strip() for c in discovery.columns]

    file_col = "chosen_file" if "chosen_file" in discovery.columns else "path"

    if file_col not in discovery.columns:
        return None, "discovery_missing_path_column"

    preferred = discovery.copy()

    if "eligible_for_backtest" in preferred.columns:
        preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

    if preferred.empty:
        return None, "no_eligible_outcome_files"

    if "score" in preferred.columns:
        preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
    else:
        preferred["_score"] = 0.0

    if "chosen_outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["chosen_outcome_col"].eq("normalized_pnl_dollars")
    elif "outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["outcome_col"].eq("normalized_pnl_dollars")
    else:
        preferred["_preferred_pnl"] = False

    preferred = preferred.sort_values(
        ["_preferred_pnl", "_score"],
        ascending=[False, False],
    )

    for _, row in preferred.iterrows():
        path = Path(row[file_col])

        if not path.exists():
            continue

        try:
            raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
        except Exception:
            continue

        raw.columns = [str(c).strip() for c in raw.columns]

        trade_date_col = first_existing_col(raw, ["trade_date", "trade_dt", "date"])
        tenor_col = first_existing_col(raw, ["tenor", "entry_tenor", "target_tenor"])
        actual_dte_col = first_existing_col(raw, ["actual_dte"])

        if trade_date_col is None or tenor_col is None or actual_dte_col is None:
            continue

        lookup = raw[[trade_date_col, tenor_col, actual_dte_col]].copy()
        lookup = lookup.rename(columns={
            trade_date_col: "trade_date",
            tenor_col: "tenor",
            actual_dte_col: "actual_dte_from_outcomes",
        })

        lookup["trade_date"] = pd.to_numeric(lookup["trade_date"], errors="coerce").astype("Int64")
        lookup["tenor"] = pd.to_numeric(lookup["tenor"], errors="coerce").astype("Int64")
        lookup["actual_dte_from_outcomes"] = pd.to_numeric(
            lookup["actual_dte_from_outcomes"],
            errors="coerce",
        )

        lookup = lookup.dropna(subset=["trade_date", "tenor"])
        lookup["trade_date"] = lookup["trade_date"].astype(int)
        lookup["tenor"] = lookup["tenor"].astype(int)

        lookup = lookup.drop_duplicates(["trade_date", "tenor"])

        return lookup, str(path)

    return None, "actual_dte_not_found_in_outcome_files"


if "actual_dte" not in panel.columns:
    actual_dte_lookup, actual_dte_source = load_actual_dte_lookup()

    if actual_dte_lookup is not None:
        panel = panel.merge(
            actual_dte_lookup,
            on=["trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )
        panel["actual_dte"] = panel["actual_dte_from_outcomes"]
        panel = panel.drop(columns=["actual_dte_from_outcomes"])
    else:
        panel["actual_dte"] = np.nan
else:
    actual_dte_source = "clean_panel_existing_actual_dte"

panel["actual_dte"] = pd.to_numeric(panel["actual_dte"], errors="coerce")
panel["actual_dte_for_pnl_day"] = panel["actual_dte"]

fallback_mask = (
    panel["actual_dte_for_pnl_day"].isna()
    | panel["actual_dte_for_pnl_day"].le(0)
)

panel.loc[fallback_mask, "actual_dte_for_pnl_day"] = panel.loc[fallback_mask, "tenor"]
panel["pnl_per_day"] = panel["outcome_value"] / panel["actual_dte_for_pnl_day"]

print()
print("Actual DTE source:")
print(actual_dte_source)

print()
print("Actual DTE availability:")
display(pd.DataFrame([{
    "rows": len(panel),
    "actual_dte_non_null": int(panel["actual_dte"].notna().sum()),
    "actual_dte_non_null_pct": float(panel["actual_dte"].notna().mean()),
    "fallback_to_tenor_rows": int(fallback_mask.sum()),
    "fallback_to_tenor_pct": float(fallback_mask.mean()),
}]))


# -----------------------------
# 3. Build Core-only revised grid
# -----------------------------

FRONT_GRID = {
    "vrp": [0.60, 0.70],
    "z": [0.40, 0.50, 0.60],
    "rsi": [68, 70, 72],
    "rv21d": [6.5, 7.0, 7.5, 8.5],
}

MIDDLE_GRID = {
    "vrp": [0.70, 0.80],
    "z": [0.60, 0.70, 0.80],
    "rsi": [66, 68, 70],
    "rv21d": [7.0, 7.5, 8.5],
}

BACK_GRID = {
    "vrp": [0.70, 0.80, 0.90],
    "z": [0.60, 0.70, 0.80],
    "rsi": [66, 68],
    "rv21d": [7.0, 7.5, 8.5],
}


def valid_increasing_triples(front_vals, middle_vals, back_vals):
    triples = []

    for f, m, b in itertools.product(front_vals, middle_vals, back_vals):
        if f <= m <= b:
            triples.append((f, m, b))

    return triples


def valid_rsi_triples(front_vals, middle_vals, back_vals):
    triples = []

    for f, m, b in itertools.product(front_vals, middle_vals, back_vals):
        if f >= m >= b:
            triples.append((f, m, b))

    return triples


CORE_VRP_TRIPLES = valid_increasing_triples(
    FRONT_GRID["vrp"],
    MIDDLE_GRID["vrp"],
    BACK_GRID["vrp"],
)

CORE_Z_TRIPLES = valid_increasing_triples(
    FRONT_GRID["z"],
    MIDDLE_GRID["z"],
    BACK_GRID["z"],
)

CORE_RSI_TRIPLES = valid_rsi_triples(
    FRONT_GRID["rsi"],
    MIDDLE_GRID["rsi"],
    BACK_GRID["rsi"],
)

CORE_RV21D_TRIPLES = valid_increasing_triples(
    FRONT_GRID["rv21d"],
    MIDDLE_GRID["rv21d"],
    BACK_GRID["rv21d"],
)

spec_rows = []

for spec_num, (vrp, z, rsi, rv21d) in enumerate(
    itertools.product(
        CORE_VRP_TRIPLES,
        CORE_Z_TRIPLES,
        CORE_RSI_TRIPLES,
        CORE_RV21D_TRIPLES,
    ),
    start=1,
):
    spec_id = (
        f"core_revised_phase1_"
        f"vrp_{vrp[0]:.2f}_{vrp[1]:.2f}_{vrp[2]:.2f}_"
        f"z_{z[0]:.2f}_{z[1]:.2f}_{z[2]:.2f}_"
        f"rsi_{int(rsi[0])}_{int(rsi[1])}_{int(rsi[2])}_"
        f"rv21d_{rv21d[0]:.1f}_{rv21d[1]:.1f}_{rv21d[2]:.1f}"
    )

    spec_rows.append({
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_revised_phase1",
        "forecast_denominator": "har_total_simple",

        "front_vrp_log_min": vrp[0],
        "middle_vrp_log_min": vrp[1],
        "back_vrp_log_min": vrp[2],

        "front_z_3m_min": z[0],
        "middle_z_3m_min": z[1],
        "back_z_3m_min": z[2],

        "front_z_1y_min": z[0],
        "middle_z_1y_min": z[1],
        "back_z_1y_min": z[2],

        "front_rsi14_max": rsi[0],
        "middle_rsi14_max": rsi[1],
        "back_rsi14_max": rsi[2],

        "front_rv21d_min": rv21d[0],
        "middle_rv21d_min": rv21d[1],
        "back_rv21d_min": rv21d[2],

        "phase1_z_3m_equals_z_1y": True,
    })

core_specs = pd.DataFrame(spec_rows)

grid_summary = pd.DataFrame([{
    "valid_vrp_triples": len(CORE_VRP_TRIPLES),
    "valid_z_triples": len(CORE_Z_TRIPLES),
    "valid_rsi_triples": len(CORE_RSI_TRIPLES),
    "valid_rv21d_triples": len(CORE_RV21D_TRIPLES),
    "total_core_specs": len(core_specs),
}])

print()
print("Revised Core-only grid summary:")
display(grid_summary)

print()
print("Revised Core specs sample:")
display(core_specs.head(20))


# -----------------------------
# 4. Build complete eligible matrix panel
# -----------------------------

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "front",
    12: "front",
    15: "front",
    18: "middle",
    21: "middle",
    24: "middle",
    27: "back",
    30: "back",
    33: "back",
}

BUCKETS = ["front", "middle", "back"]
WIN_BAND = 0.0025
MIN_CONDITIONAL_OBS = 20

eligible = panel.loc[panel["is_parameter_sweep_eligible"].astype(bool)].copy()
eligible = eligible.replace([np.inf, -np.inf], np.nan)

needed_for_matrix = [
    "trade_date",
    "date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "actual_dte_for_pnl_day",
    "pnl_per_day",
]

eligible = eligible.dropna(subset=needed_for_matrix).copy()

eligible = eligible.loc[
    eligible["tenor"].isin(TENORS)
    & eligible["actual_dte_for_pnl_day"].gt(0)
].copy()

duplicate_rows = eligible.duplicated(["trade_date", "tenor"]).sum()

if duplicate_rows:
    raise ValueError(f"Eligible panel has duplicate trade_date x tenor rows: {duplicate_rows}")

date_counts = eligible.groupby("trade_date")["tenor"].nunique()
complete_dates = date_counts.loc[date_counts.eq(len(TENORS))].index.tolist()

eligible = eligible.loc[eligible["trade_date"].isin(complete_dates)].copy()

eligible_dates = sorted(eligible["trade_date"].unique())
num_dates = len(eligible_dates)
num_tenors = len(TENORS)

date_lookup = (
    eligible[["trade_date", "date"]]
    .drop_duplicates("trade_date")
    .set_index("trade_date")
    .loc[eligible_dates]
)

date_array = date_lookup["date"].to_numpy()


def pivot_matrix(col, dtype=float):
    mat = (
        eligible
        .pivot(index="trade_date", columns="tenor", values=col)
        .reindex(index=eligible_dates, columns=TENORS)
    )

    if mat.isna().any().any():
        missing = int(mat.isna().sum().sum())
        raise ValueError(f"Matrix for {col} has missing cells after complete-date filter: {missing}")

    return mat.to_numpy(dtype=dtype)


vrp_mat = pivot_matrix("vrp_log")
z3m_mat = pivot_matrix("vrp_z_3m")
z1y_mat = pivot_matrix("vrp_z_1y")
rsi_mat = pivot_matrix("rsi14")
rv21d_mat = pivot_matrix("rv21d")
pnl_mat = pivot_matrix("outcome_value")
actual_dte_mat = pivot_matrix("actual_dte_for_pnl_day")
pnl_per_day_mat = pivot_matrix("pnl_per_day")

valid_outcome_mat = (
    np.isfinite(pnl_mat)
    & np.isfinite(pnl_per_day_mat)
    & np.isfinite(actual_dte_mat)
)

tenor_array = np.array(TENORS, dtype=int)
tenor_bucket_array = np.array([TENOR_BUCKET_MAP[int(t)] for t in tenor_array], dtype=object)

print()
print("Cell 5 input summary:")
display(pd.DataFrame([{
    "eligible_rows_after_complete_date_filter": len(eligible),
    "eligible_dates": num_dates,
    "first_date": pd.to_datetime(date_array.min()),
    "last_date": pd.to_datetime(date_array.max()),
    "unique_tenors": num_tenors,
    "core_specs_to_test": len(core_specs),
    "win_band_bps": WIN_BAND * 10000,
    "min_conditional_obs": MIN_CONDITIONAL_OBS,
    "forecast_model": ", ".join(sorted(eligible["forecast_model"].astype(str).unique())),
}]))


# -----------------------------
# 5. Matrix helper functions
# -----------------------------

def threshold_array_for_spec(spec_row, field_suffix):
    vals = []

    for tenor in TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]
        vals.append(float(spec_row[f"{bucket}_{field_suffix}"]))

    return np.array(vals, dtype=float)


def build_pass_matrix(spec_row):
    vrp_threshold = threshold_array_for_spec(spec_row, "vrp_log_min")
    z3m_threshold = threshold_array_for_spec(spec_row, "z_3m_min")
    z1y_threshold = threshold_array_for_spec(spec_row, "z_1y_min")
    rsi_cap = threshold_array_for_spec(spec_row, "rsi14_max")
    rv21d_floor = threshold_array_for_spec(spec_row, "rv21d_min")

    return (
        (vrp_mat >= vrp_threshold[None, :])
        & (z3m_mat >= z3m_threshold[None, :])
        & (z1y_mat >= z1y_threshold[None, :])
        & (rsi_mat <= rsi_cap[None, :])
        & (rv21d_mat >= rv21d_floor[None, :])
        & valid_outcome_mat
    )


def compute_conditional_arrays(pass_matrix):
    """
    Conditional stats by tenor for a single Core spec.
    """
    obs = pass_matrix.sum(axis=0).astype(float)

    wins = ((pnl_mat > 0) & pass_matrix).sum(axis=0).astype(float)

    total_pnl = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_mat, 0.0).sum(axis=0),
        0.0,
    )

    total_pnl_day = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_per_day_mat, 0.0).sum(axis=0),
        np.nan,
    )

    total_dte = np.where(
        obs > 0,
        np.where(pass_matrix, actual_dte_mat, 0.0).sum(axis=0),
        np.nan,
    )

    win_prob = np.divide(
        wins,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_day = np.divide(
        total_pnl_day,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_trade = np.divide(
        total_pnl,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    aggregate_pnl_per_day = np.divide(
        total_pnl,
        total_dte,
        out=np.full_like(obs, np.nan, dtype=float),
        where=total_dte > 0,
    )

    worst_trade = np.full(num_tenors, np.nan)
    worst_pnl_day = np.full(num_tenors, np.nan)
    avg_actual_dte = np.full(num_tenors, np.nan)

    for col_idx in range(num_tenors):
        mask = pass_matrix[:, col_idx]

        if mask.any():
            worst_trade[col_idx] = np.nanmin(pnl_mat[mask, col_idx])
            worst_pnl_day[col_idx] = np.nanmin(pnl_per_day_mat[mask, col_idx])
            avg_actual_dte[col_idx] = np.nanmean(actual_dte_mat[mask, col_idx])

    sample_ok = obs >= MIN_CONDITIONAL_OBS

    return {
        "obs": obs,
        "wins": wins,
        "win_prob": win_prob,
        "avg_pnl_per_day": avg_pnl_per_day,
        "aggregate_pnl_per_day": aggregate_pnl_per_day,
        "avg_pnl_per_trade": avg_pnl_per_trade,
        "total_pnl": total_pnl,
        "worst_trade": worst_trade,
        "worst_pnl_day": worst_pnl_day,
        "avg_actual_dte": avg_actual_dte,
        "sample_ok": sample_ok,
    }


def select_cols_from_pass_and_stats(pass_matrix, stats):
    """
    Old-process-aligned selector.

    Ranking:
      1. Prefer tenors with conditional_obs >= MIN_CONDITIONAL_OBS.
      2. Highest conditional win probability.
      3. Keep tenors within WIN_BAND of best conditional win probability.
      4. Highest conditional average P&L/day.
      5. Highest conditional aggregate P&L/day.
      6. Highest conditional win probability.
      7. Longer tenor.
    """
    obs = stats["obs"]
    win = stats["win_prob"]
    avg_day = stats["avg_pnl_per_day"]
    agg_day = stats["aggregate_pnl_per_day"]
    sample_ok = stats["sample_ok"]

    has_stats = (
        (obs > 0)
        & np.isfinite(win)
        & np.isfinite(avg_day)
        & np.isfinite(agg_day)
    )

    active = pass_matrix & has_stats[None, :]

    active_sample_ok = active & sample_ok[None, :]
    row_has_sample_ok = active_sample_ok.any(axis=1)

    ranking_universe = np.where(
        row_has_sample_ok[:, None],
        active_sample_ok,
        active,
    )

    win_masked = np.where(
        ranking_universe,
        win[None, :],
        -np.inf,
    )

    best_win = win_masked.max(axis=1)
    row_has_any = np.isfinite(best_win)

    inside_win_band = (
        ranking_universe
        & row_has_any[:, None]
        & (win[None, :] >= (best_win[:, None] - WIN_BAND))
    )

    selected_col = np.full(num_dates, -1, dtype=int)

    best_avg = np.full(num_dates, -np.inf)
    best_agg = np.full(num_dates, -np.inf)
    best_win_for_tie = np.full(num_dates, -np.inf)
    best_tenor = np.full(num_dates, -np.inf)

    for col_idx, tenor in enumerate(TENORS):
        cand = inside_win_band[:, col_idx]

        avg_val = avg_day[col_idx]
        agg_val = agg_day[col_idx]
        win_val = win[col_idx]
        tenor_val = float(tenor)

        better = (
            cand
            & (
                (avg_val > best_avg)
                | (
                    (avg_val == best_avg)
                    & (agg_val > best_agg)
                )
                | (
                    (avg_val == best_avg)
                    & (agg_val == best_agg)
                    & (win_val > best_win_for_tie)
                )
                | (
                    (avg_val == best_avg)
                    & (agg_val == best_agg)
                    & (win_val == best_win_for_tie)
                    & (tenor_val > best_tenor)
                )
            )
        )

        selected_col[better] = col_idx
        best_avg[better] = avg_val
        best_agg[better] = agg_val
        best_win_for_tie[better] = win_val
        best_tenor[better] = tenor_val

    return selected_col


def profit_factor(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    gains = values[values > 0].sum()
    losses = values[values < 0].sum()

    if losses < 0:
        return gains / abs(losses)

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def summarize_selected_cols(selected_cols, prefix):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    if date_idx.size == 0:
        return {
            f"{prefix}_trades": 0,
            f"{prefix}_frequency": 0.0,
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_day_weighted_pnl_per_day": np.nan,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade_pnl": np.nan,
            f"{prefix}_worst_pnl_per_day": np.nan,
            f"{prefix}_best_trade_pnl": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_avg_selected_tenor": np.nan,
        }

    cols = selected_cols[date_idx]

    pnl = pnl_mat[date_idx, cols]
    pnl_day = pnl_per_day_mat[date_idx, cols]
    dte = actual_dte_mat[date_idx, cols]
    selected_tenors = tenor_array[cols]

    total_dte = np.nansum(dte)

    if total_dte > 0:
        exposure_day_weighted_pnl_per_day = np.nansum(pnl) / total_dte
    else:
        exposure_day_weighted_pnl_per_day = np.nan

    return {
        f"{prefix}_trades": int(date_idx.size),
        f"{prefix}_frequency": float(date_idx.size / num_dates) if num_dates else np.nan,
        f"{prefix}_win_rate": float(np.nanmean(pnl > 0)),
        f"{prefix}_total_pnl": float(np.nansum(pnl)),
        f"{prefix}_avg_pnl_per_trade": float(np.nanmean(pnl)),
        f"{prefix}_avg_pnl_per_day": float(np.nanmean(pnl_day)),
        f"{prefix}_exposure_day_weighted_pnl_per_day": float(exposure_day_weighted_pnl_per_day),
        f"{prefix}_profit_factor": float(profit_factor(pnl)),
        f"{prefix}_worst_trade_pnl": float(np.nanmin(pnl)),
        f"{prefix}_worst_pnl_per_day": float(np.nanmin(pnl_day)),
        f"{prefix}_best_trade_pnl": float(np.nanmax(pnl)),
        f"{prefix}_avg_actual_dte": float(np.nanmean(dte)),
        f"{prefix}_avg_selected_tenor": float(np.nanmean(selected_tenors)),
    }


def bucket_metrics(selected_cols):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    out = {}

    for bucket in BUCKETS:
        out[f"{bucket}_selected_trades"] = 0
        out[f"{bucket}_selected_win_rate"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_total_pnl"] = 0.0
        out[f"{bucket}_selected_worst_trade_pnl"] = np.nan

    if date_idx.size == 0:
        return out

    cols = selected_cols[date_idx]
    selected_buckets = tenor_bucket_array[cols]

    for bucket in BUCKETS:
        bucket_mask = selected_buckets == bucket

        if not bucket_mask.any():
            continue

        bucket_date_idx = date_idx[bucket_mask]
        bucket_cols = cols[bucket_mask]

        pnl = pnl_mat[bucket_date_idx, bucket_cols]
        pnl_day = pnl_per_day_mat[bucket_date_idx, bucket_cols]
        dte = actual_dte_mat[bucket_date_idx, bucket_cols]

        total_dte = np.nansum(dte)

        out[f"{bucket}_selected_trades"] = int(bucket_mask.sum())
        out[f"{bucket}_selected_win_rate"] = float(np.nanmean(pnl > 0))
        out[f"{bucket}_selected_avg_pnl_per_day"] = float(np.nanmean(pnl_day))
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = (
            float(np.nansum(pnl) / total_dte) if total_dte > 0 else np.nan
        )
        out[f"{bucket}_selected_total_pnl"] = float(np.nansum(pnl))
        out[f"{bucket}_selected_worst_trade_pnl"] = float(np.nanmin(pnl))

    return out


def append_conditional_rows(rows, spec_id, stats):
    for col_idx, tenor in enumerate(TENORS):
        rows.append({
            "spec_id": spec_id,
            "role": "Core",
            "tenor": int(tenor),
            "tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "conditional_obs": int(stats["obs"][col_idx]),
            "conditional_wins": int(stats["wins"][col_idx]),
            "conditional_win_probability": stats["win_prob"][col_idx],
            "conditional_avg_pnl_per_day": stats["avg_pnl_per_day"][col_idx],
            "conditional_aggregate_pnl_per_day": stats["aggregate_pnl_per_day"][col_idx],
            "conditional_avg_pnl_per_trade": stats["avg_pnl_per_trade"][col_idx],
            "conditional_total_pnl": stats["total_pnl"][col_idx],
            "conditional_worst_trade_pnl": stats["worst_trade"][col_idx],
            "conditional_worst_pnl_per_day": stats["worst_pnl_day"][col_idx],
            "conditional_avg_actual_dte": stats["avg_actual_dte"][col_idx],
            "conditional_sample_ok": bool(stats["sample_ok"][col_idx]),
        })


# -----------------------------
# 6. Run revised Core-only sweep
# -----------------------------

summary_rows = []
conditional_rows = []

total_specs = len(core_specs)

print()
print(f"Running Core-only revised sweep: {total_specs:,} specs")

for i, (_, spec_row) in enumerate(core_specs.iterrows(), start=1):
    spec_id = spec_row["spec_id"]

    pass_matrix = build_pass_matrix(spec_row)
    candidate_dates = pass_matrix.any(axis=1)
    candidate_rows = int(pass_matrix.sum())

    stats = compute_conditional_arrays(pass_matrix)
    selected_cols = select_cols_from_pass_and_stats(pass_matrix, stats)

    row = {
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_revised_phase1",
        "candidate_rows": candidate_rows,
        "candidate_dates": int(candidate_dates.sum()),
        "candidate_frequency": float(candidate_dates.sum() / num_dates) if num_dates else np.nan,

        "front_vrp_log_min": spec_row["front_vrp_log_min"],
        "middle_vrp_log_min": spec_row["middle_vrp_log_min"],
        "back_vrp_log_min": spec_row["back_vrp_log_min"],

        "front_z_3m_min": spec_row["front_z_3m_min"],
        "middle_z_3m_min": spec_row["middle_z_3m_min"],
        "back_z_3m_min": spec_row["back_z_3m_min"],

        "front_z_1y_min": spec_row["front_z_1y_min"],
        "middle_z_1y_min": spec_row["middle_z_1y_min"],
        "back_z_1y_min": spec_row["back_z_1y_min"],

        "front_rsi14_max": spec_row["front_rsi14_max"],
        "middle_rsi14_max": spec_row["middle_rsi14_max"],
        "back_rsi14_max": spec_row["back_rsi14_max"],

        "front_rv21d_min": spec_row["front_rv21d_min"],
        "middle_rv21d_min": spec_row["middle_rv21d_min"],
        "back_rv21d_min": spec_row["back_rv21d_min"],
    }

    row.update(summarize_selected_cols(selected_cols, "core"))
    row.update(bucket_metrics(selected_cols))

    summary_rows.append(row)
    append_conditional_rows(conditional_rows, spec_id, stats)

    if i % 1000 == 0 or i == total_specs:
        print(f"Core specs processed: {i:,} / {total_specs:,}")

core_summary = pd.DataFrame(summary_rows)
conditional_stats = pd.DataFrame(conditional_rows)


# -----------------------------
# 7. Add labels / flags / rankings
# -----------------------------

core_summary["vrp_triple"] = (
    core_summary["front_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_vrp_log_min"].map("{:.2f}".format)
)

core_summary["z_triple"] = (
    core_summary["front_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_z_3m_min"].map("{:.2f}".format)
)

core_summary["rsi_triple"] = (
    core_summary["front_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["middle_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["back_rsi14_max"].astype(int).astype(str)
)

core_summary["rv21d_triple"] = (
    core_summary["front_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["middle_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["back_rv21d_min"].map("{:.1f}".format)
)

core_summary["meets_core_win_85"] = core_summary["core_win_rate"].ge(0.85)
core_summary["meets_core_win_82"] = core_summary["core_win_rate"].ge(0.82)
core_summary["meets_core_win_80"] = core_summary["core_win_rate"].ge(0.80)

core_summary["core_win_gap_to_85"] = 0.85 - core_summary["core_win_rate"]

core_summary = core_summary.sort_values(
    [
        "meets_core_win_85",
        "core_frequency",
        "core_win_rate",
        "core_avg_pnl_per_day",
        "core_total_pnl",
        "core_worst_trade_pnl",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        False,
        False,
    ],
).reset_index(drop=True)

core_summary["frequency_rank_with_targets_first"] = np.arange(1, len(core_summary) + 1)

core_summary_by_win = core_summary.sort_values(
    [
        "core_win_rate",
        "core_frequency",
        "core_avg_pnl_per_day",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

core_summary_by_win["win_rate_rank"] = np.arange(1, len(core_summary_by_win) + 1)


# -----------------------------
# 8. Parameter group diagnostics
# -----------------------------

def parameter_group_summary(df, group_col):
    out = (
        df
        .groupby(group_col)
        .agg(
            specs=("spec_id", "nunique"),
            max_core_win_rate=("core_win_rate", "max"),
            median_core_win_rate=("core_win_rate", "median"),
            max_core_frequency=("core_frequency", "max"),
            median_core_frequency=("core_frequency", "median"),
            max_core_avg_pnl_per_day=("core_avg_pnl_per_day", "max"),
            median_core_avg_pnl_per_day=("core_avg_pnl_per_day", "median"),
            max_core_total_pnl=("core_total_pnl", "max"),
            specs_core_win_85_plus=("meets_core_win_85", "sum"),
            specs_core_win_82_plus=("meets_core_win_82", "sum"),
            specs_core_win_80_plus=("meets_core_win_80", "sum"),
        )
        .reset_index()
        .sort_values(
            [
                "max_core_win_rate",
                "max_core_frequency",
                "max_core_avg_pnl_per_day",
            ],
            ascending=[False, False, False],
        )
    )

    return out


core_by_vrp = parameter_group_summary(core_summary, "vrp_triple")
core_by_z = parameter_group_summary(core_summary, "z_triple")
core_by_rsi = parameter_group_summary(core_summary, "rsi_triple")
core_by_rv21d = parameter_group_summary(core_summary, "rv21d_triple")


# -----------------------------
# 9. Display outputs
# -----------------------------

print()
print("Core-only revised sweep summary:")
display(pd.DataFrame([{
    "core_specs_tested": len(core_summary),
    "specs_core_win_85_plus": int(core_summary["meets_core_win_85"].sum()),
    "specs_core_win_82_plus": int(core_summary["meets_core_win_82"].sum()),
    "specs_core_win_80_plus": int(core_summary["meets_core_win_80"].sum()),
    "max_core_win_rate": core_summary["core_win_rate"].max(),
    "median_core_win_rate": core_summary["core_win_rate"].median(),
    "max_core_frequency": core_summary["core_frequency"].max(),
    "median_core_frequency": core_summary["core_frequency"].median(),
    "max_core_avg_pnl_per_day": core_summary["core_avg_pnl_per_day"].max(),
    "median_core_avg_pnl_per_day": core_summary["core_avg_pnl_per_day"].median(),
}]))

review_cols = [
    "frequency_rank_with_targets_first",
    "win_rate_rank" if "win_rate_rank" in core_summary.columns else "spec_id",
    "meets_core_win_85",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_win_gap_to_85",
    "core_avg_pnl_per_day",
    "core_exposure_day_weighted_pnl_per_day",
    "core_total_pnl",
    "core_profit_factor",
    "core_worst_trade_pnl",
    "core_worst_pnl_per_day",
    "core_avg_actual_dte",
    "core_avg_selected_tenor",

    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_day",
    "front_selected_total_pnl",
    "front_selected_worst_trade_pnl",

    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_day",
    "middle_selected_total_pnl",
    "middle_selected_worst_trade_pnl",

    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_day",
    "back_selected_total_pnl",
    "back_selected_worst_trade_pnl",

    "vrp_triple",
    "z_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

# Make a clean review frame with both ranks attached.
core_review = core_summary.merge(
    core_summary_by_win[["spec_id", "win_rate_rank"]],
    on="spec_id",
    how="left",
    validate="one_to_one",
)

review_cols = [
    "frequency_rank_with_targets_first",
    "win_rate_rank",
    "meets_core_win_85",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_win_gap_to_85",
    "core_avg_pnl_per_day",
    "core_exposure_day_weighted_pnl_per_day",
    "core_total_pnl",
    "core_profit_factor",
    "core_worst_trade_pnl",
    "core_worst_pnl_per_day",
    "core_avg_actual_dte",
    "core_avg_selected_tenor",

    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_day",
    "front_selected_total_pnl",
    "front_selected_worst_trade_pnl",

    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_day",
    "middle_selected_total_pnl",
    "middle_selected_worst_trade_pnl",

    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_day",
    "back_selected_total_pnl",
    "back_selected_worst_trade_pnl",

    "vrp_triple",
    "z_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

print()
print("Top Core specs by win rate:")
display(
    core_review
    .sort_values(
        [
            "core_win_rate",
            "core_frequency",
            "core_avg_pnl_per_day",
            "core_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    [review_cols]
    .head(100)
)

print()
print("Top Core specs with win rate >= 85%, ranked by frequency:")
display(
    core_review
    .loc[core_review["meets_core_win_85"], review_cols]
    .sort_values(
        [
            "core_frequency",
            "core_win_rate",
            "core_avg_pnl_per_day",
            "core_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Top Core specs with win rate >= 85%, ranked by average P&L/day:")
display(
    core_review
    .loc[core_review["meets_core_win_85"], review_cols]
    .sort_values(
        [
            "core_avg_pnl_per_day",
            "core_frequency",
            "core_win_rate",
            "core_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Highest-frequency Core specs with win rate >= 82%:")
display(
    core_review
    .loc[core_review["core_win_rate"].ge(0.82), review_cols]
    .sort_values(
        [
            "core_frequency",
            "core_win_rate",
            "core_avg_pnl_per_day",
            "core_total_pnl",
        ],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Core parameter diagnostics by VRP triple:")
display(core_by_vrp)

print()
print("Core parameter diagnostics by z-score triple:")
display(core_by_z)

print()
print("Core parameter diagnostics by RSI cap triple:")
display(core_by_rsi)

print()
print("Core parameter diagnostics by RV21D floor triple:")
display(core_by_rv21d)


# -----------------------------
# 10. Save outputs
# -----------------------------

safe_start = pd.to_datetime(date_array.min()).strftime("%Y%m%d")
safe_end = pd.to_datetime(date_array.max()).strftime("%Y%m%d")

core_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_specs_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

grid_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_grid_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

conditional_stats_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_conditional_stats_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

top_by_win_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_top_by_win_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_win85_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_pnl_day_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_win85_by_pnl_day_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_vrp_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_by_vrp_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_z_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_by_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rsi_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_by_rsi_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rv21d_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell5_core_only_revised_by_rv21d_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_specs.to_csv(core_specs_path, index=False)
grid_summary.to_csv(grid_summary_path, index=False)
core_review.to_csv(core_summary_path, index=False)
conditional_stats.to_csv(conditional_stats_path, index=False)

core_review.sort_values(
    [
        "core_win_rate",
        "core_frequency",
        "core_avg_pnl_per_day",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).head(1000).to_csv(top_by_win_path, index=False)

core_review.loc[
    core_review["meets_core_win_85"]
].sort_values(
    [
        "core_frequency",
        "core_win_rate",
        "core_avg_pnl_per_day",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).to_csv(target_by_frequency_path, index=False)

core_review.loc[
    core_review["meets_core_win_85"]
].sort_values(
    [
        "core_avg_pnl_per_day",
        "core_frequency",
        "core_win_rate",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).to_csv(target_by_pnl_day_path, index=False)

core_by_vrp.to_csv(core_by_vrp_path, index=False)
core_by_z.to_csv(core_by_z_path, index=False)
core_by_rsi.to_csv(core_by_rsi_path, index=False)
core_by_rv21d.to_csv(core_by_rv21d_path, index=False)

print()
print("Cell 5 outputs saved:")
print(f"Core specs:                  {core_specs_path}")
print(f"Grid summary:                {grid_summary_path}")
print(f"Core summary:                {core_summary_path}")
print(f"Conditional stats:           {conditional_stats_path}")
print(f"Top by win rate:             {top_by_win_path}")
print(f"Win >=85 by frequency:       {target_by_frequency_path}")
print(f"Win >=85 by P&L/day:         {target_by_pnl_day_path}")
print(f"Core by VRP:                 {core_by_vrp_path}")
print(f"Core by z-score:             {core_by_z_path}")
print(f"Core by RSI:                 {core_by_rsi_path}")
print(f"Core by RV21D:               {core_by_rv21d_path}")

print()
print("Cell 5 complete.")

Cell 5: Core-only revised Phase 1 sweep
Clean panel path: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
Run timestamp:    20260704_094614

Actual DTE source:
C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet

Actual DTE availability:


,rows,actual_dte_non_null,actual_dte_non_null_pct,fallback_to_tenor_rows,fallback_to_tenor_pct
0,14565,14565,1.0,0,0.0



Revised Core-only grid summary:


,valid_vrp_triples,valid_z_triples,valid_rsi_triples,valid_rv21d_triples,total_core_specs
0,10,18,13,16,37440



Revised Core specs sample:


...


Core parameter diagnostics by RSI cap triple:


,rsi_triple,specs,max_core_win_rate,median_core_win_rate,max_core_frequency,median_core_frequency,max_core_avg_pnl_per_day,median_core_avg_pnl_per_day,max_core_total_pnl,specs_core_win_85_plus,specs_core_win_82_plus,specs_core_win_80_plus
4,70 / 68 / 66,2880,0.818815,0.783394,0.240189,0.205683,418.980007,321.757054,2.537010e+06,0,0,247
9,72 / 68 / 66,2880,0.817869,0.780731,0.245602,0.209743,413.234618,317.790003,2.540427e+06,0,0,219
5,70 / 68 / 68,2880,0.815972,0.780822,0.240866,0.207037,418.014924,319.353736,2.533640e+06,0,0,141
6,70 / 70 / 66,2880,0.815331,0.782007,0.240189,0.206360,413.170746,318.244793,2.555374e+06,0,0,152
10,72 / 68 / 68,2880,0.815068,0.778146,0.246279,0.210419,412.282621,315.218436,2.537057e+06,0,0,131
11,72 / 70 / 66,2880,0.814433,0.779365,0.245602,0.209743,407.542286,313.782441,2.558791e+06,0,0,138
3,70 / 66 / 66,2880,0.814035,0.783951,0.238160,0.204330,420.739932,320.595674,2.511640e+06,0,0,121
8,72 / 66 / 66,2880,0.813149,0.781350,0.243572,0.207713,414.931299,316.630180,2.515057e+06,0,0,104
1,68 / 68 / 66,2880,0.812950,0.779661,0.232070,0.198241,424.994439,321.438558,2.501411e+06,0,0,107
7,70 / 70 / 68,2880,0.812500,0.779287,0.240866,0.207037,411.208121,315.313391,2.530808e+06,0,0,90



Core parameter diagnostics by RV21D floor triple:


,rv21d_triple,specs,max_core_win_rate,median_core_win_rate,max_core_frequency,median_core_frequency,max_core_avg_pnl_per_day,median_core_avg_pnl_per_day,max_core_total_pnl,specs_core_win_85_plus,specs_core_win_82_plus,specs_core_win_80_plus
15,8.5 / 8.5 / 8.5,2340,0.818815,0.799296,0.229364,0.194181,400.361119,328.444275,2.519449e+06,0,0,1139
14,7.5 / 8.5 / 8.5,2340,0.810631,0.788396,0.240189,0.203654,423.334796,316.370122,2.538957e+06,0,0,260
1,6.5 / 7.0 / 7.5,2340,0.807190,0.778481,0.246279,0.208390,412.872326,314.636766,2.494276e+06,0,0,16
0,6.5 / 7.0 / 7.0,2340,0.807190,0.778481,0.246279,0.208390,412.651656,314.441762,2.495948e+06,0,0,16
7,7.0 / 7.0 / 7.5,2340,0.807190,0.780120,0.244926,0.208390,420.693765,319.427826,2.519046e+06,0,0,16
6,7.0 / 7.0 / 7.0,2340,0.807190,0.780120,0.244926,0.208390,420.472304,319.198559,2.520718e+06,0,0,16
2,6.5 / 7.0 / 8.5,2340,0.805921,0.775920,0.245602,0.207713,410.498937,314.601994,2.511016e+06,0,0,14
8,7.0 / 7.0 / 8.5,2340,0.805921,0.777778,0.244249,0.207037,418.339973,319.634419,2.535786e+06,0,0,14
5,6.5 / 8.5 / 8.5,2340,0.805921,0.785714,0.243572,0.206360,418.995671,323.795793,2.534021e+06,0,0,59
11,7.0 / 8.5 / 8.5,2340,0.805921,0.787478,0.242219,0.205683,426.867271,329.309843,2.558791e+06,0,0,66



Cell 5 outputs saved:
Core specs:                  C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell5_core_only_revised_specs_20200701_20260519_20260704_094614.csv
Grid summary:                C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell5_core_only_revised_grid_summary_20200701_20260519_20260704_094614.csv
Core summary:                C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell5_core_only_revised_summary_20200701_20260519_20260704_094614.csv
Conditional stats:           C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell5_core_only_revised_conditional_stats_20200701_20260519_20260704_094614.csv
Top by win rate:             C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell5_core_only_revised_top_by_win_20200701_20260519_2

In [8]:
# Cell 6: Core-only revised Phase 2 sweep
#
# Approved scope:
#   - Core only.
#   - No Secondary.
#   - No sizing.
#   - No final lock.
#   - 3m z-score and 1y z-score are still tied within each bucket.
#
# Goal:
#   - Find Core parameter sets with win rate >= 85%.
#   - Then maximize Core frequency.
#   - Also report selection frequency and average P&L by exact selected tenor/DTE.
#
# User-approved grid:
#
#   Front 9D / 12D / 15D:
#       VRP:   [0.60, 0.70]
#       z:     [0.60, 0.70, 0.80]
#       RSI:   [69, 70, 71]
#       RV21D: [8.0, 8.5]
#
#   Middle 18D / 21D / 24D:
#       VRP:   [0.70]
#       z:     [0.75, 0.80]
#       RSI:   [67, 68, 69]
#       RV21D: [8.0, 8.5]
#
#   Back 27D / 30D / 33D:
#       VRP:   [0.70]
#       z:     [0.75, 0.80]
#       RSI:   [65, 66, 68]
#       RV21D: [8.0, 8.5]
#
# Constraints:
#   - For VRP / z / RV21D:
#         Front <= Middle <= Back
#   - For RSI cap:
#         Front >= Middle >= Back
#
# Expected valid grid size:
#   - VRP triples:   2
#   - z triples:     7
#   - RSI triples:   24
#   - RV21D triples: 4
#   - Total specs:   1,344
#
# Core selection logic:
#   - Apply thresholds across all 9 tenors.
#   - If no tenor passes on a date: no trade.
#   - If multiple tenors pass: select one tenor using old-process logic:
#       1. Prefer tenors with conditional sample size >= 20.
#       2. Highest conditional win probability.
#       3. Keep tenors within 25 bps of best win probability.
#       4. Highest conditional average P&L/day.
#       5. Highest conditional aggregate P&L/day.
#       6. Highest conditional win probability.
#       7. Longer tenor.
#
# P&L/day:
#   normalized_pnl_dollars / actual_dte if actual_dte is available.
#   Falls back to normalized_pnl_dollars / tenor if actual_dte is unavailable.

from pathlib import Path
import itertools
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest clean panel
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No clean panel found in {PARAM_SWEEP_PROCESSED_DIR}")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]

print("Cell 6: Core-only revised Phase 2 sweep")
print(f"Clean panel path: {CLEAN_PANEL_PATH}")
print(f"Run timestamp:    {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_panel_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "has_outcome",
    "is_parameter_sweep_eligible",
]

missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]

if missing_panel_cols:
    raise ValueError(f"Clean panel missing required columns: {missing_panel_cols}")


# -----------------------------
# 2. Actual DTE helper
# -----------------------------

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_actual_dte_lookup():
    outcome_discovery_candidates = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not outcome_discovery_candidates:
        return None, "no_outcome_discovery_file"

    discovery_path = outcome_discovery_candidates[-1]
    discovery = pd.read_csv(discovery_path)
    discovery.columns = [str(c).strip() for c in discovery.columns]

    file_col = "chosen_file" if "chosen_file" in discovery.columns else "path"

    if file_col not in discovery.columns:
        return None, "discovery_missing_path_column"

    preferred = discovery.copy()

    if "eligible_for_backtest" in preferred.columns:
        preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

    if preferred.empty:
        return None, "no_eligible_outcome_files"

    if "score" in preferred.columns:
        preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
    else:
        preferred["_score"] = 0.0

    if "chosen_outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["chosen_outcome_col"].eq("normalized_pnl_dollars")
    elif "outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["outcome_col"].eq("normalized_pnl_dollars")
    else:
        preferred["_preferred_pnl"] = False

    preferred = preferred.sort_values(
        ["_preferred_pnl", "_score"],
        ascending=[False, False],
    )

    for _, row in preferred.iterrows():
        path = Path(row[file_col])

        if not path.exists():
            continue

        try:
            raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
        except Exception:
            continue

        raw.columns = [str(c).strip() for c in raw.columns]

        trade_date_col = first_existing_col(raw, ["trade_date", "trade_dt", "date"])
        tenor_col = first_existing_col(raw, ["tenor", "entry_tenor", "target_tenor"])
        actual_dte_col = first_existing_col(raw, ["actual_dte"])

        if trade_date_col is None or tenor_col is None or actual_dte_col is None:
            continue

        lookup = raw[[trade_date_col, tenor_col, actual_dte_col]].copy()
        lookup = lookup.rename(columns={
            trade_date_col: "trade_date",
            tenor_col: "tenor",
            actual_dte_col: "actual_dte_from_outcomes",
        })

        lookup["trade_date"] = pd.to_numeric(lookup["trade_date"], errors="coerce").astype("Int64")
        lookup["tenor"] = pd.to_numeric(lookup["tenor"], errors="coerce").astype("Int64")
        lookup["actual_dte_from_outcomes"] = pd.to_numeric(
            lookup["actual_dte_from_outcomes"],
            errors="coerce",
        )

        lookup = lookup.dropna(subset=["trade_date", "tenor"])
        lookup["trade_date"] = lookup["trade_date"].astype(int)
        lookup["tenor"] = lookup["tenor"].astype(int)
        lookup = lookup.drop_duplicates(["trade_date", "tenor"])

        return lookup, str(path)

    return None, "actual_dte_not_found_in_outcome_files"


if "actual_dte" not in panel.columns:
    actual_dte_lookup, actual_dte_source = load_actual_dte_lookup()

    if actual_dte_lookup is not None:
        panel = panel.merge(
            actual_dte_lookup,
            on=["trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )
        panel["actual_dte"] = panel["actual_dte_from_outcomes"]
        panel = panel.drop(columns=["actual_dte_from_outcomes"])
    else:
        panel["actual_dte"] = np.nan
else:
    actual_dte_source = "clean_panel_existing_actual_dte"

panel["actual_dte"] = pd.to_numeric(panel["actual_dte"], errors="coerce")
panel["actual_dte_for_pnl_day"] = panel["actual_dte"]

fallback_mask = (
    panel["actual_dte_for_pnl_day"].isna()
    | panel["actual_dte_for_pnl_day"].le(0)
)

panel.loc[fallback_mask, "actual_dte_for_pnl_day"] = panel.loc[fallback_mask, "tenor"]
panel["pnl_per_day"] = panel["outcome_value"] / panel["actual_dte_for_pnl_day"]

print()
print("Actual DTE source:")
print(actual_dte_source)

print()
print("Actual DTE availability:")
display(pd.DataFrame([{
    "rows": len(panel),
    "actual_dte_non_null": int(panel["actual_dte"].notna().sum()),
    "actual_dte_non_null_pct": float(panel["actual_dte"].notna().mean()),
    "fallback_to_tenor_rows": int(fallback_mask.sum()),
    "fallback_to_tenor_pct": float(fallback_mask.mean()),
}]))


# -----------------------------
# 3. Build Core-only Phase 2 grid
# -----------------------------

FRONT_GRID = {
    "vrp": [0.60, 0.70],
    "z": [0.60, 0.70, 0.80],
    "rsi": [69, 70, 71],
    "rv21d": [8.0, 8.5],
}

MIDDLE_GRID = {
    "vrp": [0.70],
    "z": [0.75, 0.80],
    "rsi": [67, 68, 69],
    "rv21d": [8.0, 8.5],
}

BACK_GRID = {
    "vrp": [0.70],
    "z": [0.75, 0.80],
    "rsi": [65, 66, 68],
    "rv21d": [8.0, 8.5],
}


def valid_increasing_triples(front_vals, middle_vals, back_vals):
    return [
        (f, m, b)
        for f, m, b in itertools.product(front_vals, middle_vals, back_vals)
        if f <= m <= b
    ]


def valid_rsi_triples(front_vals, middle_vals, back_vals):
    return [
        (f, m, b)
        for f, m, b in itertools.product(front_vals, middle_vals, back_vals)
        if f >= m >= b
    ]


CORE_VRP_TRIPLES = valid_increasing_triples(
    FRONT_GRID["vrp"],
    MIDDLE_GRID["vrp"],
    BACK_GRID["vrp"],
)

CORE_Z_TRIPLES = valid_increasing_triples(
    FRONT_GRID["z"],
    MIDDLE_GRID["z"],
    BACK_GRID["z"],
)

CORE_RSI_TRIPLES = valid_rsi_triples(
    FRONT_GRID["rsi"],
    MIDDLE_GRID["rsi"],
    BACK_GRID["rsi"],
)

CORE_RV21D_TRIPLES = valid_increasing_triples(
    FRONT_GRID["rv21d"],
    MIDDLE_GRID["rv21d"],
    BACK_GRID["rv21d"],
)

spec_rows = []

for spec_num, (vrp, z, rsi, rv21d) in enumerate(
    itertools.product(
        CORE_VRP_TRIPLES,
        CORE_Z_TRIPLES,
        CORE_RSI_TRIPLES,
        CORE_RV21D_TRIPLES,
    ),
    start=1,
):
    spec_id = (
        f"core_phase2_"
        f"vrp_{vrp[0]:.2f}_{vrp[1]:.2f}_{vrp[2]:.2f}_"
        f"z_{z[0]:.2f}_{z[1]:.2f}_{z[2]:.2f}_"
        f"rsi_{int(rsi[0])}_{int(rsi[1])}_{int(rsi[2])}_"
        f"rv21d_{rv21d[0]:.1f}_{rv21d[1]:.1f}_{rv21d[2]:.1f}"
    )

    spec_rows.append({
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_phase2",
        "forecast_denominator": "har_total_simple",

        "front_vrp_log_min": vrp[0],
        "middle_vrp_log_min": vrp[1],
        "back_vrp_log_min": vrp[2],

        "front_z_3m_min": z[0],
        "middle_z_3m_min": z[1],
        "back_z_3m_min": z[2],

        "front_z_1y_min": z[0],
        "middle_z_1y_min": z[1],
        "back_z_1y_min": z[2],

        "front_rsi14_max": rsi[0],
        "middle_rsi14_max": rsi[1],
        "back_rsi14_max": rsi[2],

        "front_rv21d_min": rv21d[0],
        "middle_rv21d_min": rv21d[1],
        "back_rv21d_min": rv21d[2],

        "phase_z_3m_equals_z_1y": True,
    })

core_specs = pd.DataFrame(spec_rows)

grid_summary = pd.DataFrame([{
    "valid_vrp_triples": len(CORE_VRP_TRIPLES),
    "valid_z_triples": len(CORE_Z_TRIPLES),
    "valid_rsi_triples": len(CORE_RSI_TRIPLES),
    "valid_rv21d_triples": len(CORE_RV21D_TRIPLES),
    "total_core_specs": len(core_specs),
}])

print()
print("Core-only Phase 2 grid summary:")
display(grid_summary)

print()
print("Core-only Phase 2 specs sample:")
display(core_specs.head(20))


# -----------------------------
# 4. Build complete eligible matrix panel
# -----------------------------

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "front",
    12: "front",
    15: "front",
    18: "middle",
    21: "middle",
    24: "middle",
    27: "back",
    30: "back",
    33: "back",
}

BUCKETS = ["front", "middle", "back"]
WIN_BAND = 0.0025
MIN_CONDITIONAL_OBS = 20

eligible = panel.loc[panel["is_parameter_sweep_eligible"].astype(bool)].copy()
eligible = eligible.replace([np.inf, -np.inf], np.nan)

needed_for_matrix = [
    "trade_date",
    "date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "actual_dte_for_pnl_day",
    "pnl_per_day",
]

eligible = eligible.dropna(subset=needed_for_matrix).copy()

eligible = eligible.loc[
    eligible["tenor"].isin(TENORS)
    & eligible["actual_dte_for_pnl_day"].gt(0)
].copy()

duplicate_rows = eligible.duplicated(["trade_date", "tenor"]).sum()

if duplicate_rows:
    raise ValueError(f"Eligible panel has duplicate trade_date x tenor rows: {duplicate_rows}")

date_counts = eligible.groupby("trade_date")["tenor"].nunique()
complete_dates = date_counts.loc[date_counts.eq(len(TENORS))].index.tolist()

eligible = eligible.loc[eligible["trade_date"].isin(complete_dates)].copy()

eligible_dates = sorted(eligible["trade_date"].unique())
num_dates = len(eligible_dates)
num_tenors = len(TENORS)

date_lookup = (
    eligible[["trade_date", "date"]]
    .drop_duplicates("trade_date")
    .set_index("trade_date")
    .loc[eligible_dates]
)

date_array = date_lookup["date"].to_numpy()


def pivot_matrix(col, dtype=float):
    mat = (
        eligible
        .pivot(index="trade_date", columns="tenor", values=col)
        .reindex(index=eligible_dates, columns=TENORS)
    )

    if mat.isna().any().any():
        missing = int(mat.isna().sum().sum())
        raise ValueError(f"Matrix for {col} has missing cells after complete-date filter: {missing}")

    return mat.to_numpy(dtype=dtype)


vrp_mat = pivot_matrix("vrp_log")
z3m_mat = pivot_matrix("vrp_z_3m")
z1y_mat = pivot_matrix("vrp_z_1y")
rsi_mat = pivot_matrix("rsi14")
rv21d_mat = pivot_matrix("rv21d")
pnl_mat = pivot_matrix("outcome_value")
actual_dte_mat = pivot_matrix("actual_dte_for_pnl_day")
pnl_per_day_mat = pivot_matrix("pnl_per_day")

valid_outcome_mat = (
    np.isfinite(pnl_mat)
    & np.isfinite(pnl_per_day_mat)
    & np.isfinite(actual_dte_mat)
)

tenor_array = np.array(TENORS, dtype=int)
tenor_bucket_array = np.array([TENOR_BUCKET_MAP[int(t)] for t in tenor_array], dtype=object)

print()
print("Cell 6 input summary:")
display(pd.DataFrame([{
    "eligible_rows_after_complete_date_filter": len(eligible),
    "eligible_dates": num_dates,
    "first_date": pd.to_datetime(date_array.min()),
    "last_date": pd.to_datetime(date_array.max()),
    "unique_tenors": num_tenors,
    "core_specs_to_test": len(core_specs),
    "win_band_bps": WIN_BAND * 10000,
    "min_conditional_obs": MIN_CONDITIONAL_OBS,
    "forecast_model": ", ".join(sorted(eligible["forecast_model"].astype(str).unique())),
}]))


# -----------------------------
# 5. Matrix helper functions
# -----------------------------

def threshold_array_for_spec(spec_row, field_suffix):
    vals = []

    for tenor in TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]
        vals.append(float(spec_row[f"{bucket}_{field_suffix}"]))

    return np.array(vals, dtype=float)


def build_pass_matrix(spec_row):
    vrp_threshold = threshold_array_for_spec(spec_row, "vrp_log_min")
    z3m_threshold = threshold_array_for_spec(spec_row, "z_3m_min")
    z1y_threshold = threshold_array_for_spec(spec_row, "z_1y_min")
    rsi_cap = threshold_array_for_spec(spec_row, "rsi14_max")
    rv21d_floor = threshold_array_for_spec(spec_row, "rv21d_min")

    return (
        (vrp_mat >= vrp_threshold[None, :])
        & (z3m_mat >= z3m_threshold[None, :])
        & (z1y_mat >= z1y_threshold[None, :])
        & (rsi_mat <= rsi_cap[None, :])
        & (rv21d_mat >= rv21d_floor[None, :])
        & valid_outcome_mat
    )


def compute_conditional_arrays(pass_matrix):
    obs = pass_matrix.sum(axis=0).astype(float)
    wins = ((pnl_mat > 0) & pass_matrix).sum(axis=0).astype(float)

    total_pnl = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_mat, 0.0).sum(axis=0),
        0.0,
    )

    total_pnl_day = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_per_day_mat, 0.0).sum(axis=0),
        np.nan,
    )

    total_dte = np.where(
        obs > 0,
        np.where(pass_matrix, actual_dte_mat, 0.0).sum(axis=0),
        np.nan,
    )

    win_prob = np.divide(
        wins,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_day = np.divide(
        total_pnl_day,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_trade = np.divide(
        total_pnl,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    aggregate_pnl_per_day = np.divide(
        total_pnl,
        total_dte,
        out=np.full_like(obs, np.nan, dtype=float),
        where=total_dte > 0,
    )

    worst_trade = np.full(num_tenors, np.nan)
    worst_pnl_day = np.full(num_tenors, np.nan)
    avg_actual_dte = np.full(num_tenors, np.nan)

    for col_idx in range(num_tenors):
        mask = pass_matrix[:, col_idx]

        if mask.any():
            worst_trade[col_idx] = np.nanmin(pnl_mat[mask, col_idx])
            worst_pnl_day[col_idx] = np.nanmin(pnl_per_day_mat[mask, col_idx])
            avg_actual_dte[col_idx] = np.nanmean(actual_dte_mat[mask, col_idx])

    sample_ok = obs >= MIN_CONDITIONAL_OBS

    return {
        "obs": obs,
        "wins": wins,
        "win_prob": win_prob,
        "avg_pnl_per_day": avg_pnl_per_day,
        "aggregate_pnl_per_day": aggregate_pnl_per_day,
        "avg_pnl_per_trade": avg_pnl_per_trade,
        "total_pnl": total_pnl,
        "worst_trade": worst_trade,
        "worst_pnl_day": worst_pnl_day,
        "avg_actual_dte": avg_actual_dte,
        "sample_ok": sample_ok,
    }


def select_cols_from_pass_and_stats(pass_matrix, stats):
    obs = stats["obs"]
    win = stats["win_prob"]
    avg_day = stats["avg_pnl_per_day"]
    agg_day = stats["aggregate_pnl_per_day"]
    sample_ok = stats["sample_ok"]

    has_stats = (
        (obs > 0)
        & np.isfinite(win)
        & np.isfinite(avg_day)
        & np.isfinite(agg_day)
    )

    active = pass_matrix & has_stats[None, :]

    active_sample_ok = active & sample_ok[None, :]
    row_has_sample_ok = active_sample_ok.any(axis=1)

    ranking_universe = np.where(
        row_has_sample_ok[:, None],
        active_sample_ok,
        active,
    )

    win_masked = np.where(
        ranking_universe,
        win[None, :],
        -np.inf,
    )

    best_win = win_masked.max(axis=1)
    row_has_any = np.isfinite(best_win)

    inside_win_band = (
        ranking_universe
        & row_has_any[:, None]
        & (win[None, :] >= (best_win[:, None] - WIN_BAND))
    )

    selected_col = np.full(num_dates, -1, dtype=int)

    best_avg = np.full(num_dates, -np.inf)
    best_agg = np.full(num_dates, -np.inf)
    best_win_for_tie = np.full(num_dates, -np.inf)
    best_tenor = np.full(num_dates, -np.inf)

    for col_idx, tenor in enumerate(TENORS):
        cand = inside_win_band[:, col_idx]

        avg_val = avg_day[col_idx]
        agg_val = agg_day[col_idx]
        win_val = win[col_idx]
        tenor_val = float(tenor)

        better = (
            cand
            & (
                (avg_val > best_avg)
                | ((avg_val == best_avg) & (agg_val > best_agg))
                | ((avg_val == best_avg) & (agg_val == best_agg) & (win_val > best_win_for_tie))
                | ((avg_val == best_avg) & (agg_val == best_agg) & (win_val == best_win_for_tie) & (tenor_val > best_tenor))
            )
        )

        selected_col[better] = col_idx
        best_avg[better] = avg_val
        best_agg[better] = agg_val
        best_win_for_tie[better] = win_val
        best_tenor[better] = tenor_val

    return selected_col


def profit_factor(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    gains = values[values > 0].sum()
    losses = values[values < 0].sum()

    if losses < 0:
        return gains / abs(losses)

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def summarize_selected_cols(selected_cols, prefix):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    if date_idx.size == 0:
        return {
            f"{prefix}_trades": 0,
            f"{prefix}_frequency": 0.0,
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_day_weighted_pnl_per_day": np.nan,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade_pnl": np.nan,
            f"{prefix}_worst_pnl_per_day": np.nan,
            f"{prefix}_best_trade_pnl": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_avg_selected_tenor": np.nan,
        }

    cols = selected_cols[date_idx]

    pnl = pnl_mat[date_idx, cols]
    pnl_day = pnl_per_day_mat[date_idx, cols]
    dte = actual_dte_mat[date_idx, cols]
    selected_tenors = tenor_array[cols]

    total_dte = np.nansum(dte)

    if total_dte > 0:
        exposure_day_weighted_pnl_per_day = np.nansum(pnl) / total_dte
    else:
        exposure_day_weighted_pnl_per_day = np.nan

    return {
        f"{prefix}_trades": int(date_idx.size),
        f"{prefix}_frequency": float(date_idx.size / num_dates) if num_dates else np.nan,
        f"{prefix}_win_rate": float(np.nanmean(pnl > 0)),
        f"{prefix}_total_pnl": float(np.nansum(pnl)),
        f"{prefix}_avg_pnl_per_trade": float(np.nanmean(pnl)),
        f"{prefix}_avg_pnl_per_day": float(np.nanmean(pnl_day)),
        f"{prefix}_exposure_day_weighted_pnl_per_day": float(exposure_day_weighted_pnl_per_day),
        f"{prefix}_profit_factor": float(profit_factor(pnl)),
        f"{prefix}_worst_trade_pnl": float(np.nanmin(pnl)),
        f"{prefix}_worst_pnl_per_day": float(np.nanmin(pnl_day)),
        f"{prefix}_best_trade_pnl": float(np.nanmax(pnl)),
        f"{prefix}_avg_actual_dte": float(np.nanmean(dte)),
        f"{prefix}_avg_selected_tenor": float(np.nanmean(selected_tenors)),
    }


def bucket_metrics(selected_cols):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    out = {}

    for bucket in BUCKETS:
        out[f"{bucket}_selected_trades"] = 0
        out[f"{bucket}_selected_win_rate"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_trade"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_total_pnl"] = 0.0
        out[f"{bucket}_selected_worst_trade_pnl"] = np.nan

    if date_idx.size == 0:
        return out

    cols = selected_cols[date_idx]
    selected_buckets = tenor_bucket_array[cols]

    for bucket in BUCKETS:
        bucket_mask = selected_buckets == bucket

        if not bucket_mask.any():
            continue

        bucket_date_idx = date_idx[bucket_mask]
        bucket_cols = cols[bucket_mask]

        pnl = pnl_mat[bucket_date_idx, bucket_cols]
        pnl_day = pnl_per_day_mat[bucket_date_idx, bucket_cols]
        dte = actual_dte_mat[bucket_date_idx, bucket_cols]

        total_dte = np.nansum(dte)

        out[f"{bucket}_selected_trades"] = int(bucket_mask.sum())
        out[f"{bucket}_selected_win_rate"] = float(np.nanmean(pnl > 0))
        out[f"{bucket}_selected_avg_pnl_per_trade"] = float(np.nanmean(pnl))
        out[f"{bucket}_selected_avg_pnl_per_day"] = float(np.nanmean(pnl_day))
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = (
            float(np.nansum(pnl) / total_dte) if total_dte > 0 else np.nan
        )
        out[f"{bucket}_selected_total_pnl"] = float(np.nansum(pnl))
        out[f"{bucket}_selected_worst_trade_pnl"] = float(np.nanmin(pnl))

    return out


def tenor_selection_metrics(selected_cols, spec_id):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx_all = np.where(selected_cols >= 0)[0]

    rows = []

    for col_idx, tenor in enumerate(TENORS):
        tenor_mask = selected_cols == col_idx
        date_idx = np.where(tenor_mask)[0]

        if date_idx.size == 0:
            rows.append({
                "spec_id": spec_id,
                "selected_tenor": int(tenor),
                "selected_tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
                "selected_trades": 0,
                "selected_frequency_total_dates": 0.0,
                "selected_share_of_trades": 0.0,
                "selected_win_rate": np.nan,
                "selected_avg_pnl_per_trade": np.nan,
                "selected_avg_pnl_per_day": np.nan,
                "selected_exposure_day_weighted_pnl_per_day": np.nan,
                "selected_total_pnl": 0.0,
                "selected_worst_trade_pnl": np.nan,
                "selected_worst_pnl_per_day": np.nan,
                "selected_avg_actual_dte": np.nan,
            })
            continue

        pnl = pnl_mat[date_idx, col_idx]
        pnl_day = pnl_per_day_mat[date_idx, col_idx]
        dte = actual_dte_mat[date_idx, col_idx]

        total_dte = np.nansum(dte)

        rows.append({
            "spec_id": spec_id,
            "selected_tenor": int(tenor),
            "selected_tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "selected_trades": int(date_idx.size),
            "selected_frequency_total_dates": float(date_idx.size / num_dates) if num_dates else np.nan,
            "selected_share_of_trades": float(date_idx.size / date_idx_all.size) if date_idx_all.size else 0.0,
            "selected_win_rate": float(np.nanmean(pnl > 0)),
            "selected_avg_pnl_per_trade": float(np.nanmean(pnl)),
            "selected_avg_pnl_per_day": float(np.nanmean(pnl_day)),
            "selected_exposure_day_weighted_pnl_per_day": (
                float(np.nansum(pnl) / total_dte) if total_dte > 0 else np.nan
            ),
            "selected_total_pnl": float(np.nansum(pnl)),
            "selected_worst_trade_pnl": float(np.nanmin(pnl)),
            "selected_worst_pnl_per_day": float(np.nanmin(pnl_day)),
            "selected_avg_actual_dte": float(np.nanmean(dte)),
        })

    return rows


def append_conditional_rows(rows, spec_id, stats):
    for col_idx, tenor in enumerate(TENORS):
        rows.append({
            "spec_id": spec_id,
            "role": "Core",
            "tenor": int(tenor),
            "tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "conditional_obs": int(stats["obs"][col_idx]),
            "conditional_wins": int(stats["wins"][col_idx]),
            "conditional_win_probability": stats["win_prob"][col_idx],
            "conditional_avg_pnl_per_day": stats["avg_pnl_per_day"][col_idx],
            "conditional_aggregate_pnl_per_day": stats["aggregate_pnl_per_day"][col_idx],
            "conditional_avg_pnl_per_trade": stats["avg_pnl_per_trade"][col_idx],
            "conditional_total_pnl": stats["total_pnl"][col_idx],
            "conditional_worst_trade_pnl": stats["worst_trade"][col_idx],
            "conditional_worst_pnl_per_day": stats["worst_pnl_day"][col_idx],
            "conditional_avg_actual_dte": stats["avg_actual_dte"][col_idx],
            "conditional_sample_ok": bool(stats["sample_ok"][col_idx]),
        })


# -----------------------------
# 6. Run Core-only Phase 2 sweep
# -----------------------------

summary_rows = []
conditional_rows = []
tenor_selection_rows = []

total_specs = len(core_specs)

print()
print(f"Running Core-only Phase 2 sweep: {total_specs:,} specs")

for i, (_, spec_row) in enumerate(core_specs.iterrows(), start=1):
    spec_id = spec_row["spec_id"]

    pass_matrix = build_pass_matrix(spec_row)
    candidate_dates = pass_matrix.any(axis=1)
    candidate_rows = int(pass_matrix.sum())

    stats = compute_conditional_arrays(pass_matrix)
    selected_cols = select_cols_from_pass_and_stats(pass_matrix, stats)

    row = {
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_phase2",
        "candidate_rows": candidate_rows,
        "candidate_dates": int(candidate_dates.sum()),
        "candidate_frequency": float(candidate_dates.sum() / num_dates) if num_dates else np.nan,

        "front_vrp_log_min": spec_row["front_vrp_log_min"],
        "middle_vrp_log_min": spec_row["middle_vrp_log_min"],
        "back_vrp_log_min": spec_row["back_vrp_log_min"],

        "front_z_3m_min": spec_row["front_z_3m_min"],
        "middle_z_3m_min": spec_row["middle_z_3m_min"],
        "back_z_3m_min": spec_row["back_z_3m_min"],

        "front_z_1y_min": spec_row["front_z_1y_min"],
        "middle_z_1y_min": spec_row["middle_z_1y_min"],
        "back_z_1y_min": spec_row["back_z_1y_min"],

        "front_rsi14_max": spec_row["front_rsi14_max"],
        "middle_rsi14_max": spec_row["middle_rsi14_max"],
        "back_rsi14_max": spec_row["back_rsi14_max"],

        "front_rv21d_min": spec_row["front_rv21d_min"],
        "middle_rv21d_min": spec_row["middle_rv21d_min"],
        "back_rv21d_min": spec_row["back_rv21d_min"],
    }

    row.update(summarize_selected_cols(selected_cols, "core"))
    row.update(bucket_metrics(selected_cols))

    summary_rows.append(row)
    append_conditional_rows(conditional_rows, spec_id, stats)
    tenor_selection_rows.extend(tenor_selection_metrics(selected_cols, spec_id))

    if i % 250 == 0 or i == total_specs:
        print(f"Core specs processed: {i:,} / {total_specs:,}")

core_summary = pd.DataFrame(summary_rows)
conditional_stats = pd.DataFrame(conditional_rows)
tenor_selection_summary = pd.DataFrame(tenor_selection_rows)


# -----------------------------
# 7. Add labels / flags / rankings
# -----------------------------

core_summary["vrp_triple"] = (
    core_summary["front_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_vrp_log_min"].map("{:.2f}".format)
)

core_summary["z_triple"] = (
    core_summary["front_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_z_3m_min"].map("{:.2f}".format)
)

core_summary["rsi_triple"] = (
    core_summary["front_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["middle_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["back_rsi14_max"].astype(int).astype(str)
)

core_summary["rv21d_triple"] = (
    core_summary["front_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["middle_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["back_rv21d_min"].map("{:.1f}".format)
)

core_summary["meets_core_win_85"] = core_summary["core_win_rate"].ge(0.85)
core_summary["meets_core_win_82"] = core_summary["core_win_rate"].ge(0.82)
core_summary["meets_core_win_80"] = core_summary["core_win_rate"].ge(0.80)

core_summary["core_win_gap_to_85"] = 0.85 - core_summary["core_win_rate"]

core_summary_by_win = core_summary.sort_values(
    [
        "core_win_rate",
        "core_frequency",
        "core_avg_pnl_per_day",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

core_summary_by_win["win_rate_rank"] = np.arange(1, len(core_summary_by_win) + 1)

core_summary_by_frequency_targets_first = core_summary.sort_values(
    [
        "meets_core_win_85",
        "core_frequency",
        "core_win_rate",
        "core_avg_pnl_per_day",
        "core_total_pnl",
        "core_worst_trade_pnl",
    ],
    ascending=[False, False, False, False, False, False],
).reset_index(drop=True)

core_summary_by_frequency_targets_first["frequency_rank_with_targets_first"] = (
    np.arange(1, len(core_summary_by_frequency_targets_first) + 1)
)

core_review = core_summary.merge(
    core_summary_by_win[["spec_id", "win_rate_rank"]],
    on="spec_id",
    how="left",
    validate="one_to_one",
)

core_review = core_review.merge(
    core_summary_by_frequency_targets_first[["spec_id", "frequency_rank_with_targets_first"]],
    on="spec_id",
    how="left",
    validate="one_to_one",
)


# -----------------------------
# 8. Parameter group diagnostics
# -----------------------------

def parameter_group_summary(df, group_col):
    out = (
        df
        .groupby(group_col)
        .agg(
            specs=("spec_id", "nunique"),
            max_core_win_rate=("core_win_rate", "max"),
            median_core_win_rate=("core_win_rate", "median"),
            max_core_frequency=("core_frequency", "max"),
            median_core_frequency=("core_frequency", "median"),
            max_core_avg_pnl_per_day=("core_avg_pnl_per_day", "max"),
            median_core_avg_pnl_per_day=("core_avg_pnl_per_day", "median"),
            max_core_total_pnl=("core_total_pnl", "max"),
            specs_core_win_85_plus=("meets_core_win_85", "sum"),
            specs_core_win_82_plus=("meets_core_win_82", "sum"),
            specs_core_win_80_plus=("meets_core_win_80", "sum"),
        )
        .reset_index()
        .sort_values(
            [
                "max_core_win_rate",
                "max_core_frequency",
                "max_core_avg_pnl_per_day",
            ],
            ascending=[False, False, False],
        )
    )

    return out


core_by_vrp = parameter_group_summary(core_review, "vrp_triple")
core_by_z = parameter_group_summary(core_review, "z_triple")
core_by_rsi = parameter_group_summary(core_review, "rsi_triple")
core_by_rv21d = parameter_group_summary(core_review, "rv21d_triple")


# -----------------------------
# 9. Tenor diagnostics for top specs
# -----------------------------

def tenor_table_for_specs(spec_ids):
    out = tenor_selection_summary.loc[
        tenor_selection_summary["spec_id"].isin(spec_ids)
    ].merge(
        core_review[
            [
                "spec_id",
                "win_rate_rank",
                "frequency_rank_with_targets_first",
                "core_trades",
                "core_frequency",
                "core_win_rate",
                "core_avg_pnl_per_day",
                "core_total_pnl",
                "vrp_triple",
                "z_triple",
                "rsi_triple",
                "rv21d_triple",
            ]
        ],
        on="spec_id",
        how="left",
        validate="many_to_one",
    )

    return out.sort_values(
        ["win_rate_rank", "selected_tenor"],
        ascending=[True, True],
    )


top_win_spec_ids = (
    core_review
    .sort_values(
        ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

target_frequency_spec_ids = (
    core_review
    .loc[core_review["meets_core_win_85"]]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

above_82_frequency_spec_ids = (
    core_review
    .loc[core_review["core_win_rate"].ge(0.82)]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

tenor_top_win = tenor_table_for_specs(top_win_spec_ids)
tenor_target_frequency = tenor_table_for_specs(target_frequency_spec_ids)
tenor_above_82_frequency = tenor_table_for_specs(above_82_frequency_spec_ids)


# -----------------------------
# 10. Display outputs
# -----------------------------

print()
print("Core-only Phase 2 sweep summary:")
display(pd.DataFrame([{
    "core_specs_tested": len(core_review),
    "specs_core_win_85_plus": int(core_review["meets_core_win_85"].sum()),
    "specs_core_win_82_plus": int(core_review["meets_core_win_82"].sum()),
    "specs_core_win_80_plus": int(core_review["meets_core_win_80"].sum()),
    "max_core_win_rate": core_review["core_win_rate"].max(),
    "median_core_win_rate": core_review["core_win_rate"].median(),
    "max_core_frequency": core_review["core_frequency"].max(),
    "median_core_frequency": core_review["core_frequency"].median(),
    "max_core_avg_pnl_per_day": core_review["core_avg_pnl_per_day"].max(),
    "median_core_avg_pnl_per_day": core_review["core_avg_pnl_per_day"].median(),
}]))

review_cols = [
    "frequency_rank_with_targets_first",
    "win_rate_rank",
    "meets_core_win_85",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_win_gap_to_85",
    "core_avg_pnl_per_trade",
    "core_avg_pnl_per_day",
    "core_exposure_day_weighted_pnl_per_day",
    "core_total_pnl",
    "core_profit_factor",
    "core_worst_trade_pnl",
    "core_worst_pnl_per_day",
    "core_avg_actual_dte",
    "core_avg_selected_tenor",

    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_trade",
    "front_selected_avg_pnl_per_day",
    "front_selected_total_pnl",
    "front_selected_worst_trade_pnl",

    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_trade",
    "middle_selected_avg_pnl_per_day",
    "middle_selected_total_pnl",
    "middle_selected_worst_trade_pnl",

    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_trade",
    "back_selected_avg_pnl_per_day",
    "back_selected_total_pnl",
    "back_selected_worst_trade_pnl",

    "vrp_triple",
    "z_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

tenor_cols = [
    "win_rate_rank",
    "frequency_rank_with_targets_first",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_avg_pnl_per_day",
    "core_total_pnl",

    "selected_tenor",
    "selected_tenor_bucket",
    "selected_trades",
    "selected_frequency_total_dates",
    "selected_share_of_trades",
    "selected_win_rate",
    "selected_avg_pnl_per_trade",
    "selected_avg_pnl_per_day",
    "selected_exposure_day_weighted_pnl_per_day",
    "selected_total_pnl",
    "selected_worst_trade_pnl",
    "selected_worst_pnl_per_day",
    "selected_avg_actual_dte",

    "vrp_triple",
    "z_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

print()
print("Top Core specs by win rate:")
display(
    core_review
    .sort_values(
        ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    [review_cols]
    .head(100)
)

print()
print("Top Core specs with win rate >= 85%, ranked by frequency:")
display(
    core_review
    .loc[core_review["meets_core_win_85"], review_cols]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Top Core specs with win rate >= 85%, ranked by average P&L/day:")
display(
    core_review
    .loc[core_review["meets_core_win_85"], review_cols]
    .sort_values(
        ["core_avg_pnl_per_day", "core_frequency", "core_win_rate", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Highest-frequency Core specs with win rate >= 82%:")
display(
    core_review
    .loc[core_review["core_win_rate"].ge(0.82), review_cols]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Tenor / DTE selection diagnostics for top Core specs by win rate:")
display(tenor_top_win[tenor_cols].head(180))

print()
print("Tenor / DTE selection diagnostics for Core specs with win rate >= 85%, ranked by frequency:")
display(tenor_target_frequency[tenor_cols].head(180))

print()
print("Tenor / DTE selection diagnostics for highest-frequency Core specs with win rate >= 82%:")
display(tenor_above_82_frequency[tenor_cols].head(180))

print()
print("Core parameter diagnostics by VRP triple:")
display(core_by_vrp)

print()
print("Core parameter diagnostics by z-score triple:")
display(core_by_z)

print()
print("Core parameter diagnostics by RSI cap triple:")
display(core_by_rsi)

print()
print("Core parameter diagnostics by RV21D floor triple:")
display(core_by_rv21d)


# -----------------------------
# 11. Save outputs
# -----------------------------

safe_start = pd.to_datetime(date_array.min()).strftime("%Y%m%d")
safe_end = pd.to_datetime(date_array.max()).strftime("%Y%m%d")

core_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_specs_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

grid_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_grid_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

conditional_stats_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_conditional_stats_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_selection_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_tenor_selection_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

top_by_win_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_top_by_win_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_win85_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_pnl_day_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_win85_by_pnl_day_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

above_82_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_win82_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_top_win_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_tenor_top_by_win_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_target_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_tenor_win85_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_above_82_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_tenor_win82_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_vrp_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_by_vrp_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_z_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_by_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rsi_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_by_rsi_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rv21d_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell6_core_only_phase2_by_rv21d_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_specs.to_csv(core_specs_path, index=False)
grid_summary.to_csv(grid_summary_path, index=False)
core_review.to_csv(core_summary_path, index=False)
conditional_stats.to_csv(conditional_stats_path, index=False)
tenor_selection_summary.to_csv(tenor_selection_path, index=False)

core_review.sort_values(
    ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).head(1000).to_csv(top_by_win_path, index=False)

core_review.loc[
    core_review["meets_core_win_85"]
].sort_values(
    ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(target_by_frequency_path, index=False)

core_review.loc[
    core_review["meets_core_win_85"]
].sort_values(
    ["core_avg_pnl_per_day", "core_frequency", "core_win_rate", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(target_by_pnl_day_path, index=False)

core_review.loc[
    core_review["core_win_rate"].ge(0.82)
].sort_values(
    ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(above_82_frequency_path, index=False)

tenor_top_win.to_csv(tenor_top_win_path, index=False)
tenor_target_frequency.to_csv(tenor_target_frequency_path, index=False)
tenor_above_82_frequency.to_csv(tenor_above_82_frequency_path, index=False)

core_by_vrp.to_csv(core_by_vrp_path, index=False)
core_by_z.to_csv(core_by_z_path, index=False)
core_by_rsi.to_csv(core_by_rsi_path, index=False)
core_by_rv21d.to_csv(core_by_rv21d_path, index=False)

print()
print("Cell 6 outputs saved:")
print(f"Core specs:                    {core_specs_path}")
print(f"Grid summary:                  {grid_summary_path}")
print(f"Core summary:                  {core_summary_path}")
print(f"Conditional stats:             {conditional_stats_path}")
print(f"Tenor selection summary:       {tenor_selection_path}")
print(f"Top by win rate:               {top_by_win_path}")
print(f"Win >=85 by frequency:         {target_by_frequency_path}")
print(f"Win >=85 by P&L/day:           {target_by_pnl_day_path}")
print(f"Win >=82 by frequency:         {above_82_frequency_path}")
print(f"Tenor top by win:              {tenor_top_win_path}")
print(f"Tenor win >=85 by frequency:   {tenor_target_frequency_path}")
print(f"Tenor win >=82 by frequency:   {tenor_above_82_frequency_path}")
print(f"Core by VRP:                   {core_by_vrp_path}")
print(f"Core by z-score:               {core_by_z_path}")
print(f"Core by RSI:                   {core_by_rsi_path}")
print(f"Core by RV21D:                 {core_by_rv21d_path}")

print()
print("Cell 6 complete.")

Cell 6: Core-only revised Phase 2 sweep
Clean panel path: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
Run timestamp:    20260704_100154

Actual DTE source:
C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet

Actual DTE availability:


,rows,actual_dte_non_null,actual_dte_non_null_pct,fallback_to_tenor_rows,fallback_to_tenor_pct
0,14565,14565,1.0,0,0.0



Core-only Phase 2 grid summary:


,valid_vrp_triples,valid_z_triples,valid_rsi_triples,valid_rv21d_triples,total_core_specs
0,2,7,24,4,1344



Core-only Phase 2 specs sample:


...


Core parameter diagnostics by RSI cap triple:


,rsi_triple,specs,max_core_win_rate,median_core_win_rate,max_core_frequency,median_core_frequency,max_core_avg_pnl_per_day,median_core_avg_pnl_per_day,max_core_total_pnl,specs_core_win_85_plus,specs_core_win_82_plus,specs_core_win_80_plus
10,70 / 68 / 65,56,0.829268,0.820131,0.199594,0.181326,432.567309,383.488801,2.533807e+06,0,28,56
18,71 / 68 / 65,56,0.829268,0.820131,0.199594,0.181326,432.567309,383.488801,2.533807e+06,0,28,56
11,70 / 68 / 66,56,0.829268,0.820131,0.199594,0.181326,431.229694,382.177232,2.533858e+06,0,28,56
19,71 / 68 / 66,56,0.829268,0.820131,0.199594,0.181326,431.229694,382.177232,2.533858e+06,0,28,56
12,70 / 68 / 68,56,0.825911,0.817111,0.200271,0.182003,427.636439,379.005794,2.536017e+06,0,17,56
20,71 / 68 / 68,56,0.825911,0.817111,0.200271,0.182003,427.636439,379.005794,2.536017e+06,0,17,56
13,70 / 69 / 65,56,0.825911,0.816872,0.199594,0.182003,428.352749,380.478891,2.534642e+06,0,17,56
21,71 / 69 / 65,56,0.825911,0.816872,0.199594,0.182003,428.352749,380.478891,2.534642e+06,0,17,56
14,70 / 69 / 66,56,0.825911,0.816872,0.199594,0.182003,427.020443,379.175199,2.534693e+06,0,17,56
22,71 / 69 / 66,56,0.825911,0.816872,0.199594,0.182003,427.020443,379.175199,2.534693e+06,0,17,56



Core parameter diagnostics by RV21D floor triple:


,rv21d_triple,specs,max_core_win_rate,median_core_win_rate,max_core_frequency,median_core_frequency,max_core_avg_pnl_per_day,median_core_avg_pnl_per_day,max_core_total_pnl,specs_core_win_85_plus,specs_core_win_82_plus,specs_core_win_80_plus
3,8.5 / 8.5 / 8.5,336,0.829268,0.818182,0.194858,0.175237,431.673362,376.192830,2.503340e+06,0,112,336
2,8.0 / 8.5 / 8.5,336,0.828685,0.818182,0.200271,0.179635,432.567309,380.067665,2.531038e+06,0,127,336
0,8.0 / 8.0 / 8.0,336,0.826087,0.815789,0.200271,0.179973,426.269040,374.262849,2.534693e+06,0,56,336
1,8.0 / 8.0 / 8.5,336,0.822222,0.812000,0.200271,0.179973,426.222051,372.669034,2.536852e+06,0,8,336



Cell 6 outputs saved:
Core specs:                    C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell6_core_only_phase2_specs_20200701_20260519_20260704_100154.csv
Grid summary:                  C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell6_core_only_phase2_grid_summary_20200701_20260519_20260704_100154.csv
Core summary:                  C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell6_core_only_phase2_summary_20200701_20260519_20260704_100154.csv
Conditional stats:             C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell6_core_only_phase2_conditional_stats_20200701_20260519_20260704_100154.csv
Tenor selection summary:       C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell6_core_only_phase2_tenor_selection_summary_2

In [9]:
# Cell 7: Core-only Phase 3 sweep — untied z-scores
#
# Approved scope:
#   - Core only.
#   - No Secondary.
#   - No sizing.
#   - No final lock.
#   - 3m z-score and 1y z-score are tested independently.
#   - Tenor selection uses strict highest conditional win probability first.
#   - No 25 bps win-rate band.
#
# Goal:
#   - Core win rate >= 85%.
#   - Then maximize Core frequency.
#   - Then evaluate P&L/day and exact DTE / tenor selection mix.
#
# User-approved grid:
#
#   Front 9D / 12D / 15D:
#       VRP:      0.60 locked
#       3m z:     [0.50, 0.60, 0.70]
#       1y z:     [0.50, 0.60, 0.70]
#       RSI:      70 locked
#       RV21D:    [7.5, 8.0]
#
#   Middle 18D / 21D / 24D:
#       VRP:      0.70 locked
#       3m z:     [0.70, 0.75, 0.80]
#       1y z:     [0.70, 0.75, 0.80]
#       RSI:      68 locked
#       RV21D:    8.5 locked
#
#   Back 27D / 30D / 33D:
#       VRP:      0.70 locked
#       3m z:     [0.70, 0.75, 0.80]
#       1y z:     [0.70, 0.75, 0.80]
#       RSI:      [65, 66, 67]
#       RV21D:    8.5 locked
#
# Constraints:
#   - Front 3m z <= Middle 3m z <= Back 3m z
#   - Front 1y z <= Middle 1y z <= Back 1y z
#   - Front VRP <= Middle VRP <= Back VRP
#   - Front RV21D <= Middle RV21D <= Back RV21D
#   - Front RSI >= Middle RSI >= Back RSI
#
# Expected valid grid size:
#   - VRP triples:       1
#   - 3m z triples:      18
#   - 1y z triples:      18
#   - RSI triples:       3
#   - RV21D triples:     2
#   - Total Core specs:  1,944
#
# Tenor selection rule when multiple tenors pass:
#   1. Prefer tenors with conditional sample size >= 20.
#   2. Highest conditional win probability.
#   3. Highest conditional average P&L/day.
#   4. Highest conditional aggregate P&L/day.
#   5. Longer tenor.
#
# P&L/day:
#   normalized_pnl_dollars / actual_dte if actual_dte is available.
#   Falls back to normalized_pnl_dollars / tenor if actual_dte is unavailable.

from pathlib import Path
import itertools
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest clean panel
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No clean panel found in {PARAM_SWEEP_PROCESSED_DIR}")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]

print("Cell 7: Core-only Phase 3 sweep — untied z-scores")
print(f"Clean panel path: {CLEAN_PANEL_PATH}")
print(f"Run timestamp:    {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_panel_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "has_outcome",
    "is_parameter_sweep_eligible",
]

missing_panel_cols = [c for c in required_panel_cols if c not in panel.columns]

if missing_panel_cols:
    raise ValueError(f"Clean panel missing required columns: {missing_panel_cols}")


# -----------------------------
# 2. Actual DTE helper
# -----------------------------

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_actual_dte_lookup():
    outcome_discovery_candidates = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not outcome_discovery_candidates:
        return None, "no_outcome_discovery_file"

    discovery_path = outcome_discovery_candidates[-1]
    discovery = pd.read_csv(discovery_path)
    discovery.columns = [str(c).strip() for c in discovery.columns]

    file_col = "chosen_file" if "chosen_file" in discovery.columns else "path"

    if file_col not in discovery.columns:
        return None, "discovery_missing_path_column"

    preferred = discovery.copy()

    if "eligible_for_backtest" in preferred.columns:
        preferred = preferred.loc[preferred["eligible_for_backtest"].astype(bool)].copy()

    if preferred.empty:
        return None, "no_eligible_outcome_files"

    if "score" in preferred.columns:
        preferred["_score"] = pd.to_numeric(preferred["score"], errors="coerce")
    else:
        preferred["_score"] = 0.0

    if "chosen_outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["chosen_outcome_col"].eq("normalized_pnl_dollars")
    elif "outcome_col" in preferred.columns:
        preferred["_preferred_pnl"] = preferred["outcome_col"].eq("normalized_pnl_dollars")
    else:
        preferred["_preferred_pnl"] = False

    preferred = preferred.sort_values(
        ["_preferred_pnl", "_score"],
        ascending=[False, False],
    )

    for _, row in preferred.iterrows():
        path = Path(row[file_col])

        if not path.exists():
            continue

        try:
            raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
        except Exception:
            continue

        raw.columns = [str(c).strip() for c in raw.columns]

        trade_date_col = first_existing_col(raw, ["trade_date", "trade_dt", "date"])
        tenor_col = first_existing_col(raw, ["tenor", "entry_tenor", "target_tenor"])
        actual_dte_col = first_existing_col(raw, ["actual_dte"])

        if trade_date_col is None or tenor_col is None or actual_dte_col is None:
            continue

        lookup = raw[[trade_date_col, tenor_col, actual_dte_col]].copy()
        lookup = lookup.rename(columns={
            trade_date_col: "trade_date",
            tenor_col: "tenor",
            actual_dte_col: "actual_dte_from_outcomes",
        })

        lookup["trade_date"] = pd.to_numeric(lookup["trade_date"], errors="coerce").astype("Int64")
        lookup["tenor"] = pd.to_numeric(lookup["tenor"], errors="coerce").astype("Int64")
        lookup["actual_dte_from_outcomes"] = pd.to_numeric(
            lookup["actual_dte_from_outcomes"],
            errors="coerce",
        )

        lookup = lookup.dropna(subset=["trade_date", "tenor"])
        lookup["trade_date"] = lookup["trade_date"].astype(int)
        lookup["tenor"] = lookup["tenor"].astype(int)
        lookup = lookup.drop_duplicates(["trade_date", "tenor"])

        return lookup, str(path)

    return None, "actual_dte_not_found_in_outcome_files"


if "actual_dte" not in panel.columns:
    actual_dte_lookup, actual_dte_source = load_actual_dte_lookup()

    if actual_dte_lookup is not None:
        panel = panel.merge(
            actual_dte_lookup,
            on=["trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )
        panel["actual_dte"] = panel["actual_dte_from_outcomes"]
        panel = panel.drop(columns=["actual_dte_from_outcomes"])
    else:
        panel["actual_dte"] = np.nan
else:
    actual_dte_source = "clean_panel_existing_actual_dte"

panel["actual_dte"] = pd.to_numeric(panel["actual_dte"], errors="coerce")
panel["actual_dte_for_pnl_day"] = panel["actual_dte"]

fallback_mask = (
    panel["actual_dte_for_pnl_day"].isna()
    | panel["actual_dte_for_pnl_day"].le(0)
)

panel.loc[fallback_mask, "actual_dte_for_pnl_day"] = panel.loc[fallback_mask, "tenor"]
panel["pnl_per_day"] = panel["outcome_value"] / panel["actual_dte_for_pnl_day"]

print()
print("Actual DTE source:")
print(actual_dte_source)

print()
print("Actual DTE availability:")
display(pd.DataFrame([{
    "rows": len(panel),
    "actual_dte_non_null": int(panel["actual_dte"].notna().sum()),
    "actual_dte_non_null_pct": float(panel["actual_dte"].notna().mean()),
    "fallback_to_tenor_rows": int(fallback_mask.sum()),
    "fallback_to_tenor_pct": float(fallback_mask.mean()),
}]))


# -----------------------------
# 3. Build Core-only Phase 3 grid
# -----------------------------

FRONT_GRID = {
    "vrp": [0.60],
    "z_3m": [0.50, 0.60, 0.70],
    "z_1y": [0.50, 0.60, 0.70],
    "rsi": [70],
    "rv21d": [7.5, 8.0],
}

MIDDLE_GRID = {
    "vrp": [0.70],
    "z_3m": [0.70, 0.75, 0.80],
    "z_1y": [0.70, 0.75, 0.80],
    "rsi": [68],
    "rv21d": [8.5],
}

BACK_GRID = {
    "vrp": [0.70],
    "z_3m": [0.70, 0.75, 0.80],
    "z_1y": [0.70, 0.75, 0.80],
    "rsi": [65, 66, 67],
    "rv21d": [8.5],
}


def valid_increasing_triples(front_vals, middle_vals, back_vals):
    return [
        (f, m, b)
        for f, m, b in itertools.product(front_vals, middle_vals, back_vals)
        if f <= m <= b
    ]


def valid_rsi_triples(front_vals, middle_vals, back_vals):
    return [
        (f, m, b)
        for f, m, b in itertools.product(front_vals, middle_vals, back_vals)
        if f >= m >= b
    ]


CORE_VRP_TRIPLES = valid_increasing_triples(
    FRONT_GRID["vrp"],
    MIDDLE_GRID["vrp"],
    BACK_GRID["vrp"],
)

CORE_Z3M_TRIPLES = valid_increasing_triples(
    FRONT_GRID["z_3m"],
    MIDDLE_GRID["z_3m"],
    BACK_GRID["z_3m"],
)

CORE_Z1Y_TRIPLES = valid_increasing_triples(
    FRONT_GRID["z_1y"],
    MIDDLE_GRID["z_1y"],
    BACK_GRID["z_1y"],
)

CORE_RSI_TRIPLES = valid_rsi_triples(
    FRONT_GRID["rsi"],
    MIDDLE_GRID["rsi"],
    BACK_GRID["rsi"],
)

CORE_RV21D_TRIPLES = valid_increasing_triples(
    FRONT_GRID["rv21d"],
    MIDDLE_GRID["rv21d"],
    BACK_GRID["rv21d"],
)

spec_rows = []

for spec_num, (vrp, z3m, z1y, rsi, rv21d) in enumerate(
    itertools.product(
        CORE_VRP_TRIPLES,
        CORE_Z3M_TRIPLES,
        CORE_Z1Y_TRIPLES,
        CORE_RSI_TRIPLES,
        CORE_RV21D_TRIPLES,
    ),
    start=1,
):
    spec_id = (
        f"core_phase3_"
        f"vrp_{vrp[0]:.2f}_{vrp[1]:.2f}_{vrp[2]:.2f}_"
        f"z3m_{z3m[0]:.2f}_{z3m[1]:.2f}_{z3m[2]:.2f}_"
        f"z1y_{z1y[0]:.2f}_{z1y[1]:.2f}_{z1y[2]:.2f}_"
        f"rsi_{int(rsi[0])}_{int(rsi[1])}_{int(rsi[2])}_"
        f"rv21d_{rv21d[0]:.1f}_{rv21d[1]:.1f}_{rv21d[2]:.1f}"
    )

    spec_rows.append({
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_phase3_untied_z",
        "forecast_denominator": "har_total_simple",

        "front_vrp_log_min": vrp[0],
        "middle_vrp_log_min": vrp[1],
        "back_vrp_log_min": vrp[2],

        "front_z_3m_min": z3m[0],
        "middle_z_3m_min": z3m[1],
        "back_z_3m_min": z3m[2],

        "front_z_1y_min": z1y[0],
        "middle_z_1y_min": z1y[1],
        "back_z_1y_min": z1y[2],

        "front_rsi14_max": rsi[0],
        "middle_rsi14_max": rsi[1],
        "back_rsi14_max": rsi[2],

        "front_rv21d_min": rv21d[0],
        "middle_rv21d_min": rv21d[1],
        "back_rv21d_min": rv21d[2],

        "phase_z_3m_equals_z_1y": False,
    })

core_specs = pd.DataFrame(spec_rows)

grid_summary = pd.DataFrame([{
    "valid_vrp_triples": len(CORE_VRP_TRIPLES),
    "valid_z_3m_triples": len(CORE_Z3M_TRIPLES),
    "valid_z_1y_triples": len(CORE_Z1Y_TRIPLES),
    "valid_rsi_triples": len(CORE_RSI_TRIPLES),
    "valid_rv21d_triples": len(CORE_RV21D_TRIPLES),
    "total_core_specs": len(core_specs),
}])

print()
print("Core-only Phase 3 grid summary:")
display(grid_summary)

print()
print("Core-only Phase 3 specs sample:")
display(core_specs.head(20))


# -----------------------------
# 4. Build complete eligible matrix panel
# -----------------------------

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "front",
    12: "front",
    15: "front",
    18: "middle",
    21: "middle",
    24: "middle",
    27: "back",
    30: "back",
    33: "back",
}

BUCKETS = ["front", "middle", "back"]
MIN_CONDITIONAL_OBS = 20

eligible = panel.loc[panel["is_parameter_sweep_eligible"].astype(bool)].copy()
eligible = eligible.replace([np.inf, -np.inf], np.nan)

needed_for_matrix = [
    "trade_date",
    "date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "actual_dte_for_pnl_day",
    "pnl_per_day",
]

eligible = eligible.dropna(subset=needed_for_matrix).copy()

eligible = eligible.loc[
    eligible["tenor"].isin(TENORS)
    & eligible["actual_dte_for_pnl_day"].gt(0)
].copy()

duplicate_rows = eligible.duplicated(["trade_date", "tenor"]).sum()

if duplicate_rows:
    raise ValueError(f"Eligible panel has duplicate trade_date x tenor rows: {duplicate_rows}")

date_counts = eligible.groupby("trade_date")["tenor"].nunique()
complete_dates = date_counts.loc[date_counts.eq(len(TENORS))].index.tolist()

eligible = eligible.loc[eligible["trade_date"].isin(complete_dates)].copy()

eligible_dates = sorted(eligible["trade_date"].unique())
num_dates = len(eligible_dates)
num_tenors = len(TENORS)

date_lookup = (
    eligible[["trade_date", "date"]]
    .drop_duplicates("trade_date")
    .set_index("trade_date")
    .loc[eligible_dates]
)

date_array = date_lookup["date"].to_numpy()


def pivot_matrix(col, dtype=float):
    mat = (
        eligible
        .pivot(index="trade_date", columns="tenor", values=col)
        .reindex(index=eligible_dates, columns=TENORS)
    )

    if mat.isna().any().any():
        missing = int(mat.isna().sum().sum())
        raise ValueError(f"Matrix for {col} has missing cells after complete-date filter: {missing}")

    return mat.to_numpy(dtype=dtype)


vrp_mat = pivot_matrix("vrp_log")
z3m_mat = pivot_matrix("vrp_z_3m")
z1y_mat = pivot_matrix("vrp_z_1y")
rsi_mat = pivot_matrix("rsi14")
rv21d_mat = pivot_matrix("rv21d")
pnl_mat = pivot_matrix("outcome_value")
actual_dte_mat = pivot_matrix("actual_dte_for_pnl_day")
pnl_per_day_mat = pivot_matrix("pnl_per_day")

valid_outcome_mat = (
    np.isfinite(pnl_mat)
    & np.isfinite(pnl_per_day_mat)
    & np.isfinite(actual_dte_mat)
)

tenor_array = np.array(TENORS, dtype=int)
tenor_bucket_array = np.array([TENOR_BUCKET_MAP[int(t)] for t in tenor_array], dtype=object)

print()
print("Cell 7 input summary:")
display(pd.DataFrame([{
    "eligible_rows_after_complete_date_filter": len(eligible),
    "eligible_dates": num_dates,
    "first_date": pd.to_datetime(date_array.min()),
    "last_date": pd.to_datetime(date_array.max()),
    "unique_tenors": num_tenors,
    "core_specs_to_test": len(core_specs),
    "min_conditional_obs": MIN_CONDITIONAL_OBS,
    "forecast_model": ", ".join(sorted(eligible["forecast_model"].astype(str).unique())),
    "selection_rule": "strict_highest_conditional_win_probability_no_25bp_band",
}]))


# -----------------------------
# 5. Matrix helper functions
# -----------------------------

def threshold_array_for_spec(spec_row, field_suffix):
    vals = []

    for tenor in TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]
        vals.append(float(spec_row[f"{bucket}_{field_suffix}"]))

    return np.array(vals, dtype=float)


def build_pass_matrix(spec_row):
    vrp_threshold = threshold_array_for_spec(spec_row, "vrp_log_min")
    z3m_threshold = threshold_array_for_spec(spec_row, "z_3m_min")
    z1y_threshold = threshold_array_for_spec(spec_row, "z_1y_min")
    rsi_cap = threshold_array_for_spec(spec_row, "rsi14_max")
    rv21d_floor = threshold_array_for_spec(spec_row, "rv21d_min")

    return (
        (vrp_mat >= vrp_threshold[None, :])
        & (z3m_mat >= z3m_threshold[None, :])
        & (z1y_mat >= z1y_threshold[None, :])
        & (rsi_mat <= rsi_cap[None, :])
        & (rv21d_mat >= rv21d_floor[None, :])
        & valid_outcome_mat
    )


def compute_conditional_arrays(pass_matrix):
    obs = pass_matrix.sum(axis=0).astype(float)
    wins = ((pnl_mat > 0) & pass_matrix).sum(axis=0).astype(float)

    total_pnl = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_mat, 0.0).sum(axis=0),
        0.0,
    )

    total_pnl_day = np.where(
        obs > 0,
        np.where(pass_matrix, pnl_per_day_mat, 0.0).sum(axis=0),
        np.nan,
    )

    total_dte = np.where(
        obs > 0,
        np.where(pass_matrix, actual_dte_mat, 0.0).sum(axis=0),
        np.nan,
    )

    win_prob = np.divide(
        wins,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_day = np.divide(
        total_pnl_day,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    avg_pnl_per_trade = np.divide(
        total_pnl,
        obs,
        out=np.full_like(obs, np.nan, dtype=float),
        where=obs > 0,
    )

    aggregate_pnl_per_day = np.divide(
        total_pnl,
        total_dte,
        out=np.full_like(obs, np.nan, dtype=float),
        where=total_dte > 0,
    )

    worst_trade = np.full(num_tenors, np.nan)
    worst_pnl_day = np.full(num_tenors, np.nan)
    avg_actual_dte = np.full(num_tenors, np.nan)

    for col_idx in range(num_tenors):
        mask = pass_matrix[:, col_idx]

        if mask.any():
            worst_trade[col_idx] = np.nanmin(pnl_mat[mask, col_idx])
            worst_pnl_day[col_idx] = np.nanmin(pnl_per_day_mat[mask, col_idx])
            avg_actual_dte[col_idx] = np.nanmean(actual_dte_mat[mask, col_idx])

    sample_ok = obs >= MIN_CONDITIONAL_OBS

    return {
        "obs": obs,
        "wins": wins,
        "win_prob": win_prob,
        "avg_pnl_per_day": avg_pnl_per_day,
        "aggregate_pnl_per_day": aggregate_pnl_per_day,
        "avg_pnl_per_trade": avg_pnl_per_trade,
        "total_pnl": total_pnl,
        "worst_trade": worst_trade,
        "worst_pnl_day": worst_pnl_day,
        "avg_actual_dte": avg_actual_dte,
        "sample_ok": sample_ok,
    }


def select_cols_from_pass_and_stats(pass_matrix, stats):
    """
    Strict selection rule:
      1. Prefer sample_ok tenors if any passing sample_ok tenor exists on date.
      2. Highest conditional win probability.
      3. Highest conditional average P&L/day.
      4. Highest conditional aggregate P&L/day.
      5. Longer tenor.
    """
    obs = stats["obs"]
    win = stats["win_prob"]
    avg_day = stats["avg_pnl_per_day"]
    agg_day = stats["aggregate_pnl_per_day"]
    sample_ok = stats["sample_ok"]

    has_stats = (
        (obs > 0)
        & np.isfinite(win)
        & np.isfinite(avg_day)
        & np.isfinite(agg_day)
    )

    active = pass_matrix & has_stats[None, :]

    active_sample_ok = active & sample_ok[None, :]
    row_has_sample_ok = active_sample_ok.any(axis=1)

    ranking_universe = np.where(
        row_has_sample_ok[:, None],
        active_sample_ok,
        active,
    )

    selected_col = np.full(num_dates, -1, dtype=int)

    best_win = np.full(num_dates, -np.inf)
    best_avg = np.full(num_dates, -np.inf)
    best_agg = np.full(num_dates, -np.inf)
    best_tenor = np.full(num_dates, -np.inf)

    for col_idx, tenor in enumerate(TENORS):
        cand = ranking_universe[:, col_idx]

        win_val = win[col_idx]
        avg_val = avg_day[col_idx]
        agg_val = agg_day[col_idx]
        tenor_val = float(tenor)

        better = (
            cand
            & (
                (win_val > best_win)
                | ((win_val == best_win) & (avg_val > best_avg))
                | ((win_val == best_win) & (avg_val == best_avg) & (agg_val > best_agg))
                | ((win_val == best_win) & (avg_val == best_avg) & (agg_val == best_agg) & (tenor_val > best_tenor))
            )
        )

        selected_col[better] = col_idx
        best_win[better] = win_val
        best_avg[better] = avg_val
        best_agg[better] = agg_val
        best_tenor[better] = tenor_val

    return selected_col


def profit_factor(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    gains = values[values > 0].sum()
    losses = values[values < 0].sum()

    if losses < 0:
        return gains / abs(losses)

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def summarize_selected_cols(selected_cols, prefix):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    if date_idx.size == 0:
        return {
            f"{prefix}_trades": 0,
            f"{prefix}_frequency": 0.0,
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_day_weighted_pnl_per_day": np.nan,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade_pnl": np.nan,
            f"{prefix}_worst_pnl_per_day": np.nan,
            f"{prefix}_best_trade_pnl": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_avg_selected_tenor": np.nan,
        }

    cols = selected_cols[date_idx]

    pnl = pnl_mat[date_idx, cols]
    pnl_day = pnl_per_day_mat[date_idx, cols]
    dte = actual_dte_mat[date_idx, cols]
    selected_tenors = tenor_array[cols]

    total_dte = np.nansum(dte)

    exposure_day_weighted_pnl_per_day = (
        np.nansum(pnl) / total_dte if total_dte > 0 else np.nan
    )

    return {
        f"{prefix}_trades": int(date_idx.size),
        f"{prefix}_frequency": float(date_idx.size / num_dates) if num_dates else np.nan,
        f"{prefix}_win_rate": float(np.nanmean(pnl > 0)),
        f"{prefix}_total_pnl": float(np.nansum(pnl)),
        f"{prefix}_avg_pnl_per_trade": float(np.nanmean(pnl)),
        f"{prefix}_avg_pnl_per_day": float(np.nanmean(pnl_day)),
        f"{prefix}_exposure_day_weighted_pnl_per_day": float(exposure_day_weighted_pnl_per_day),
        f"{prefix}_profit_factor": float(profit_factor(pnl)),
        f"{prefix}_worst_trade_pnl": float(np.nanmin(pnl)),
        f"{prefix}_worst_pnl_per_day": float(np.nanmin(pnl_day)),
        f"{prefix}_best_trade_pnl": float(np.nanmax(pnl)),
        f"{prefix}_avg_actual_dte": float(np.nanmean(dte)),
        f"{prefix}_avg_selected_tenor": float(np.nanmean(selected_tenors)),
    }


def bucket_metrics(selected_cols):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    out = {}

    for bucket in BUCKETS:
        out[f"{bucket}_selected_trades"] = 0
        out[f"{bucket}_selected_win_rate"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_trade"] = np.nan
        out[f"{bucket}_selected_avg_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = np.nan
        out[f"{bucket}_selected_total_pnl"] = 0.0
        out[f"{bucket}_selected_worst_trade_pnl"] = np.nan

    if date_idx.size == 0:
        return out

    cols = selected_cols[date_idx]
    selected_buckets = tenor_bucket_array[cols]

    for bucket in BUCKETS:
        bucket_mask = selected_buckets == bucket

        if not bucket_mask.any():
            continue

        bucket_date_idx = date_idx[bucket_mask]
        bucket_cols = cols[bucket_mask]

        pnl = pnl_mat[bucket_date_idx, bucket_cols]
        pnl_day = pnl_per_day_mat[bucket_date_idx, bucket_cols]
        dte = actual_dte_mat[bucket_date_idx, bucket_cols]

        total_dte = np.nansum(dte)

        out[f"{bucket}_selected_trades"] = int(bucket_mask.sum())
        out[f"{bucket}_selected_win_rate"] = float(np.nanmean(pnl > 0))
        out[f"{bucket}_selected_avg_pnl_per_trade"] = float(np.nanmean(pnl))
        out[f"{bucket}_selected_avg_pnl_per_day"] = float(np.nanmean(pnl_day))
        out[f"{bucket}_selected_exposure_day_weighted_pnl_per_day"] = (
            float(np.nansum(pnl) / total_dte) if total_dte > 0 else np.nan
        )
        out[f"{bucket}_selected_total_pnl"] = float(np.nansum(pnl))
        out[f"{bucket}_selected_worst_trade_pnl"] = float(np.nanmin(pnl))

    return out


def tenor_selection_metrics(selected_cols, spec_id):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx_all = np.where(selected_cols >= 0)[0]

    rows = []

    for col_idx, tenor in enumerate(TENORS):
        date_idx = np.where(selected_cols == col_idx)[0]

        if date_idx.size == 0:
            rows.append({
                "spec_id": spec_id,
                "selected_tenor": int(tenor),
                "selected_tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
                "selected_trades": 0,
                "selected_frequency_total_dates": 0.0,
                "selected_share_of_trades": 0.0,
                "selected_win_rate": np.nan,
                "selected_avg_pnl_per_trade": np.nan,
                "selected_avg_pnl_per_day": np.nan,
                "selected_exposure_day_weighted_pnl_per_day": np.nan,
                "selected_total_pnl": 0.0,
                "selected_worst_trade_pnl": np.nan,
                "selected_worst_pnl_per_day": np.nan,
                "selected_avg_actual_dte": np.nan,
            })
            continue

        pnl = pnl_mat[date_idx, col_idx]
        pnl_day = pnl_per_day_mat[date_idx, col_idx]
        dte = actual_dte_mat[date_idx, col_idx]

        total_dte = np.nansum(dte)

        rows.append({
            "spec_id": spec_id,
            "selected_tenor": int(tenor),
            "selected_tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "selected_trades": int(date_idx.size),
            "selected_frequency_total_dates": float(date_idx.size / num_dates) if num_dates else np.nan,
            "selected_share_of_trades": float(date_idx.size / date_idx_all.size) if date_idx_all.size else 0.0,
            "selected_win_rate": float(np.nanmean(pnl > 0)),
            "selected_avg_pnl_per_trade": float(np.nanmean(pnl)),
            "selected_avg_pnl_per_day": float(np.nanmean(pnl_day)),
            "selected_exposure_day_weighted_pnl_per_day": (
                float(np.nansum(pnl) / total_dte) if total_dte > 0 else np.nan
            ),
            "selected_total_pnl": float(np.nansum(pnl)),
            "selected_worst_trade_pnl": float(np.nanmin(pnl)),
            "selected_worst_pnl_per_day": float(np.nanmin(pnl_day)),
            "selected_avg_actual_dte": float(np.nanmean(dte)),
        })

    return rows


def append_conditional_rows(rows, spec_id, stats):
    for col_idx, tenor in enumerate(TENORS):
        rows.append({
            "spec_id": spec_id,
            "role": "Core",
            "tenor": int(tenor),
            "tenor_bucket": TENOR_BUCKET_MAP[int(tenor)],
            "conditional_obs": int(stats["obs"][col_idx]),
            "conditional_wins": int(stats["wins"][col_idx]),
            "conditional_win_probability": stats["win_prob"][col_idx],
            "conditional_avg_pnl_per_day": stats["avg_pnl_per_day"][col_idx],
            "conditional_aggregate_pnl_per_day": stats["aggregate_pnl_per_day"][col_idx],
            "conditional_avg_pnl_per_trade": stats["avg_pnl_per_trade"][col_idx],
            "conditional_total_pnl": stats["total_pnl"][col_idx],
            "conditional_worst_trade_pnl": stats["worst_trade"][col_idx],
            "conditional_worst_pnl_per_day": stats["worst_pnl_day"][col_idx],
            "conditional_avg_actual_dte": stats["avg_actual_dte"][col_idx],
            "conditional_sample_ok": bool(stats["sample_ok"][col_idx]),
        })


def selected_trade_detail_rows(selected_cols, spec_row):
    selected_cols = np.asarray(selected_cols, dtype=int)
    date_idx = np.where(selected_cols >= 0)[0]

    rows = []

    for idx in date_idx:
        col_idx = selected_cols[idx]
        tenor = int(tenor_array[col_idx])

        rows.append({
            "spec_id": spec_row["spec_id"],
            "date": pd.to_datetime(date_array[idx]),
            "trade_date": int(eligible_dates[idx]),
            "selected_tenor": tenor,
            "selected_tenor_bucket": TENOR_BUCKET_MAP[tenor],

            "outcome_value": float(pnl_mat[idx, col_idx]),
            "pnl_per_day": float(pnl_per_day_mat[idx, col_idx]),
            "actual_dte": float(actual_dte_mat[idx, col_idx]),

            "vrp_log": float(vrp_mat[idx, col_idx]),
            "vrp_z_3m": float(z3m_mat[idx, col_idx]),
            "vrp_z_1y": float(z1y_mat[idx, col_idx]),
            "rsi14": float(rsi_mat[idx, col_idx]),
            "rv21d": float(rv21d_mat[idx, col_idx]),

            "front_vrp_log_min": spec_row["front_vrp_log_min"],
            "middle_vrp_log_min": spec_row["middle_vrp_log_min"],
            "back_vrp_log_min": spec_row["back_vrp_log_min"],

            "front_z_3m_min": spec_row["front_z_3m_min"],
            "middle_z_3m_min": spec_row["middle_z_3m_min"],
            "back_z_3m_min": spec_row["back_z_3m_min"],

            "front_z_1y_min": spec_row["front_z_1y_min"],
            "middle_z_1y_min": spec_row["middle_z_1y_min"],
            "back_z_1y_min": spec_row["back_z_1y_min"],

            "front_rsi14_max": spec_row["front_rsi14_max"],
            "middle_rsi14_max": spec_row["middle_rsi14_max"],
            "back_rsi14_max": spec_row["back_rsi14_max"],

            "front_rv21d_min": spec_row["front_rv21d_min"],
            "middle_rv21d_min": spec_row["middle_rv21d_min"],
            "back_rv21d_min": spec_row["back_rv21d_min"],
        })

    return rows


# -----------------------------
# 6. Run Core-only Phase 3 sweep
# -----------------------------

summary_rows = []
conditional_rows = []
tenor_selection_rows = []

# Store selected cols for top-detail reconstruction later without recomputing all if possible.
selected_cols_by_spec = {}

total_specs = len(core_specs)

print()
print(f"Running Core-only Phase 3 sweep: {total_specs:,} specs")

for i, (_, spec_row) in enumerate(core_specs.iterrows(), start=1):
    spec_id = spec_row["spec_id"]

    pass_matrix = build_pass_matrix(spec_row)
    candidate_dates = pass_matrix.any(axis=1)
    candidate_rows = int(pass_matrix.sum())

    stats = compute_conditional_arrays(pass_matrix)
    selected_cols = select_cols_from_pass_and_stats(pass_matrix, stats)

    selected_cols_by_spec[spec_id] = selected_cols

    row = {
        "spec_id": spec_id,
        "role": "Core",
        "phase": "core_only_phase3_untied_z",
        "candidate_rows": candidate_rows,
        "candidate_dates": int(candidate_dates.sum()),
        "candidate_frequency": float(candidate_dates.sum() / num_dates) if num_dates else np.nan,

        "front_vrp_log_min": spec_row["front_vrp_log_min"],
        "middle_vrp_log_min": spec_row["middle_vrp_log_min"],
        "back_vrp_log_min": spec_row["back_vrp_log_min"],

        "front_z_3m_min": spec_row["front_z_3m_min"],
        "middle_z_3m_min": spec_row["middle_z_3m_min"],
        "back_z_3m_min": spec_row["back_z_3m_min"],

        "front_z_1y_min": spec_row["front_z_1y_min"],
        "middle_z_1y_min": spec_row["middle_z_1y_min"],
        "back_z_1y_min": spec_row["back_z_1y_min"],

        "front_rsi14_max": spec_row["front_rsi14_max"],
        "middle_rsi14_max": spec_row["middle_rsi14_max"],
        "back_rsi14_max": spec_row["back_rsi14_max"],

        "front_rv21d_min": spec_row["front_rv21d_min"],
        "middle_rv21d_min": spec_row["middle_rv21d_min"],
        "back_rv21d_min": spec_row["back_rv21d_min"],
    }

    row.update(summarize_selected_cols(selected_cols, "core"))
    row.update(bucket_metrics(selected_cols))

    summary_rows.append(row)
    append_conditional_rows(conditional_rows, spec_id, stats)
    tenor_selection_rows.extend(tenor_selection_metrics(selected_cols, spec_id))

    if i % 250 == 0 or i == total_specs:
        print(f"Core specs processed: {i:,} / {total_specs:,}")

core_summary = pd.DataFrame(summary_rows)
conditional_stats = pd.DataFrame(conditional_rows)
tenor_selection_summary = pd.DataFrame(tenor_selection_rows)


# -----------------------------
# 7. Add labels / flags / rankings
# -----------------------------

core_summary["vrp_triple"] = (
    core_summary["front_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_vrp_log_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_vrp_log_min"].map("{:.2f}".format)
)

core_summary["z_3m_triple"] = (
    core_summary["front_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_z_3m_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_z_3m_min"].map("{:.2f}".format)
)

core_summary["z_1y_triple"] = (
    core_summary["front_z_1y_min"].map("{:.2f}".format)
    + " / "
    + core_summary["middle_z_1y_min"].map("{:.2f}".format)
    + " / "
    + core_summary["back_z_1y_min"].map("{:.2f}".format)
)

core_summary["rsi_triple"] = (
    core_summary["front_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["middle_rsi14_max"].astype(int).astype(str)
    + " / "
    + core_summary["back_rsi14_max"].astype(int).astype(str)
)

core_summary["rv21d_triple"] = (
    core_summary["front_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["middle_rv21d_min"].map("{:.1f}".format)
    + " / "
    + core_summary["back_rv21d_min"].map("{:.1f}".format)
)

core_summary["meets_core_win_85"] = core_summary["core_win_rate"].ge(0.85)
core_summary["meets_core_win_83"] = core_summary["core_win_rate"].ge(0.83)
core_summary["meets_core_win_82"] = core_summary["core_win_rate"].ge(0.82)
core_summary["meets_core_win_80"] = core_summary["core_win_rate"].ge(0.80)

core_summary["core_win_gap_to_85"] = 0.85 - core_summary["core_win_rate"]

core_summary_by_win = core_summary.sort_values(
    [
        "core_win_rate",
        "core_frequency",
        "core_avg_pnl_per_day",
        "core_total_pnl",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

core_summary_by_win["win_rate_rank"] = np.arange(1, len(core_summary_by_win) + 1)

core_summary_by_frequency_targets_first = core_summary.sort_values(
    [
        "meets_core_win_85",
        "core_frequency",
        "core_win_rate",
        "core_avg_pnl_per_day",
        "core_total_pnl",
        "core_worst_trade_pnl",
    ],
    ascending=[False, False, False, False, False, False],
).reset_index(drop=True)

core_summary_by_frequency_targets_first["frequency_rank_with_targets_first"] = (
    np.arange(1, len(core_summary_by_frequency_targets_first) + 1)
)

core_review = core_summary.merge(
    core_summary_by_win[["spec_id", "win_rate_rank"]],
    on="spec_id",
    how="left",
    validate="one_to_one",
)

core_review = core_review.merge(
    core_summary_by_frequency_targets_first[["spec_id", "frequency_rank_with_targets_first"]],
    on="spec_id",
    how="left",
    validate="one_to_one",
)


# -----------------------------
# 8. Parameter group diagnostics
# -----------------------------

def parameter_group_summary(df, group_col):
    out = (
        df
        .groupby(group_col)
        .agg(
            specs=("spec_id", "nunique"),
            max_core_win_rate=("core_win_rate", "max"),
            median_core_win_rate=("core_win_rate", "median"),
            max_core_frequency=("core_frequency", "max"),
            median_core_frequency=("core_frequency", "median"),
            max_core_avg_pnl_per_day=("core_avg_pnl_per_day", "max"),
            median_core_avg_pnl_per_day=("core_avg_pnl_per_day", "median"),
            max_core_total_pnl=("core_total_pnl", "max"),
            specs_core_win_85_plus=("meets_core_win_85", "sum"),
            specs_core_win_83_plus=("meets_core_win_83", "sum"),
            specs_core_win_82_plus=("meets_core_win_82", "sum"),
            specs_core_win_80_plus=("meets_core_win_80", "sum"),
        )
        .reset_index()
        .sort_values(
            [
                "max_core_win_rate",
                "max_core_frequency",
                "max_core_avg_pnl_per_day",
            ],
            ascending=[False, False, False],
        )
    )

    return out


core_by_vrp = parameter_group_summary(core_review, "vrp_triple")
core_by_z3m = parameter_group_summary(core_review, "z_3m_triple")
core_by_z1y = parameter_group_summary(core_review, "z_1y_triple")
core_by_rsi = parameter_group_summary(core_review, "rsi_triple")
core_by_rv21d = parameter_group_summary(core_review, "rv21d_triple")


# -----------------------------
# 9. Tenor diagnostics and selected trade detail
# -----------------------------

def tenor_table_for_specs(spec_ids):
    out = tenor_selection_summary.loc[
        tenor_selection_summary["spec_id"].isin(spec_ids)
    ].merge(
        core_review[
            [
                "spec_id",
                "win_rate_rank",
                "frequency_rank_with_targets_first",
                "core_trades",
                "core_frequency",
                "core_win_rate",
                "core_avg_pnl_per_day",
                "core_total_pnl",
                "vrp_triple",
                "z_3m_triple",
                "z_1y_triple",
                "rsi_triple",
                "rv21d_triple",
            ]
        ],
        on="spec_id",
        how="left",
        validate="many_to_one",
    )

    return out.sort_values(
        ["win_rate_rank", "selected_tenor"],
        ascending=[True, True],
    )


top_win_spec_ids = (
    core_review
    .sort_values(
        ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

target_frequency_spec_ids = (
    core_review
    .loc[core_review["meets_core_win_85"]]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

above_83_frequency_spec_ids = (
    core_review
    .loc[core_review["core_win_rate"].ge(0.83)]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

above_82_frequency_spec_ids = (
    core_review
    .loc[core_review["core_win_rate"].ge(0.82)]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(20)["spec_id"]
    .tolist()
)

tenor_top_win = tenor_table_for_specs(top_win_spec_ids)
tenor_target_frequency = tenor_table_for_specs(target_frequency_spec_ids)
tenor_above_83_frequency = tenor_table_for_specs(above_83_frequency_spec_ids)
tenor_above_82_frequency = tenor_table_for_specs(above_82_frequency_spec_ids)

detail_spec_ids = list(dict.fromkeys(
    top_win_spec_ids
    + target_frequency_spec_ids
    + above_83_frequency_spec_ids
    + above_82_frequency_spec_ids
))

selected_trade_detail = []

spec_lookup = core_specs.set_index("spec_id")

for spec_id in detail_spec_ids:
    selected_cols = selected_cols_by_spec[spec_id]
    spec_row = spec_lookup.loc[spec_id].copy()
    spec_row["spec_id"] = spec_id
    selected_trade_detail.extend(selected_trade_detail_rows(selected_cols, spec_row))

selected_trade_detail = pd.DataFrame(selected_trade_detail)


# -----------------------------
# 10. Display outputs
# -----------------------------

print()
print("Core-only Phase 3 sweep summary:")
display(pd.DataFrame([{
    "core_specs_tested": len(core_review),
    "specs_core_win_85_plus": int(core_review["meets_core_win_85"].sum()),
    "specs_core_win_83_plus": int(core_review["meets_core_win_83"].sum()),
    "specs_core_win_82_plus": int(core_review["meets_core_win_82"].sum()),
    "specs_core_win_80_plus": int(core_review["meets_core_win_80"].sum()),
    "max_core_win_rate": core_review["core_win_rate"].max(),
    "median_core_win_rate": core_review["core_win_rate"].median(),
    "max_core_frequency": core_review["core_frequency"].max(),
    "median_core_frequency": core_review["core_frequency"].median(),
    "max_core_avg_pnl_per_day": core_review["core_avg_pnl_per_day"].max(),
    "median_core_avg_pnl_per_day": core_review["core_avg_pnl_per_day"].median(),
}]))

review_cols = [
    "frequency_rank_with_targets_first",
    "win_rate_rank",
    "meets_core_win_85",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_win_gap_to_85",
    "core_avg_pnl_per_trade",
    "core_avg_pnl_per_day",
    "core_exposure_day_weighted_pnl_per_day",
    "core_total_pnl",
    "core_profit_factor",
    "core_worst_trade_pnl",
    "core_worst_pnl_per_day",
    "core_avg_actual_dte",
    "core_avg_selected_tenor",

    "front_selected_trades",
    "front_selected_win_rate",
    "front_selected_avg_pnl_per_trade",
    "front_selected_avg_pnl_per_day",
    "front_selected_total_pnl",
    "front_selected_worst_trade_pnl",

    "middle_selected_trades",
    "middle_selected_win_rate",
    "middle_selected_avg_pnl_per_trade",
    "middle_selected_avg_pnl_per_day",
    "middle_selected_total_pnl",
    "middle_selected_worst_trade_pnl",

    "back_selected_trades",
    "back_selected_win_rate",
    "back_selected_avg_pnl_per_trade",
    "back_selected_avg_pnl_per_day",
    "back_selected_total_pnl",
    "back_selected_worst_trade_pnl",

    "vrp_triple",
    "z_3m_triple",
    "z_1y_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

tenor_cols = [
    "win_rate_rank",
    "frequency_rank_with_targets_first",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_avg_pnl_per_day",
    "core_total_pnl",

    "selected_tenor",
    "selected_tenor_bucket",
    "selected_trades",
    "selected_frequency_total_dates",
    "selected_share_of_trades",
    "selected_win_rate",
    "selected_avg_pnl_per_trade",
    "selected_avg_pnl_per_day",
    "selected_exposure_day_weighted_pnl_per_day",
    "selected_total_pnl",
    "selected_worst_trade_pnl",
    "selected_worst_pnl_per_day",
    "selected_avg_actual_dte",

    "vrp_triple",
    "z_3m_triple",
    "z_1y_triple",
    "rsi_triple",
    "rv21d_triple",
    "spec_id",
]

print()
print("Top Core specs by win rate:")
display(
    core_review
    .sort_values(
        ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    [review_cols]
    .head(100)
)

print()
print("Top Core specs with win rate >= 85%, ranked by frequency:")
display(
    core_review
    .loc[core_review["meets_core_win_85"], review_cols]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Highest-frequency Core specs with win rate >= 83%:")
display(
    core_review
    .loc[core_review["core_win_rate"].ge(0.83), review_cols]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Highest-frequency Core specs with win rate >= 82%:")
display(
    core_review
    .loc[core_review["core_win_rate"].ge(0.82), review_cols]
    .sort_values(
        ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
        ascending=[False, False, False, False],
    )
    .head(100)
)

print()
print("Tenor / DTE selection diagnostics for top Core specs by win rate:")
display(tenor_top_win[tenor_cols].head(180))

print()
print("Tenor / DTE selection diagnostics for Core specs with win rate >= 85%, ranked by frequency:")
display(tenor_target_frequency[tenor_cols].head(180))

print()
print("Tenor / DTE selection diagnostics for highest-frequency Core specs with win rate >= 83%:")
display(tenor_above_83_frequency[tenor_cols].head(180))

print()
print("Tenor / DTE selection diagnostics for highest-frequency Core specs with win rate >= 82%:")
display(tenor_above_82_frequency[tenor_cols].head(180))

print()
print("Core parameter diagnostics by VRP triple:")
display(core_by_vrp)

print()
print("Core parameter diagnostics by 3m z-score triple:")
display(core_by_z3m)

print()
print("Core parameter diagnostics by 1y z-score triple:")
display(core_by_z1y)

print()
print("Core parameter diagnostics by RSI cap triple:")
display(core_by_rsi)

print()
print("Core parameter diagnostics by RV21D floor triple:")
display(core_by_rv21d)

print()
print("Selected trade detail sample for top specs:")
display(selected_trade_detail.head(100))


# -----------------------------
# 11. Save outputs
# -----------------------------

safe_start = pd.to_datetime(date_array.min()).strftime("%Y%m%d")
safe_end = pd.to_datetime(date_array.max()).strftime("%Y%m%d")

core_specs_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_specs_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

grid_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_grid_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_summary_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

conditional_stats_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_conditional_stats_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_selection_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_tenor_selection_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

selected_trade_detail_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_selected_trade_detail_top_specs_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

top_by_win_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_top_by_win_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_win85_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

above_83_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_win83_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

above_82_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_win82_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_top_win_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_tenor_top_by_win_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_target_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_tenor_win85_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_above_83_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_tenor_win83_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

tenor_above_82_frequency_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_tenor_win82_by_frequency_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_vrp_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_by_vrp_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_z3m_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_by_z3m_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_z1y_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_by_z1y_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rsi_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_by_rsi_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_by_rv21d_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_by_rv21d_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

core_specs.to_csv(core_specs_path, index=False)
grid_summary.to_csv(grid_summary_path, index=False)
core_review.to_csv(core_summary_path, index=False)
conditional_stats.to_csv(conditional_stats_path, index=False)
tenor_selection_summary.to_csv(tenor_selection_path, index=False)
selected_trade_detail.to_csv(selected_trade_detail_path, index=False)

core_review.sort_values(
    ["core_win_rate", "core_frequency", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).head(1000).to_csv(top_by_win_path, index=False)

core_review.loc[
    core_review["meets_core_win_85"]
].sort_values(
    ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(target_by_frequency_path, index=False)

core_review.loc[
    core_review["core_win_rate"].ge(0.83)
].sort_values(
    ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(above_83_frequency_path, index=False)

core_review.loc[
    core_review["core_win_rate"].ge(0.82)
].sort_values(
    ["core_frequency", "core_win_rate", "core_avg_pnl_per_day", "core_total_pnl"],
    ascending=[False, False, False, False],
).to_csv(above_82_frequency_path, index=False)

tenor_top_win.to_csv(tenor_top_win_path, index=False)
tenor_target_frequency.to_csv(tenor_target_frequency_path, index=False)
tenor_above_83_frequency.to_csv(tenor_above_83_frequency_path, index=False)
tenor_above_82_frequency.to_csv(tenor_above_82_frequency_path, index=False)

core_by_vrp.to_csv(core_by_vrp_path, index=False)
core_by_z3m.to_csv(core_by_z3m_path, index=False)
core_by_z1y.to_csv(core_by_z1y_path, index=False)
core_by_rsi.to_csv(core_by_rsi_path, index=False)
core_by_rv21d.to_csv(core_by_rv21d_path, index=False)

print()
print("Cell 7 outputs saved:")
print(f"Core specs:                    {core_specs_path}")
print(f"Grid summary:                  {grid_summary_path}")
print(f"Core summary:                  {core_summary_path}")
print(f"Conditional stats:             {conditional_stats_path}")
print(f"Tenor selection summary:       {tenor_selection_path}")
print(f"Selected trade detail:         {selected_trade_detail_path}")
print(f"Top by win rate:               {top_by_win_path}")
print(f"Win >=85 by frequency:         {target_by_frequency_path}")
print(f"Win >=83 by frequency:         {above_83_frequency_path}")
print(f"Win >=82 by frequency:         {above_82_frequency_path}")
print(f"Tenor top by win:              {tenor_top_win_path}")
print(f"Tenor win >=85 by frequency:   {tenor_target_frequency_path}")
print(f"Tenor win >=83 by frequency:   {tenor_above_83_frequency_path}")
print(f"Tenor win >=82 by frequency:   {tenor_above_82_frequency_path}")
print(f"Core by VRP:                   {core_by_vrp_path}")
print(f"Core by 3m z-score:            {core_by_z3m_path}")
print(f"Core by 1y z-score:            {core_by_z1y_path}")
print(f"Core by RSI:                   {core_by_rsi_path}")
print(f"Core by RV21D:                 {core_by_rv21d_path}")

print()
print("Cell 7 complete.")

Cell 7: Core-only Phase 3 sweep — untied z-scores
Clean panel path: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
Run timestamp:    20260704_101706

Actual DTE source:
C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet

Actual DTE availability:


,rows,actual_dte_non_null,actual_dte_non_null_pct,fallback_to_tenor_rows,fallback_to_tenor_pct
0,14565,14565,1.0,0,0.0



Core-only Phase 3 grid summary:


,valid_vrp_triples,valid_z_3m_triples,valid_z_1y_triples,valid_rsi_triples,valid_rv21d_triples,total_core_specs
0,1,18,18,3,2,1944



Core-only Phase 3 specs sample:


...


Core parameter diagnostics by RV21D floor triple:


,rv21d_triple,specs,max_core_win_rate,median_core_win_rate,max_core_frequency,median_core_frequency,max_core_avg_pnl_per_day,median_core_avg_pnl_per_day,max_core_total_pnl,specs_core_win_85_plus,specs_core_win_83_plus,specs_core_win_82_plus,specs_core_win_80_plus
1,8.0 / 8.5 / 8.5,972,0.828996,0.815436,0.212449,0.200609,378.467721,338.889529,2.587287e+06,0,0,243,918
0,7.5 / 8.5 / 8.5,972,0.821818,0.809211,0.217185,0.204668,360.268843,319.623381,2.546824e+06,0,0,24,663



Selected trade detail sample for top specs:


,spec_id,date,trade_date,selected_tenor,selected_tenor_bucket,outcome_value,pnl_per_day,actual_dte,vrp_log,vrp_z_3m,...,back_z_3m_min,front_z_1y_min,middle_z_1y_min,back_z_1y_min,front_rsi14_max,middle_rsi14_max,back_rsi14_max,front_rv21d_min,middle_rv21d_min,back_rv21d_min
0,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2020-07-27,20200727,33,back,23677.151086,739.910971,32.0,1.506651,0.881128,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
1,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2020-07-28,20200728,30,back,23489.640944,757.730353,31.0,1.520873,0.937214,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
2,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2020-08-19,20200819,24,middle,7547.002089,328.130526,23.0,1.567036,1.555944,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
3,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2020-08-21,20200821,15,front,11715.668382,836.833456,14.0,1.462593,0.993049,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
4,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2020-10-19,20201019,24,middle,27021.348616,1080.853945,25.0,1.399671,0.978145,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2022-10-26,20221026,18,middle,23364.485981,1460.280374,16.0,0.780646,0.837228,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
96,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2022-10-27,20221027,21,middle,24111.575132,1095.980688,22.0,0.745012,0.746331,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
97,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2022-10-28,20221028,30,back,26198.007721,935.643133,28.0,0.768543,0.989065,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5
98,core_phase3_vrp_0.60_0.70_0.70_z3m_0.70_0.70_0...,2022-10-31,20221031,18,middle,21591.020615,1199.501145,18.0,0.810560,0.869521,...,0.8,0.7,0.75,0.75,70,68,65,8.0,8.5,8.5



Cell 7 outputs saved:
Core specs:                    C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_specs_20200701_20260519_20260704_101706.csv
Grid summary:                  C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_grid_summary_20200701_20260519_20260704_101706.csv
Core summary:                  C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_summary_20200701_20260519_20260704_101706.csv
Conditional stats:             C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell7_core_only_phase3_untied_z_conditional_stats_20200701_20260519_20260704_101706.csv
Tenor selection summary:       C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell7_core_o

In [10]:
# Cell 8: Z-score leakage audit
#
# Purpose:
#   Confirm whether stored VRP z-scores were calculated using:
#       A) prior-only rolling history: shift(1), correct for our use
#       B) inclusive rolling history: current value included in rolling mean/std
#
# This is audit-only:
#   - No parameter testing
#   - No model changes
#   - No signal changes
#   - No overwrite of clean panel
#
# Stored columns audited:
#   vrp_z_3m
#   vrp_z_1y
#
# Source VRP column:
#   vrp_log
#
# Expected correct formula:
#   z_t = (vrp_log_t - mean(vrp_log_{t-1...t-window})) / std(vrp_log_{t-1...t-window})

from pathlib import Path
import itertools
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / load latest clean panel
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

PARAM_SWEEP_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"
PARAM_SWEEP_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_clean"

PARAM_SWEEP_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    PARAM_SWEEP_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No clean panel found in {PARAM_SWEEP_PROCESSED_DIR}")

CLEAN_PANEL_PATH = clean_panel_candidates[-1]

print("Cell 8: Z-score leakage audit")
print(f"Clean panel path: {CLEAN_PANEL_PATH}")
print(f"Run timestamp:    {RUN_TIMESTAMP}")

panel = pd.read_parquet(CLEAN_PANEL_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_cols = [
    "date",
    "trade_date",
    "tenor",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
]

missing_cols = [c for c in required_cols if c not in panel.columns]

if missing_cols:
    raise ValueError(f"Panel missing required columns for z-score audit: {missing_cols}")

audit_df = (
    panel[required_cols]
    .copy()
    .replace([np.inf, -np.inf], np.nan)
    .sort_values(["tenor", "date", "trade_date"])
    .reset_index(drop=True)
)

print()
print("Stored z-score availability:")
display(pd.DataFrame([{
    "rows": len(audit_df),
    "vrp_log_non_null": int(audit_df["vrp_log"].notna().sum()),
    "vrp_z_3m_non_null": int(audit_df["vrp_z_3m"].notna().sum()),
    "vrp_z_1y_non_null": int(audit_df["vrp_z_1y"].notna().sum()),
    "vrp_z_3m_null": int(audit_df["vrp_z_3m"].isna().sum()),
    "vrp_z_1y_null": int(audit_df["vrp_z_1y"].isna().sum()),
    "first_date": audit_df["date"].min(),
    "last_date": audit_df["date"].max(),
    "unique_tenors": audit_df["tenor"].nunique(),
}]))


# -----------------------------
# 2. Recompute candidate z-scores
# -----------------------------

def compute_candidate_z(df, value_col, window, min_periods, convention, ddof):
    """
    convention:
        prior_only = rolling history uses group shift(1)
        inclusive  = rolling history includes current observation

    z_t always uses current value in numerator:
        current value - rolling mean
    """
    if convention == "prior_only_shift1":
        history = df.groupby("tenor")[value_col].shift(1)
    elif convention == "inclusive_current":
        history = df[value_col].copy()
    else:
        raise ValueError(f"Unknown convention: {convention}")

    roll_mean = (
        history
        .groupby(df["tenor"])
        .rolling(window=window, min_periods=min_periods)
        .mean()
        .reset_index(level=0, drop=True)
    )

    roll_std = (
        history
        .groupby(df["tenor"])
        .rolling(window=window, min_periods=min_periods)
        .std(ddof=ddof)
        .reset_index(level=0, drop=True)
    )

    z = (df[value_col] - roll_mean) / roll_std
    z = z.replace([np.inf, -np.inf], np.nan)

    return z


def evaluate_candidate(df, stored_col, candidate_z, horizon, convention, window, min_periods, ddof):
    stored = pd.to_numeric(df[stored_col], errors="coerce")
    cand = pd.to_numeric(candidate_z, errors="coerce")

    stored_finite = np.isfinite(stored)
    cand_finite = np.isfinite(cand)
    both = stored_finite & cand_finite

    diffs = stored[both] - cand[both]

    if both.sum() > 0:
        corr = np.corrcoef(stored[both], cand[both])[0, 1]
        mae = float(np.nanmean(np.abs(diffs)))
        rmse = float(np.sqrt(np.nanmean(np.square(diffs))))
        max_abs = float(np.nanmax(np.abs(diffs)))
        median_abs = float(np.nanmedian(np.abs(diffs)))
    else:
        corr = np.nan
        mae = np.nan
        rmse = np.nan
        max_abs = np.nan
        median_abs = np.nan

    null_mismatch = stored_finite.ne(cand_finite)

    return {
        "horizon": horizon,
        "stored_col": stored_col,
        "convention": convention,
        "window": window,
        "min_periods": min_periods,
        "ddof": ddof,

        "stored_non_null": int(stored_finite.sum()),
        "candidate_non_null": int(cand_finite.sum()),
        "overlap_non_null": int(both.sum()),
        "only_stored_non_null": int((stored_finite & ~cand_finite).sum()),
        "only_candidate_non_null": int((~stored_finite & cand_finite).sum()),
        "null_mismatch_count": int(null_mismatch.sum()),
        "null_mismatch_rate": float(null_mismatch.mean()),

        "corr": float(corr) if np.isfinite(corr) else np.nan,
        "mae": mae,
        "median_abs_error": median_abs,
        "rmse": rmse,
        "max_abs_error": max_abs,
    }


def evaluate_candidate_by_tenor(df, stored_col, candidate_z, horizon, convention, window, min_periods, ddof):
    tmp = df[["date", "trade_date", "tenor", stored_col]].copy()
    tmp["candidate_z"] = candidate_z

    rows = []

    for tenor, g in tmp.groupby("tenor", sort=True):
        stored = pd.to_numeric(g[stored_col], errors="coerce")
        cand = pd.to_numeric(g["candidate_z"], errors="coerce")

        stored_finite = np.isfinite(stored)
        cand_finite = np.isfinite(cand)
        both = stored_finite & cand_finite

        diffs = stored[both] - cand[both]

        if both.sum() > 0:
            corr = np.corrcoef(stored[both], cand[both])[0, 1]
            mae = float(np.nanmean(np.abs(diffs)))
            rmse = float(np.sqrt(np.nanmean(np.square(diffs))))
            max_abs = float(np.nanmax(np.abs(diffs)))
            median_abs = float(np.nanmedian(np.abs(diffs)))
        else:
            corr = np.nan
            mae = np.nan
            rmse = np.nan
            max_abs = np.nan
            median_abs = np.nan

        null_mismatch = stored_finite.ne(cand_finite)

        rows.append({
            "horizon": horizon,
            "stored_col": stored_col,
            "tenor": int(tenor),
            "convention": convention,
            "window": window,
            "min_periods": min_periods,
            "ddof": ddof,

            "rows": len(g),
            "stored_non_null": int(stored_finite.sum()),
            "candidate_non_null": int(cand_finite.sum()),
            "overlap_non_null": int(both.sum()),
            "only_stored_non_null": int((stored_finite & ~cand_finite).sum()),
            "only_candidate_non_null": int((~stored_finite & cand_finite).sum()),
            "null_mismatch_count": int(null_mismatch.sum()),
            "null_mismatch_rate": float(null_mismatch.mean()),

            "corr": float(corr) if np.isfinite(corr) else np.nan,
            "mae": mae,
            "median_abs_error": median_abs,
            "rmse": rmse,
            "max_abs_error": max_abs,
        })

    return pd.DataFrame(rows)


# Candidate search grids.
# Include likely production choices and nearby alternatives.
AUDIT_CONFIG = {
    "3m": {
        "stored_col": "vrp_z_3m",
        "windows": [60, 63, 64, 65],
        "min_periods": [20, 21, 30, 42, 60, 63],
    },
    "1y": {
        "stored_col": "vrp_z_1y",
        "windows": [250, 252, 253],
        "min_periods": [63, 90, 120, 126, 180, 252],
    },
}

CONVENTIONS = [
    "prior_only_shift1",
    "inclusive_current",
]

DDOF_OPTIONS = [0, 1]

candidate_rows = []
candidate_z_cache = {}

for horizon, cfg in AUDIT_CONFIG.items():
    stored_col = cfg["stored_col"]

    for convention, window, min_periods, ddof in itertools.product(
        CONVENTIONS,
        cfg["windows"],
        cfg["min_periods"],
        DDOF_OPTIONS,
    ):
        if min_periods > window:
            continue

        z = compute_candidate_z(
            audit_df,
            value_col="vrp_log",
            window=window,
            min_periods=min_periods,
            convention=convention,
            ddof=ddof,
        )

        key = (horizon, convention, window, min_periods, ddof)
        candidate_z_cache[key] = z

        candidate_rows.append(
            evaluate_candidate(
                audit_df,
                stored_col=stored_col,
                candidate_z=z,
                horizon=horizon,
                convention=convention,
                window=window,
                min_periods=min_periods,
                ddof=ddof,
            )
        )

candidate_results = pd.DataFrame(candidate_rows)

candidate_results["score"] = (
    candidate_results["null_mismatch_rate"].fillna(1.0) * 1_000_000
    + candidate_results["rmse"].fillna(1_000_000)
    + candidate_results["mae"].fillna(1_000_000)
)

candidate_results = candidate_results.sort_values(
    ["horizon", "score", "null_mismatch_count", "rmse", "mae", "max_abs_error"],
    ascending=[True, True, True, True, True, True],
).reset_index(drop=True)

best_candidates = (
    candidate_results
    .sort_values(["horizon", "score"], ascending=[True, True])
    .groupby("horizon", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print()
print("Best matching z-score convention by horizon:")
display(best_candidates)

print()
print("Top 20 candidate matches by horizon:")
display(
    candidate_results
    .groupby("horizon", as_index=False)
    .head(20)
    .reset_index(drop=True)
)


# -----------------------------
# 3. Per-tenor diagnostics for best candidates
# -----------------------------

best_by_tenor_tables = []

for _, row in best_candidates.iterrows():
    horizon = row["horizon"]
    stored_col = row["stored_col"]
    convention = row["convention"]
    window = int(row["window"])
    min_periods = int(row["min_periods"])
    ddof = int(row["ddof"])

    key = (horizon, convention, window, min_periods, ddof)
    z = candidate_z_cache[key]

    best_by_tenor_tables.append(
        evaluate_candidate_by_tenor(
            audit_df,
            stored_col=stored_col,
            candidate_z=z,
            horizon=horizon,
            convention=convention,
            window=window,
            min_periods=min_periods,
            ddof=ddof,
        )
    )

best_by_tenor = pd.concat(best_by_tenor_tables, ignore_index=True)

print()
print("Per-tenor diagnostics for best matching convention:")
display(best_by_tenor)


# -----------------------------
# 4. Hard conclusion
# -----------------------------

TOLERANCE_MAX_ABS = 1e-8
TOLERANCE_RMSE = 1e-10

conclusion_rows = []

for _, row in best_candidates.iterrows():
    horizon = row["horizon"]
    convention = row["convention"]
    window = int(row["window"])
    min_periods = int(row["min_periods"])
    ddof = int(row["ddof"])
    null_mismatch_count = int(row["null_mismatch_count"])
    rmse = float(row["rmse"])
    max_abs_error = float(row["max_abs_error"])

    exact_match = (
        null_mismatch_count == 0
        and np.isfinite(rmse)
        and np.isfinite(max_abs_error)
        and rmse <= TOLERANCE_RMSE
        and max_abs_error <= TOLERANCE_MAX_ABS
    )

    if exact_match and convention == "prior_only_shift1":
        status = "PASS_PRIOR_ONLY"
        interpretation = "Stored z-score matches prior-only rolling history. No current-value inclusion detected."
    elif exact_match and convention == "inclusive_current":
        status = "FAIL_INCLUSIVE_CURRENT"
        interpretation = "Stored z-score matches inclusive rolling history. Current value appears included in rolling mean/std."
    else:
        status = "NO_EXACT_MATCH"
        interpretation = "Stored z-score does not exactly match tested conventions. Review top candidates and source code."

    conclusion_rows.append({
        "horizon": horizon,
        "status": status,
        "best_convention": convention,
        "best_window": window,
        "best_min_periods": min_periods,
        "best_ddof": ddof,
        "null_mismatch_count": null_mismatch_count,
        "rmse": rmse,
        "max_abs_error": max_abs_error,
        "interpretation": interpretation,
    })

conclusion = pd.DataFrame(conclusion_rows)

print()
print("Z-score leakage audit conclusion:")
display(conclusion)


# -----------------------------
# 5. Save audit outputs
# -----------------------------

safe_start = audit_df["date"].min().strftime("%Y%m%d")
safe_end = audit_df["date"].max().strftime("%Y%m%d")

candidate_results_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_candidates_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

best_candidates_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_best_candidates_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

best_by_tenor_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_best_by_tenor_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

conclusion_path = (
    PARAM_SWEEP_AUDIT_DIR
    / f"02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_conclusion_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

candidate_results.to_csv(candidate_results_path, index=False)
best_candidates.to_csv(best_candidates_path, index=False)
best_by_tenor.to_csv(best_by_tenor_path, index=False)
conclusion.to_csv(conclusion_path, index=False)

print()
print("Cell 8 outputs saved:")
print(f"Candidate matches:       {candidate_results_path}")
print(f"Best candidates:         {best_candidates_path}")
print(f"Best by tenor:           {best_by_tenor_path}")
print(f"Conclusion:              {conclusion_path}")

print()
print("Cell 8 complete.")

Cell 8: Z-score leakage audit
Clean panel path: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
Run timestamp:    20260704_104426

Stored z-score availability:


,rows,vrp_log_non_null,vrp_z_3m_non_null,vrp_z_1y_non_null,vrp_z_3m_null,vrp_z_1y_null,first_date,last_date,unique_tenors
0,14565,14565,14304,13440,261,1125,2020-01-02,2026-06-23,9



Best matching z-score convention by horizon:


,horizon,stored_col,convention,window,min_periods,ddof,stored_non_null,candidate_non_null,overlap_non_null,only_stored_non_null,only_candidate_non_null,null_mismatch_count,null_mismatch_rate,corr,mae,median_abs_error,rmse,max_abs_error,score
0,1y,vrp_z_1y,inclusive_current,252,126,1,13440,13440,13440,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,3m,vrp_z_3m,inclusive_current,63,30,1,14304,14304,14304,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0



Top 20 candidate matches by horizon:


...


Per-tenor diagnostics for best matching convention:


,horizon,stored_col,tenor,convention,window,min_periods,ddof,rows,stored_non_null,candidate_non_null,overlap_non_null,only_stored_non_null,only_candidate_non_null,null_mismatch_count,null_mismatch_rate,corr,mae,median_abs_error,rmse,max_abs_error
0,1y,vrp_z_1y,9,inclusive_current,252,126,1,1626,1501,1501,1501,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
1,1y,vrp_z_1y,12,inclusive_current,252,126,1,1624,1499,1499,1499,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
2,1y,vrp_z_1y,15,inclusive_current,252,126,1,1623,1498,1498,1498,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
3,1y,vrp_z_1y,18,inclusive_current,252,126,1,1620,1495,1495,1495,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
4,1y,vrp_z_1y,21,inclusive_current,252,126,1,1619,1494,1494,1494,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
5,1y,vrp_z_1y,24,inclusive_current,252,126,1,1616,1491,1491,1491,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
6,1y,vrp_z_1y,27,inclusive_current,252,126,1,1615,1490,1490,1490,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
7,1y,vrp_z_1y,30,inclusive_current,252,126,1,1612,1487,1487,1487,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
8,1y,vrp_z_1y,33,inclusive_current,252,126,1,1610,1485,1485,1485,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0
9,3m,vrp_z_3m,9,inclusive_current,63,30,1,1626,1597,1597,1597,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0



Z-score leakage audit conclusion:


,horizon,status,best_convention,best_window,best_min_periods,best_ddof,null_mismatch_count,rmse,max_abs_error,interpretation
0,1y,FAIL_INCLUSIVE_CURRENT,inclusive_current,252,126,1,0,0.0,0.0,Stored z-score matches inclusive rolling histo...
1,3m,FAIL_INCLUSIVE_CURRENT,inclusive_current,63,30,1,0,0.0,0.0,Stored z-score matches inclusive rolling histo...



Cell 8 outputs saved:
Candidate matches:       C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_candidates_20200102_20260623_20260704_104426.csv
Best candidates:         C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_best_candidates_20200102_20260623_20260704_104426.csv
Best by tenor:           C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_best_by_tenor_20200102_20260623_20260704_104426.csv
Conclusion:              C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_cell8_zscore_leakage_audit_conclusion_20200102_20260623_20260704_104426.csv

Cell 8 complete.


In [11]:
# Cell 9: Rebuild clean parameter sweep panel with prior-only z-scores
#
# Purpose:
#   Start a clean branch after discovering that the stored z-scores were inclusive-current.
#
# This cell:
#   1. Loads the latest existing clean panel.
#   2. Preserves old inclusive-current z-score columns.
#   3. Recomputes prior-only z-scores by tenor:
#        3m: window=63,  min_periods=30,  ddof=1, shift(1)
#        1y: window=252, min_periods=126, ddof=1, shift(1)
#   4. Replaces vrp_z_3m and vrp_z_1y with prior-only versions.
#   5. Recomputes parameter sweep eligibility.
#   6. Saves a new corrected panel in a new branch directory.
#   7. Saves diagnostics and an internal verification audit.
#
# This does NOT:
#   - Change har_total_simple forecast variance
#   - Change VRP log
#   - Change implied variance
#   - Change outcomes
#   - Change RSI / RV21D
#   - Run a parameter sweep
#   - Overwrite the old inclusive-current clean panel

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

OLD_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean"

NEW_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_prior_only_z"
NEW_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "corsi_parameter_sweep_prior_only_z"

NEW_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NEW_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

clean_panel_candidates = sorted(
    OLD_PROCESSED_DIR.glob("02_corsi_parameter_sweep_clean_panel_*.parquet")
)

if not clean_panel_candidates:
    raise FileNotFoundError(f"No existing clean panel found in {OLD_PROCESSED_DIR}")

SOURCE_CLEAN_PANEL_PATH = clean_panel_candidates[-1]

print("Cell 9: Rebuild clean panel with prior-only z-scores")
print(f"Source clean panel: {SOURCE_CLEAN_PANEL_PATH}")
print(f"New processed dir:  {NEW_PROCESSED_DIR}")
print(f"New audit dir:      {NEW_AUDIT_DIR}")
print(f"Run timestamp:      {RUN_TIMESTAMP}")


# -----------------------------
# 2. Load panel
# -----------------------------

panel = pd.read_parquet(SOURCE_CLEAN_PANEL_PATH).copy()

panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

required_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "forecast_model",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
    "has_outcome",
    "is_parameter_sweep_eligible",
]

missing_cols = [c for c in required_cols if c not in panel.columns]

if missing_cols:
    raise ValueError(f"Source panel missing required columns: {missing_cols}")

panel = panel.replace([np.inf, -np.inf], np.nan)

print()
print("Source panel summary:")
display(pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "unique_dates": panel["trade_date"].nunique(),
    "unique_tenors": panel["tenor"].nunique(),
    "source_vrp_z_3m_non_null": int(panel["vrp_z_3m"].notna().sum()),
    "source_vrp_z_1y_non_null": int(panel["vrp_z_1y"].notna().sum()),
    "source_eligible_rows": int(panel["is_parameter_sweep_eligible"].astype(bool).sum()),
}]))


# -----------------------------
# 3. Preserve old inclusive-current z-scores
# -----------------------------

panel["vrp_z_3m_inclusive_current_old"] = panel["vrp_z_3m"]
panel["vrp_z_1y_inclusive_current_old"] = panel["vrp_z_1y"]
panel["is_parameter_sweep_eligible_inclusive_current_old"] = panel["is_parameter_sweep_eligible"]


# -----------------------------
# 4. Recompute prior-only z-scores
# -----------------------------

def compute_prior_only_z_by_tenor(df, value_col, window, min_periods, ddof):
    """
    For each tenor:
        z_t = (x_t - mean(x_{t-1}, ..., x_{t-window}))
              / std(x_{t-1}, ..., x_{t-window})

    Uses shift(1), so the current value is NOT included in rolling mean/std.
    """
    out = pd.Series(index=df.index, dtype=float)

    for tenor, g in df.groupby("tenor", sort=True):
        g = g.sort_values(["date", "trade_date"]).copy()

        x = pd.to_numeric(g[value_col], errors="coerce")
        history = x.shift(1)

        roll_mean = history.rolling(
            window=window,
            min_periods=min_periods,
        ).mean()

        roll_std = history.rolling(
            window=window,
            min_periods=min_periods,
        ).std(ddof=ddof)

        z = (x - roll_mean) / roll_std
        z = z.replace([np.inf, -np.inf], np.nan)

        out.loc[g.index] = z

    return out


panel = panel.sort_values(["tenor", "date", "trade_date"]).copy()

panel["vrp_z_3m_prior_only"] = compute_prior_only_z_by_tenor(
    panel,
    value_col="vrp_log",
    window=63,
    min_periods=30,
    ddof=1,
)

panel["vrp_z_1y_prior_only"] = compute_prior_only_z_by_tenor(
    panel,
    value_col="vrp_log",
    window=252,
    min_periods=126,
    ddof=1,
)

panel["vrp_z_3m"] = panel["vrp_z_3m_prior_only"]
panel["vrp_z_1y"] = panel["vrp_z_1y_prior_only"]


# -----------------------------
# 5. Recompute parameter sweep eligibility
# -----------------------------

eligibility_required_cols = [
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "rv21d",
    "outcome_value",
]

for c in eligibility_required_cols:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel["has_outcome"] = panel["has_outcome"].astype(bool)

panel["is_parameter_sweep_eligible"] = (
    panel["has_outcome"]
    & panel[eligibility_required_cols].notna().all(axis=1)
)

panel = panel.sort_values(["date", "tenor"]).reset_index(drop=True)


# -----------------------------
# 6. Diagnostics: old vs new z-scores
# -----------------------------

def summarize_old_new_z(df, old_col, new_col, label):
    old = pd.to_numeric(df[old_col], errors="coerce")
    new = pd.to_numeric(df[new_col], errors="coerce")

    both = old.notna() & new.notna()
    diff = new[both] - old[both]

    return {
        "z_horizon": label,
        "old_col": old_col,
        "new_col": new_col,
        "rows": len(df),
        "old_non_null": int(old.notna().sum()),
        "new_non_null": int(new.notna().sum()),
        "overlap_non_null": int(both.sum()),
        "old_only_non_null": int((old.notna() & new.isna()).sum()),
        "new_only_non_null": int((old.isna() & new.notna()).sum()),
        "mean_old": float(old.mean()),
        "mean_new": float(new.mean()),
        "median_old": float(old.median()),
        "median_new": float(new.median()),
        "std_old": float(old.std(ddof=1)),
        "std_new": float(new.std(ddof=1)),
        "mean_new_minus_old": float(diff.mean()) if len(diff) else np.nan,
        "median_new_minus_old": float(diff.median()) if len(diff) else np.nan,
        "mean_abs_new_minus_old": float(diff.abs().mean()) if len(diff) else np.nan,
        "max_abs_new_minus_old": float(diff.abs().max()) if len(diff) else np.nan,
        "corr_old_new": float(old[both].corr(new[both])) if both.sum() > 1 else np.nan,
    }


z_comparison_summary = pd.DataFrame([
    summarize_old_new_z(
        panel,
        old_col="vrp_z_3m_inclusive_current_old",
        new_col="vrp_z_3m",
        label="3m",
    ),
    summarize_old_new_z(
        panel,
        old_col="vrp_z_1y_inclusive_current_old",
        new_col="vrp_z_1y",
        label="1y",
    ),
])

eligibility_summary = pd.DataFrame([{
    "rows": len(panel),
    "old_inclusive_eligible_rows": int(panel["is_parameter_sweep_eligible_inclusive_current_old"].astype(bool).sum()),
    "new_prior_only_eligible_rows": int(panel["is_parameter_sweep_eligible"].astype(bool).sum()),
    "eligible_rows_lost": int(
        panel["is_parameter_sweep_eligible_inclusive_current_old"].astype(bool).sum()
        - panel["is_parameter_sweep_eligible"].astype(bool).sum()
    ),
    "old_vrp_z_3m_non_null": int(panel["vrp_z_3m_inclusive_current_old"].notna().sum()),
    "new_vrp_z_3m_non_null": int(panel["vrp_z_3m"].notna().sum()),
    "old_vrp_z_1y_non_null": int(panel["vrp_z_1y_inclusive_current_old"].notna().sum()),
    "new_vrp_z_1y_non_null": int(panel["vrp_z_1y"].notna().sum()),
}])

by_tenor_summary_rows = []

for tenor, g in panel.groupby("tenor", sort=True):
    row = {
        "tenor": int(tenor),
        "rows": len(g),
        "old_eligible_rows": int(g["is_parameter_sweep_eligible_inclusive_current_old"].astype(bool).sum()),
        "new_eligible_rows": int(g["is_parameter_sweep_eligible"].astype(bool).sum()),
        "old_z_3m_non_null": int(g["vrp_z_3m_inclusive_current_old"].notna().sum()),
        "new_z_3m_non_null": int(g["vrp_z_3m"].notna().sum()),
        "old_z_1y_non_null": int(g["vrp_z_1y_inclusive_current_old"].notna().sum()),
        "new_z_1y_non_null": int(g["vrp_z_1y"].notna().sum()),
    }

    by_tenor_summary_rows.append(row)

by_tenor_summary = pd.DataFrame(by_tenor_summary_rows)

print()
print("Old inclusive-current z vs new prior-only z comparison:")
display(z_comparison_summary)

print()
print("Eligibility summary:")
display(eligibility_summary)

print()
print("Eligibility / z-score availability by tenor:")
display(by_tenor_summary)


# -----------------------------
# 7. Internal verification audit
# -----------------------------

def verify_prior_only_match(df, stored_col, value_col, window, min_periods, ddof, label):
    recomputed = compute_prior_only_z_by_tenor(
        df,
        value_col=value_col,
        window=window,
        min_periods=min_periods,
        ddof=ddof,
    )

    stored = pd.to_numeric(df[stored_col], errors="coerce")

    stored_finite = np.isfinite(stored)
    recomputed_finite = np.isfinite(recomputed)
    both = stored_finite & recomputed_finite

    diff = stored[both] - recomputed[both]

    if both.sum() > 0:
        rmse = float(np.sqrt(np.nanmean(np.square(diff))))
        max_abs_error = float(np.nanmax(np.abs(diff)))
        mae = float(np.nanmean(np.abs(diff)))
        corr = float(stored[both].corr(recomputed[both]))
    else:
        rmse = np.nan
        max_abs_error = np.nan
        mae = np.nan
        corr = np.nan

    null_mismatch_count = int(stored_finite.ne(recomputed_finite).sum())

    exact_match = (
        null_mismatch_count == 0
        and np.isfinite(rmse)
        and np.isfinite(max_abs_error)
        and rmse == 0.0
        and max_abs_error == 0.0
    )

    status = "PASS_PRIOR_ONLY" if exact_match else "FAIL_RECHECK"

    return {
        "z_horizon": label,
        "stored_col": stored_col,
        "convention": "prior_only_shift1",
        "window": window,
        "min_periods": min_periods,
        "ddof": ddof,
        "stored_non_null": int(stored_finite.sum()),
        "recomputed_non_null": int(recomputed_finite.sum()),
        "overlap_non_null": int(both.sum()),
        "null_mismatch_count": null_mismatch_count,
        "corr": corr,
        "mae": mae,
        "rmse": rmse,
        "max_abs_error": max_abs_error,
        "status": status,
    }


verification = pd.DataFrame([
    verify_prior_only_match(
        panel,
        stored_col="vrp_z_3m",
        value_col="vrp_log",
        window=63,
        min_periods=30,
        ddof=1,
        label="3m",
    ),
    verify_prior_only_match(
        panel,
        stored_col="vrp_z_1y",
        value_col="vrp_log",
        window=252,
        min_periods=126,
        ddof=1,
        label="1y",
    ),
])

print()
print("Prior-only z-score verification:")
display(verification)

if not verification["status"].eq("PASS_PRIOR_ONLY").all():
    raise ValueError("Prior-only z-score verification failed. Do not continue to sweeps.")


# -----------------------------
# 8. Save corrected panel and diagnostics
# -----------------------------

safe_start = panel["date"].min().strftime("%Y%m%d")
safe_end = panel["date"].max().strftime("%Y%m%d")

corrected_panel_path = (
    NEW_PROCESSED_DIR
    / f"03_corsi_parameter_sweep_prior_only_z_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

z_comparison_summary_path = (
    NEW_AUDIT_DIR
    / f"03_corsi_parameter_sweep_prior_only_z_cell9_z_comparison_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

eligibility_summary_path = (
    NEW_AUDIT_DIR
    / f"03_corsi_parameter_sweep_prior_only_z_cell9_eligibility_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

by_tenor_summary_path = (
    NEW_AUDIT_DIR
    / f"03_corsi_parameter_sweep_prior_only_z_cell9_by_tenor_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

verification_path = (
    NEW_AUDIT_DIR
    / f"03_corsi_parameter_sweep_prior_only_z_cell9_prior_only_verification_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

panel.to_parquet(corrected_panel_path, index=False)
z_comparison_summary.to_csv(z_comparison_summary_path, index=False)
eligibility_summary.to_csv(eligibility_summary_path, index=False)
by_tenor_summary.to_csv(by_tenor_summary_path, index=False)
verification.to_csv(verification_path, index=False)

print()
print("Cell 9 outputs saved:")
print(f"Corrected prior-only z panel: {corrected_panel_path}")
print(f"Z comparison summary:        {z_comparison_summary_path}")
print(f"Eligibility summary:         {eligibility_summary_path}")
print(f"By-tenor summary:            {by_tenor_summary_path}")
print(f"Prior-only verification:     {verification_path}")

print()
print("Cell 9 complete.")

Cell 9: Rebuild clean panel with prior-only z-scores
Source clean panel: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_clean\02_corsi_parameter_sweep_clean_panel_20200102_20260623_20260704_085138.parquet
New processed dir:  C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_prior_only_z
New audit dir:      C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_prior_only_z
Run timestamp:      20260704_104831

Source panel summary:


,rows,first_date,last_date,unique_dates,unique_tenors,source_vrp_z_3m_non_null,source_vrp_z_1y_non_null,source_eligible_rows
0,14565,2020-01-02,2026-06-23,1626,9,14304,13440,13374



Old inclusive-current z vs new prior-only z comparison:


,z_horizon,old_col,new_col,rows,old_non_null,new_non_null,overlap_non_null,old_only_non_null,new_only_non_null,mean_old,mean_new,median_old,median_new,std_old,std_new,mean_new_minus_old,median_new_minus_old,mean_abs_new_minus_old,max_abs_new_minus_old,corr_old_new
0,3m,vrp_z_3m_inclusive_current_old,vrp_z_3m,14565,14304,14295,14295,9,0,0.007087,0.006522,0.021842,0.022975,1.086962,1.135762,-0.000637,0.000183,0.034691,1.054203,0.999118
1,1y,vrp_z_1y_inclusive_current_old,vrp_z_1y,14565,13440,13431,13431,9,0,-0.129884,-0.131333,-0.113547,-0.115075,1.015115,1.025501,-0.001027,-0.000760,0.007776,0.226356,0.999949



Eligibility summary:


...


Eligibility / z-score availability by tenor:


,tenor,rows,old_eligible_rows,new_eligible_rows,old_z_3m_non_null,new_z_3m_non_null,old_z_1y_non_null,new_z_1y_non_null
0,9,1626,1495,1494,1597,1596,1501,1500
1,12,1624,1492,1491,1595,1594,1499,1498
2,15,1623,1490,1489,1594,1593,1498,1497
3,18,1620,1488,1487,1591,1590,1495,1494
4,21,1619,1485,1484,1590,1589,1494,1493
5,24,1616,1484,1483,1587,1586,1491,1490
6,27,1615,1481,1480,1586,1585,1490,1489
7,30,1612,1481,1480,1583,1582,1487,1486
8,33,1610,1478,1477,1581,1580,1485,1484



Prior-only z-score verification:


,z_horizon,stored_col,convention,window,min_periods,ddof,stored_non_null,recomputed_non_null,overlap_non_null,null_mismatch_count,corr,mae,rmse,max_abs_error,status
0,3m,vrp_z_3m,prior_only_shift1,63,30,1,14295,14295,14295,0,1.0,0.0,0.0,0.0,PASS_PRIOR_ONLY
1,1y,vrp_z_1y,prior_only_shift1,252,126,1,13431,13431,13431,0,1.0,0.0,0.0,0.0,PASS_PRIOR_ONLY



Cell 9 outputs saved:
Corrected prior-only z panel: C:\Users\patri\vrp_project\data\processed\corsi_parameter_sweep_prior_only_z\03_corsi_parameter_sweep_prior_only_z_panel_20200102_20260623_20260704_104831.parquet
Z comparison summary:        C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_prior_only_z\03_corsi_parameter_sweep_prior_only_z_cell9_z_comparison_summary_20200102_20260623_20260704_104831.csv
Eligibility summary:         C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_prior_only_z\03_corsi_parameter_sweep_prior_only_z_cell9_eligibility_summary_20200102_20260623_20260704_104831.csv
By-tenor summary:            C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_prior_only_z\03_corsi_parameter_sweep_prior_only_z_cell9_by_tenor_summary_20200102_20260623_20260704_104831.csv
Prior-only verification:     C:\Users\patri\vrp_project\data\audit\corsi_parameter_sweep_prior_only_z\03_corsi_parameter_sweep_prior_only_z_cell9_prior_only_verification_2020